# 05. 소둔로(CAL01) 전체 데이터 시계열 분석

**담당:** 정수진 · **Issue:** #65  
**대상:** `P1-CR1-CAL01` 연속 소둔로  
**분석 단위:** LOT별 관측 구간 · 동일 강종 LOT 비교  
**작성일:** 2026-10-07

## 분석 목적

이 노트북의 목적은 단순히 `Z-score가 큰 값`을 찾는 것이 아니다.

**정상적인 조건 차이를 먼저 분리하고, 같은 조건에서도 시간적으로 특이한 LOT/COIL을 찾은 뒤,  
강종 · 용접 · 더미코일 후보 · 정기수리 · BM/CBM/TBM · 부품교체와 연결하여  
끝까지 설명되지 않는 지속적인 변화를 최종 이상 후보로 좁히는 것**이 목적이다.

전체 흐름은 다음과 같다.

| 단계 | 분석 항목 | 핵심 질문 | 주요 분석 내용 | 결과 / 다음 단계 |
|---:|---|---|---|---|
| **0** | **데이터 정의 및 연결 가능성** | 각 데이터는 무엇을 의미하고 서로 언제 연결할 수 있는가? | CAL01 온도, WLD01 용접, 강종변경, 정기수리, 정비, 부품교체의 컬럼·시간범위·연결키 확인 | 분석 가능한 데이터 관계 정의 |
| **1** | **2024년 전체 통합 타임라인** | 센서 관측과 조업·정비·용접 사건은 언제 겹치는가? | LOT, 강종변경, 용접, BM/CBM/TBM, 부품교체, 정기수리를 연간 시간축에 통합 | 상세 분석할 시간대와 Focus LOT 후보 확인 |
| **2** | **데이터 품질 / 관측범위** | 실제 공정 변화와 데이터 수집 문제를 구분할 수 있는가? | Sampling interval, 결측, 관측시간, LOT 길이, WLD 39 SEQ 완전성, PWR=0 확인 | 분석 가능 / 주의 / 연결 불가 구간 구분 |
| **3** | **강종별 LOT 분리** | 강종 자체의 차이를 이상으로 오해하고 있지 않은가? | 동일 강종 LOT 구성, 반복 LOT 수, 강종별 기준수준 비교 | 동일 강종 비교 가능 범위 결정 |
| **4** | **강종별·공정별 전체 시계열** | 같은 강종에서도 어느 공정부터 LOT 차이가 나타나는가? | 세정 → 건조 → 가열·균열 → 과시효 → 냉각 순으로 TC/OP/CP 시간흐름 비교 | 특이 공정과 센서 후보 선정 |
| **5** | **Level / Trend / Variance 분석** | 차이가 수준·추세·변동성 중 무엇 때문인가? | Median/MAD/IQR, 시작·종료 수준, slope, rolling variation | 그래프에서 보인 차이를 통계량으로 수치화 |
| **6** | **Robust Z 기반 LOT 후보 스크리닝** | 같은 강종 안에서도 통계적으로 다른 LOT은 무엇인가? | 동일 강종·동일 센서 기준 Level/Trend/Variance Robust Z | 상세 분석할 LOT 우선순위 결정 |
| **7** | **후보 LOT 실제시간 상세분석** | 특이 변화는 언제 시작해 얼마나 지속되는가? | 실제 경과시간(min), TC↔OP↔CP, rolling 통계, 변화 시작·안정화 시점 | 변화 Episode 후보와 공정 반응 순서 확인 |
| **8** | **WLD01 용접 패턴 / Recipe / COIL 분석** | 용접 자체의 정상 패턴과 특이 COIL은 무엇인가? | 39 SEQ 구조, SPD/LEN/SET-PWR/GATE-MS Recipe, PWR 추세, 조건보정 Robust Z | 용접조건 차이와 실제 용접 변화 후보 분리 |
| **9** | **더미코일·Shutdown·Restart 가설 검증** | 특이한 LOT이 더미 운전이나 재가동으로 설명되는가? | 더미후보 강종, 정기수리 전후 위치, WLD 용접, 온도 ramp-up/down 연결 | 정상 상태전환인지 이상 후보인지 구분 |
| **10** | **정기수리·BM·CBM·TBM·부품교체 연결** | 센서 변화와 실제 보전 사건이 시간적으로 연결되는가? | DETC/STRT/END, FAIL_MODE, REMARK, PART_CHG, 교체부품, ±24/72h Window | 정비 관련 변화 후보 분리 |
| **11** | **반복사례·반례 검증** | 발견한 패턴이 다른 사례에서도 반복되는가? | 같은 현상의 반복, 수리 후 미발생 사례, SPHC 안정 사례 등 반례 비교 | 가설 신뢰도 평가 및 과도한 해석 방지 |
| **12** | **Episode 생성 및 최종 이상 후보 분류** | 최종적으로 어떤 변화를 이상 후보로 남길 것인가? | 지속조건을 적용해 Episode 생성 후 정상/조건차이/상태전환/정비관련/미설명 변화로 분류 | 모델링 단계에 전달할 최종 이상 후보 확정 |

> **중요**
> - 센서 데이터가 없는 기간을 0으로 채우거나 보간하지 않는다.
> - `센서 없음 = 설비정지`, `용접 데이터 없음 = 용접 없음`으로 해석하지 않는다.
> - 통계적으로 크다고 바로 고장/이상으로 확정하지 않는다.

가로축은 실제 날짜 또는 경과 **분**을 사용합니다. OP의 단위는 **%**, OP 차이(Δ)는 **%p**, CP는 **무차원**입니다.

아래 공통 설정 셀을 그대로 사용합니다. 작업 디렉토리는 `notebooks/`입니다. 그림은 `../figures/05_all_data_timeseries_jinny_01/` 아래에 주제별로 저장하며, 파일명은 `번호_설명.png` 형식입니다. 계산된 표의 결과값은 **마크다운 셀**에 보존하며, 표를 출력하기 위한 코드나 결과 CSV를 생성하지 않습니다.

분석 설명·정비 원문·탐색 기록·최종 해석은 해당 분석 구간의 **마크다운 셀**에 보존한다. 별도의 보고서 폴더나 마크다운 파일은 생성하지 않는다.


## 0-1. 공통 환경 설정

팀 공통 규칙 셀을 그대로 사용한다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 0-2. 이번 노트북에서 추가로 사용할 파일 / 출력 폴더

용접 CSV만 공통셀 외에 추가한다.

In [ ]:
WLD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 이 노트북의 그림만 저장한다. 원본 CSV는 읽기 전용으로 사용한다.
OUT_DIR = os.path.join(FIG_DIR, "05_all_data_timeseries_jinny_01")
ANNUAL_DIR = os.path.join(OUT_DIR, "01_연간타임라인")
OBS_DIR = os.path.join(OUT_DIR, "02_관측범위")
GRADE_DIR = os.path.join(OUT_DIR, "03_강종별비교")
GRADE_SUMMARY_DIR = os.path.join(GRADE_DIR, "01_공정요약")
STATS_DIR = os.path.join(GRADE_DIR, "03_실제값비교")
DETAIL_DIR = os.path.join(OUT_DIR, "04_LOT별상세")
EVENT_DIR = os.path.join(OUT_DIR, "05_사건연결")
WLD_DIR = os.path.join(OUT_DIR, "06_용접")
FINAL_DIR = os.path.join(OUT_DIR, "07_최종결과")
for folder in [ANNUAL_DIR, OBS_DIR, GRADE_SUMMARY_DIR, STATS_DIR,
               DETAIL_DIR, EVENT_DIR, WLD_DIR, FINAL_DIR]:
    os.makedirs(folder, exist_ok=True)


def tl_detail_filename(name):
    prefixes = {"plan_": "01_계획수리_", "lot_": "02_LOT_",
                "maintenance_": "03_TBM_작업겹침_", "parts_": "04_부품연결_",
                "gap_": "05_관측공백_", "bm_": "06_BM_"}
    for old_prefix, new_prefix in prefixes.items():
        if name.startswith(old_prefix):
            return new_prefix + name[len(old_prefix):]
    return name


from IPython.display import Markdown, display


print("그림 저장 폴더:", OUT_DIR)


# 0. 데이터 정의 및 시간 연결 가능성

## 질문

- 각 파일은 무엇을 설명하는가?
- 직접 키로 연결 가능한가, 시간 기준으로만 연결 가능한가?
- 어느 데이터가 실제로 같은 시간대에 존재하는가?

### 데이터 역할

- `CAL01 온도` : LOT 내부 공정 시계열
- `조업이벤트` : 강종변경
- `WLD01 용접` : COIL 접합 시퀀스
- `정기수리` : 계획된 수리 경계
- `정비이력` : BM / CBM / TBM
- `부품교체` : Work Order와 부품 사건
- `설비/부품마스터` : 설비 및 부품 속성

In [ ]:
temp = pd.read_csv(TEMP_CSV)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)
wld = pd.read_csv(WLD_CSV, encoding=ENC)

temp["MEAS_DT"] = pd.to_datetime(temp["MEAS_DT"])
event["EVT_DT"] = pd.to_datetime(event["EVT_DT"])
repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
repair["END_DT"] = pd.to_datetime(repair["END_DT"])
maint["DETC_DT"] = pd.to_datetime(maint["DETC_DT"])
maint["STRT_DT"] = pd.to_datetime(maint["STRT_DT"])
maint["END_DT"] = pd.to_datetime(maint["END_DT"])
wld["MEAS_DT"] = pd.to_datetime(wld["MEAS_DT"])

print("CAL01:", temp.shape)
print("EVENT:", event.shape)
print("REPAIR:", repair.shape)
print("MAINT:", maint.shape)
print("PARTCHG:", partchg.shape)
print("WLD01:", wld.shape)

## 공통 함수

강건 척도는 참고 순위를 계산하는 용도다. 그림은 원자료 값과 실제 경과 초/분을 우선 사용한다.

In [ ]:
def robust_scale(series):
    x = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if len(x) < 2:
        return np.nan

    med = np.median(x)
    mad = np.median(np.abs(x - med))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= 1e-12:
        q25, q75 = np.quantile(x, [0.25, 0.75])
        scale = (q75 - q25) / 1.349

    if not np.isfinite(scale) or scale <= 1e-12:
        scale = np.std(x, ddof=1)

    if not np.isfinite(scale) or scale <= 1e-12:
        return np.nan

    return float(scale)


def robust_z(value, reference_values):
    ref = pd.to_numeric(pd.Series(reference_values), errors="coerce").dropna()
    if len(ref) < 2 or pd.isna(value):
        return np.nan

    center = ref.median()
    scale = robust_scale(ref)

    if not np.isfinite(scale):
        return np.nan

    return (value - center) / scale


def safe_minutes(td):
    return td.total_seconds() / 60 if pd.notna(td) else np.nan

# 1. 연간 시간 관계에서 확인 질문으로

연간 그림은 관측 범위·강종변경·정비·교체의 위치를 찾는 출발점이다.
확대 그림에는 강종 기록, 실제 관측·작업 구간, 확인할 겹침만 표시한다.
정확한 시각·REMARK·교체사유·관계표와 해석은 그림 아래 마크다운에서 읽는다.
색은 CSV 출처를, 붉은 구간은 실제 시간 교집합을 나타낸다. 감지 시각은 작업 구간과 구분해 마크다운에 보존한다.


**작업 범위는 CAL01 소둔로 44건으로 한정한다.** 부품교체는 이 작업번호와 연결된 9행이다. 다른 CR1 설비의 작업과 용접기 자체 정비는 제외한다. 그림 1B는 건수표가 아니라 실제 신호와 사건의 선후관계를, 그림 1C는 계획수리·작업번호·부품·후속 LOT의 연결을 보여준다. 0건을 포함한 14 LOT의 수치표는 마크다운으로 보존한다.


In [ ]:
# LOT 시간
lot_time = (
    temp.groupby("LOT_NO", as_index=False)
        .agg(
            LOT_START=("MEAS_DT", "min"),
            LOT_END=("MEAS_DT", "max")
        )
)

lot_time["DURATION_MIN"] = (
    lot_time["LOT_END"] - lot_time["LOT_START"]
).dt.total_seconds() / 60

# CR1 강종변경
grade_evt = (
    event[
        (event["PLANT_CD"] == "CR1")
        & (event["EVT_TYPE"] == "강종변경")
    ][["EVT_DT", "BEF_VAL", "AFT_VAL"]]
    .sort_values("EVT_DT")
    .copy()
)

# LOT 시작시각과 정확히 일치하는 강종
lot_info = (
    lot_time.merge(
        grade_evt[["EVT_DT", "AFT_VAL"]].rename(
            columns={"EVT_DT": "LOT_START", "AFT_VAL": "GRADE"}
        ),
        on="LOT_START",
        how="left"
    )
    .sort_values("LOT_START")
    .reset_index(drop=True)
)

# CAL01 정비
cal_maint = maint[maint["EQP_TAG"] == "P1-CR1-CAL01"].copy()

# CAL01 부품교체
cal_partchg = (
    cal_maint[["WO_NO", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "REMARK"]]
    .merge(partchg, on="WO_NO", how="inner")
)

# WLD01 코일 시작/종료
wld_coil = (
    wld.groupby("COIL_ID", as_index=False)
       .agg(
           WLD_START=("MEAS_DT", "min"),
           WLD_END=("MEAS_DT", "max")
       )
)

**결과 — LOT별 관측 기간과 강종**

| 행 | LOT_NO | LOT_START | LOT_END | DURATION_MIN | GRADE |
| --- | --- | --- | --- | --- | --- |
| 0 | 240001 | 2024-01-02 03:36:00 | 2024-01-02 15:04:30 | 688.500000 | SGCC |
| 1 | 240024 | 2024-03-07 21:22:00 | 2024-03-08 06:29:48 | 547.800000 | SPFH590 |
| 2 | 240037 | 2024-04-13 23:16:00 | 2024-04-14 09:27:19 | 611.316667 | SPCE |
| 3 | 240038 | 2024-04-16 11:35:00 | 2024-04-16 13:39:56 | 124.933333 | SPCEN |
| 4 | 240044 | 2024-05-03 18:50:00 | 2024-05-03 20:04:38 | 74.633333 | SPHC |
| 5 | 240058 | 2024-06-09 05:26:00 | 2024-06-09 09:02:53 | 216.883333 | DP590 |
| 6 | 240061 | 2024-06-15 01:04:00 | 2024-06-15 04:49:23 | 225.383333 | SPHC |
| 7 | 240069 | 2024-07-07 16:57:00 | 2024-07-08 06:29:34 | 812.566667 | SPHC |
| 8 | 240091 | 2024-09-08 22:03:00 | 2024-09-09 08:44:12 | 641.200000 | SPHC |
| 9 | 240108 | 2024-10-18 04:05:00 | 2024-10-18 11:50:53 | 465.883333 | SPFH590 |
| 10 | 240114 | 2024-11-04 15:40:00 | 2024-11-04 20:59:02 | 319.033333 | SPCEN |
| 11 | 240117 | 2024-11-13 07:08:00 | 2024-11-13 14:32:19 | 444.316667 | SPCD |
| 12 | 240121 | 2024-11-24 07:31:00 | 2024-11-24 18:22:12 | 651.200000 | SPCE |
| 13 | 240133 | 2024-12-28 14:57:00 | 2024-12-28 16:44:24 | 107.400000 | SPCE |

In [ ]:
# %% 준비: 원본 분석 변수는 유지하고 시각화용 표만 별도로 만든다.
import textwrap
from pathlib import Path
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

TIMELINE_YEAR = 2024
YEAR_START = pd.Timestamp(TIMELINE_YEAR, 1, 1)
YEAR_END = pd.Timestamp(TIMELINE_YEAR + 1, 1, 1)
TL_COLORS = {"LOT": "#086fae", "WLD": "#df8a09", "BM": "#d92d20",
             "CBM": "#0077b6", "TBM": "#64748b", "PART": "#7841bc",
             "PLAN": "#dff0f8", "OVERLAP": "#c02646", "FOCUS": "#7841bc"}
TL_DETAIL_DIR = Path(EVENT_DIR) / "01_기간확대"
TL_DETAIL_DIR.mkdir(parents=True, exist_ok=True)

# 날짜 단위 계획은 종료일 전체를 포함한다. 원본 END_DT는 바꾸지 않는다.
cr1_repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
cr1_repair["END_EXCLUSIVE"] = cr1_repair["END_DT"].dt.normalize() + pd.Timedelta(days=1)
tl_plans = cr1_repair.loc[(cr1_repair["STRT_DT"] < YEAR_END) &
                          (cr1_repair["END_EXCLUSIVE"] > YEAR_START)].copy()
tl_lots = lot_info.loc[(lot_info["LOT_START"] < YEAR_END) &
                       (lot_info["LOT_END"] >= YEAR_START)].copy()
tl_maint = cal_maint.loc[(cal_maint["DETC_DT"].ge(YEAR_START) &
                        cal_maint["DETC_DT"].lt(YEAR_END)) |
                       ((cal_maint["STRT_DT"] < YEAR_END) &
                        (cal_maint["END_DT"] >= YEAR_START))].sort_values("STRT_DT").copy()
if (tl_maint["END_DT"] < tl_maint["STRT_DT"]).any():
    raise ValueError("정비 종료가 착수보다 빠른 행을 확인하세요.")

# 교체 자체의 시각은 제공되지 않는다. WO 작업 구간과 완료시각을 연결한다.
tl_parts = (tl_maint[["WO_NO", "STRT_DT", "END_DT", "MNT_TYPE"]]
            .merge(partchg, on="WO_NO", how="inner", validate="one_to_many")
            .merge(part[["PART_CD", "PART_NM"]], on="PART_CD", how="left",
                   validate="many_to_one").sort_values("END_DT"))
tl_wld = wld_coil.loc[(wld_coil["WLD_START"] < YEAR_END) &
                      (wld_coil["WLD_END"] >= YEAR_START)].copy()
tl_grades = grade_evt.loc[grade_evt["EVT_DT"].ge(YEAR_START) &
                          grade_evt["EVT_DT"].lt(YEAR_END)].copy()

def tl_intersection(a0, a1, b0, b1):
    """관측/작업의 닫힌 구간 교집합. 날짜 계획은 별도로 반열린 구간 처리."""
    start, end = max(a0, b0), min(a1, b1)
    return (start, end) if start <= end else None

tl_overlap_rows = []
def tl_add_overlap(kind_a, id_a, a0, a1, kind_b, id_b, b0, b1):
    hit = tl_intersection(a0, a1, b0, b1)
    if hit is not None:
        tl_overlap_rows.append({"종류_A": kind_a, "ID_A": str(id_a),
                                "종류_B": kind_b, "ID_B": str(id_b),
                                "겹침_시작": hit[0], "겹침_종료": hit[1],
                                "겹침_분": (hit[1] - hit[0]).total_seconds() / 60})

tl_focus_reasons = {int(r.LOT_NO): [] for r in tl_lots.itertuples()}
for p in tl_plans.itertuples():
    before = tl_lots.loc[tl_lots["LOT_END"] < p.STRT_DT].tail(1)
    after = tl_lots.loc[tl_lots["LOT_START"] >= p.END_EXCLUSIVE].head(1)
    for frame, reason in [(before, "수리 전 마지막 관측"), (after, "수리 후 첫 관측")]:
        for lot in frame["LOT_NO"]:
            tl_focus_reasons[int(lot)].append(reason)

tl_lot_rows = []
for lot in tl_lots.itertuples():
    tasks = tl_maint.loc[(tl_maint["STRT_DT"] <= lot.LOT_END) &
                         (tl_maint["END_DT"] >= lot.LOT_START)]
    detected = tl_maint.loc[tl_maint["DETC_DT"].between(lot.LOT_START, lot.LOT_END)]
    coils = tl_wld.loc[(tl_wld["WLD_START"] <= lot.LOT_END) &
                       (tl_wld["WLD_END"] >= lot.LOT_START)]
    if len(tasks):
        tl_focus_reasons[int(lot.LOT_NO)].append("정비 작업 겹침")
    if len(detected):
        tl_focus_reasons[int(lot.LOT_NO)].append("정비 감지 포함")
    for task in tasks.itertuples():
        tl_add_overlap("LOT", int(lot.LOT_NO), lot.LOT_START, lot.LOT_END,
                       task.MNT_TYPE, task.WO_NO, task.STRT_DT, task.END_DT)
    for coil in coils.itertuples():
        tl_add_overlap("LOT", int(lot.LOT_NO), lot.LOT_START, lot.LOT_END,
                       "WLD", coil.COIL_ID, coil.WLD_START, coil.WLD_END)
    for p in tl_plans.itertuples():
        if lot.LOT_START < p.END_EXCLUSIVE and lot.LOT_END >= p.STRT_DT:
            tl_add_overlap("LOT", int(lot.LOT_NO), lot.LOT_START, lot.LOT_END,
                           "계획수리", p.PLAN_ID, p.STRT_DT, p.END_EXCLUSIVE)
    tl_lot_rows.append({"LOT": int(lot.LOT_NO), "강종": lot.GRADE,
                        "관측_시작": lot.LOT_START, "관측_종료": lot.LOT_END,
                        "관측_분": lot.DURATION_MIN, "정비_작업": ", ".join(tasks["WO_NO"]),
                        "정비_감지": ", ".join(detected["WO_NO"]),
                        "WLD_COIL수": len(coils),
                        "Focus_이유": " / ".join(tl_focus_reasons[int(lot.LOT_NO)])})
for i, a in enumerate(tl_maint.itertuples()):
    for b in list(tl_maint.itertuples())[i + 1:]:
        tl_add_overlap(a.MNT_TYPE, a.WO_NO, a.STRT_DT, a.END_DT,
                       b.MNT_TYPE, b.WO_NO, b.STRT_DT, b.END_DT)
    for p in tl_plans.itertuples():
        if a.STRT_DT < p.END_EXCLUSIVE and a.END_DT >= p.STRT_DT:
            tl_add_overlap(a.MNT_TYPE, a.WO_NO, a.STRT_DT, a.END_DT,
                           "계획수리", p.PLAN_ID, p.STRT_DT, p.END_EXCLUSIVE)
    coils = tl_wld.loc[(tl_wld["WLD_START"] <= a.END_DT) & (tl_wld["WLD_END"] >= a.STRT_DT)]
    for coil in coils.itertuples():
        tl_add_overlap(a.MNT_TYPE, a.WO_NO, a.STRT_DT, a.END_DT,
                       "WLD", coil.COIL_ID, coil.WLD_START, coil.WLD_END)
for p in tl_plans.itertuples():
    coils = tl_wld.loc[(tl_wld["WLD_START"] < p.END_EXCLUSIVE) & (tl_wld["WLD_END"] >= p.STRT_DT)]
    for coil in coils.itertuples():
        tl_add_overlap("계획수리", p.PLAN_ID, p.STRT_DT, p.END_EXCLUSIVE,
                       "WLD", coil.COIL_ID, coil.WLD_START, coil.WLD_END)

tl_lot_table = pd.DataFrame(tl_lot_rows)
tl_overlap = pd.DataFrame(tl_overlap_rows, columns=["종류_A", "ID_A", "종류_B", "ID_B",
                                                 "겹침_시작", "겹침_종료", "겹침_분"])
tl_focus = tl_lot_table.loc[tl_lot_table["Focus_이유"].ne(""), "LOT"].tolist()
tl_focus_ids = {lot: f"F{i + 1}" for i, lot in enumerate(tl_focus)}
tl_lot_table.insert(0, "Focus", tl_lot_table["LOT"].map(tl_focus_ids).fillna(""))
tl_wld_day = (tl_wld.assign(DATE=tl_wld["WLD_START"].dt.floor("D"))
              .groupby("DATE")["COIL_ID"].nunique().reset_index(name="N_COIL"))

def tl_style(ax):
    ax.set_facecolor("white")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#94a3b8")
    ax.tick_params(axis="both", labelsize=9, color="#94a3b8")
    ax.tick_params(axis="y", length=0, pad=15)
    ax.grid(axis="x", color="#e4eaf0", linewidth=.7)
    ax.set_axisbelow(True)

def tl_stack(times, min_days=2.8):
    """화면에서 구별하기 어려운 인접 이벤트는 같은 행 안의 세부 행에 배치."""
    ends, positions = [], []
    for start, end in times:
        x0, x1 = mdates.date2num(start), mdates.date2num(end)
        lane = next((i for i, last in enumerate(ends) if x0 - last >= min_days), len(ends))
        if lane == len(ends):
            ends.append(x1)
        else:
            ends[lane] = x1
        positions.append(lane)
    n = max(len(ends), 1)
    return [(p - (n - 1) / 2) * min(.15, .52 / n) for p in positions]

def tl_place_lot_labels(ax, records, y):
    """실제 글자 경계 상자로 충돌을 검사하고 라벨 높이를 자동으로 올린다."""
    fig = ax.figure
    fig.canvas.draw()
    boxes, labels = [], []
    for lot in records:
        focus = tl_focus_ids.get(int(lot.LOT_NO), "")
        label = f"{int(lot.LOT_NO)}\n{lot.GRADE}\n{lot.LOT_START:%m/%d}"
        for lane in range(7):
            artist = ax.annotate(label, (lot.LOT_START, y), xytext=(0, 17 + 49 * lane),
                                 textcoords="offset points", ha="center", va="bottom",
                                 fontsize=8.5, color=TL_COLORS["LOT"],
                                 bbox=dict(boxstyle="round,pad=.32", fc="white",
                                           ec="#d8e5ef", lw=.7),
                                 arrowprops=dict(arrowstyle="-", color="#a7b6c5", lw=.7), zorder=8)
            fig.canvas.draw()
            box = artist.get_bbox_patch().get_window_extent().expanded(1.12, 1.12)
            if not any(box.overlaps(other) for other in boxes):
                boxes.append(box)
                labels.append(artist)
                break
            artist.remove()
        else:
            raise RuntimeError("LOT 라벨 공간 부족: 그림 너비 또는 상단 여백을 늘리세요.")
    if boxes and max(b.y1 for b in boxes) > fig.bbox.y1 - 95:
        raise RuntimeError("LOT 라벨이 제목 영역에 접근합니다. 상단 여백을 늘리세요.")
    return labels

def tl_period(start, end, seconds=False):
    fmt = "%m/%d %H:%M:%S" if seconds else "%m/%d %H:%M"
    return f"{start:{fmt}} ~ {end:{fmt}}"

# 데이터 출처별 색상: 동일 CSV는 같은 색 계열을 사용한다.
TL_COLORS.update({"LOT": "#086f85", "GRADE": "#487dad", "WLD": "#d88a12",
                  "BM": "#a32b45", "CBM": "#bc5267", "TBM": "#cf7b8a",
                  "PART": "#7742a8", "PLAN": "#dff0f8", "OVERLAP": "#c32c39"})
TL_KIND_NAMES = {"BM": "사후정비", "CBM": "상태기반정비", "TBM": "시간기반정비"}
TL_PART_COLORS = {"INS-006": "#6934a2", "SEL-006": "#8752b8", "SEL-009": "#a477cb"}
TL_PART_MEANING = {"INS-006": "가스 분석계의 셀 · 계장 부품", "SEL-006": "오일씰 · 실링 부품 · 규격 150×190×18",
                   "SEL-009": "O링 묶음 · 실링 부품"}
assert tl_maint.EQP_TAG.eq("P1-CR1-CAL01").all(), "CAL01 이외 정비가 섞였습니다."

tl_maint.insert(0, "MAINT_ID", [f"M{i + 1:02d}" for i in range(len(tl_maint))])
# 부품 설명에는 마스터 분류와 원문의 교체사유·REMARK를 함께 보존한다.
tl_parts = (tl_maint[["MAINT_ID", "WO_NO", "STRT_DT", "END_DT", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "REMARK"]]
            .merge(partchg, on="WO_NO", how="inner", validate="one_to_many")
            .merge(part[["PART_CD", "PART_NM", "PART_TYPE", "APPLY_EQP_TYPE"]], on="PART_CD", how="left", validate="many_to_one")
            .sort_values(["END_DT", "PART_CD"]))
tl_part_rows = []
for item in tl_parts.itertuples():
    # 별도 교체시각이 없으므로 관측/교체 자체의 교집합과 섞지 않는다.
    for kind, target, c0, c1, key in [("LOT", tl_lots, "LOT_START", "LOT_END", "LOT_NO"),
                                    ("WLD", tl_wld, "WLD_START", "WLD_END", "COIL_ID")]:
        matching = target.loc[(target[c0] <= item.END_DT) & (target[c1] >= item.STRT_DT)]
        for other in matching.itertuples():
            hit = tl_intersection(item.STRT_DT, item.END_DT, getattr(other, c0), getattr(other, c1))
            tl_part_rows.append({"부품작업_ID": item.WO_NO + "/" + item.PART_CD, "WO_NO": item.WO_NO,
                                 "상대_종류": kind, "상대_ID": str(getattr(other, key)),
                                 "겹침_시작": hit[0], "겹침_종료": hit[1],
                                 "겹침_분": (hit[1] - hit[0]).total_seconds() / 60,
                                 "시간_기준": "부품교체 연결 작업 구간"})
tl_part_overlap = pd.DataFrame(tl_part_rows, columns=["부품작업_ID", "WO_NO", "상대_종류", "상대_ID",
                                                     "겹침_시작", "겹침_종료", "겹침_분", "시간_기준"])

def tl_clean(value):
    return str(value).strip() if pd.notna(value) and str(value).strip() else "기록 없음"

def tl_draw_interval(ax, start, end, yy, kind, color):
    if kind in ["BM", "CBM", "TBM"]:
        ax.plot([start, end], [yy] * 2, color=color, lw=7, solid_capstyle="butt", zorder=3)
        ax.scatter([start, end], [yy, yy], marker="x", s=64, linewidths=1.8, color=color, zorder=7)
    elif kind == "PART":
        ax.plot([start, end], [yy] * 2, color=color, lw=5, alpha=.4, solid_capstyle="butt", zorder=3)
        ax.scatter(end, yy, marker="D", s=55, color=color, zorder=7)
    else:
        ax.plot([start, end], [yy] * 2, color="#3386a5" if kind == "PLAN" else color,
                lw=9, solid_capstyle="butt", zorder=3)

def tl_check_text(fig, groups):
    """렌더링된 텍스트 경계로 행간 겹침·잘림을 확인한다."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for group in groups:
        boxes = [a.get_bbox_patch().get_window_extent(renderer) if hasattr(a, "get_bbox_patch") and a.get_bbox_patch() is not None
                 else a.get_window_extent(renderer) for a in group]
        if any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:]):
            raise RuntimeError("글자끼리 겹칩니다. 행 간격이나 열 너비를 조정하세요.")
        if any(b.x0 < 0 or b.x1 > fig.bbox.x1 or b.y0 < 0 or b.y1 > fig.bbox.y1 for b in boxes):
            raise RuntimeError("글자가 그림 경계를 넘습니다.")
    text_boxes = [a.get_window_extent(renderer) for a in fig.texts]
    if any(a.overlaps(b) for i, a in enumerate(text_boxes) for b in text_boxes[i + 1:]):
        raise RuntimeError("제목과 설명이 겹칩니다.")
    if any(b.x0 < 0 or b.x1 > fig.bbox.x1 or b.y0 < 0 or b.y1 > fig.bbox.y1 for b in text_boxes):
        raise RuntimeError("제목 또는 설명이 그림 경계를 넘습니다.")

In [ ]:
# %% 탐색 준비: 강종 기록 상태와 실제 센서 관측을 분리하여 질문을 확인한다.
IQ_FIG = Path(EVENT_DIR) / "02_관계사례"
IQ_FIG.mkdir(parents=True, exist_ok=True)
# 분석 설명과 표는 각 분석 구간의 마크다운 셀에 보존한다.


iq_events = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT").copy()
assert not iq_events.EVT_DT.duplicated().any()
assert iq_events.BEF_VAL.iloc[1:].reset_index(drop=True).eq(iq_events.AFT_VAL.iloc[:-1].reset_index(drop=True)).all()
iq_states = iq_events[["EVT_ID", "EVT_DT", "AFT_VAL"]].rename(columns={"EVT_DT": "START", "AFT_VAL": "GRADE"})
iq_states["END"] = iq_states.START.shift(-1).fillna(YEAR_END)
# 마지막 변경 기록을 다음 변경까지 유지한 '기록 상태'다. 실제 생산·가동 시간으로 해석하지 않는다.
iq_states["HOURS"] = (iq_states.END.clip(upper=YEAR_END) - iq_states.START.clip(lower=YEAR_START)).dt.total_seconds() / 3600
iq_work = tl_maint.copy()
for time_col, prefix in [("DETC_DT", "DETECT"), ("STRT_DT", "START")]:
    match = pd.merge_asof(iq_work[["WO_NO", time_col]].sort_values(time_col),
                          iq_events[["EVT_DT", "AFT_VAL"]], left_on=time_col, right_on="EVT_DT", direction="backward")
    iq_work[prefix + "_GRADE"] = iq_work.WO_NO.map(match.set_index("WO_NO").AFT_VAL)
iq_bm = iq_work.loc[iq_work.MNT_TYPE.eq("BM")].copy()
iq_grade_summary = (iq_states.groupby("GRADE").agg(기록상태_시간=("HOURS", "sum"), 변경기록=("EVT_ID", "size"))
                    .join(iq_bm.groupby("START_GRADE").size().rename("BM_착수건수"))
                    .join(iq_bm.groupby("DETECT_GRADE").size().rename("BM_감지건수"))
                    .fillna({"BM_착수건수": 0, "BM_감지건수": 0}).reset_index())
iq_grade_summary[["BM_착수건수", "BM_감지건수"]] = iq_grade_summary[["BM_착수건수", "BM_감지건수"]].astype(int)
iq_bm_hits = tl_overlap.loc[tl_overlap["종류_A"].eq("LOT") & tl_overlap["종류_B"].eq("BM")]
iq_gap_start, iq_gap_end = pd.Timestamp("2024-02-01"), pd.Timestamp("2024-03-07 21:22")
iq_gap_events = iq_events.loc[(iq_events.EVT_DT >= iq_gap_start) & (iq_events.EVT_DT < iq_gap_end)]
iq_gap_work = iq_work.loc[(iq_work.STRT_DT < iq_gap_end) & (iq_work.END_DT >= iq_gap_start)]
iq_gap_temp = temp.loc[(temp.MEAS_DT >= iq_gap_start) & (temp.MEAS_DT < iq_gap_end)]
iq_gap_weld = wld.loc[(wld.MEAS_DT >= iq_gap_start) & (wld.MEAS_DT < iq_gap_end)]
iq_plan_changes = []
for p in tl_plans.itertuples():
    changes = iq_events.loc[(iq_events.EVT_DT >= p.STRT_DT) & (iq_events.EVT_DT < p.END_EXCLUSIVE)].copy()
    changes.insert(0, "PLAN_ID", p.PLAN_ID)
    iq_plan_changes.append(changes)
iq_plan_changes = pd.concat(iq_plan_changes, ignore_index=True)


In [ ]:
# %% 소둔로 연결 대조: 정비와 교체는 CAL01 태그/작업번호로 한정한다.
CAL_TAG='P1-CR1-CAL01'
assert eqp.set_index('EQP_TAG').loc[CAL_TAG,'EQP_NM']=='연속 소둔로'
audit_work=maint.loc[maint.EQP_TAG.eq(CAL_TAG)&(maint.STRT_DT<YEAR_END)&(maint.END_DT>=YEAR_START)].copy()
audit_work['SCOPE']='CAL01'
assert audit_work.WO_NO.is_unique and len(audit_work)==44
audit_parts=(partchg.merge(audit_work[['WO_NO','EQP_TAG','SCOPE','MNT_TYPE','DETC_DT','STRT_DT','END_DT','FAIL_MODE','REMARK']],
    on='WO_NO',how='inner',validate='many_to_one').merge(part,on='PART_CD',validate='many_to_one'))
assert len(audit_parts)==9 and audit_parts.WO_NO.nunique()==8
assert set(audit_parts.WO_NO)==set(audit_work.loc[audit_work.PART_CHG.eq('Y'),'WO_NO'])
audit_pairs=[]
def audit_hit(kind_a,id_a,eqp_a,a0,a1,kind_b,id_b,eqp_b,b0,b1):
    lo,hi=max(a0,b0),min(a1,b1)
    if hi>=lo:
        audit_pairs.append(dict(KIND_A=kind_a,ID_A=str(id_a),EQP_A=eqp_a,KIND_B=kind_b,ID_B=str(id_b),EQP_B=eqp_b,
            START=lo,END=hi,MINUTES=(hi-lo).total_seconds()/60,RELATION='시간 겹침' if hi>lo else '경계 시각 일치'))
for lot in tl_lots.itertuples():
    for task in audit_work.loc[(audit_work.STRT_DT<=lot.LOT_END)&(audit_work.END_DT>=lot.LOT_START)].itertuples():
        audit_hit('LOT',int(lot.LOT_NO),CAL_TAG,lot.LOT_START,lot.LOT_END,task.MNT_TYPE,task.WO_NO,CAL_TAG,task.STRT_DT,task.END_DT)
    for coil in tl_wld.loc[(tl_wld.WLD_START<=lot.LOT_END)&(tl_wld.WLD_END>=lot.LOT_START)].itertuples():
        audit_hit('LOT',int(lot.LOT_NO),CAL_TAG,lot.LOT_START,lot.LOT_END,'WLD',coil.COIL_ID,'P1-CR1-WLD01',coil.WLD_START,coil.WLD_END)
    for p in tl_plans.itertuples():
        if lot.LOT_START<p.END_EXCLUSIVE and lot.LOT_END>=p.STRT_DT:
            audit_hit('LOT',int(lot.LOT_NO),CAL_TAG,lot.LOT_START,lot.LOT_END,'계획수리',p.PLAN_ID,'CR1',p.STRT_DT,p.END_EXCLUSIVE)
tasks=list(audit_work.itertuples())
for i,a in enumerate(tasks):
    for b in tasks[i+1:]:
        audit_hit(a.MNT_TYPE,a.WO_NO,CAL_TAG,a.STRT_DT,a.END_DT,b.MNT_TYPE,b.WO_NO,CAL_TAG,b.STRT_DT,b.END_DT)
    for c in tl_wld.loc[(tl_wld.WLD_START<=a.END_DT)&(tl_wld.WLD_END>=a.STRT_DT)].itertuples():
        audit_hit(a.MNT_TYPE,a.WO_NO,CAL_TAG,a.STRT_DT,a.END_DT,'WLD',c.COIL_ID,'P1-CR1-WLD01',c.WLD_START,c.WLD_END)
    for p in tl_plans.itertuples():
        if a.STRT_DT<p.END_EXCLUSIVE and a.END_DT>=p.STRT_DT:
            audit_hit(a.MNT_TYPE,a.WO_NO,CAL_TAG,a.STRT_DT,a.END_DT,'계획수리',p.PLAN_ID,'CR1',p.STRT_DT,p.END_EXCLUSIVE)
audit_overlap=pd.DataFrame(audit_pairs)
observed={('LOT',str(int(k))):pd.DatetimeIndex(g.MEAS_DT).as_unit('ns').asi8 for k,g in temp.sort_values('MEAS_DT').groupby('LOT_NO')}
observed.update({('WLD',str(k)):pd.DatetimeIndex(g.MEAS_DT).as_unit('ns').asi8 for k,g in wld.sort_values('MEAS_DT').groupby('COIL_ID')})
for side in ['A','B']:
    counts=[]
    for hit in audit_overlap.itertuples():
        times=observed.get((getattr(hit,'KIND_'+side),getattr(hit,'ID_'+side)))
        counts.append(pd.NA if times is None else int(np.searchsorted(times,hit.END.value,side='right')-np.searchsorted(times,hit.START.value,side='left')))
    audit_overlap['RECORDED_'+side]=pd.array(counts,dtype='Int64')
audit_points=[]
for lot in tl_lots.itertuples():
    for t in audit_work.loc[audit_work.DETC_DT.between(lot.LOT_START,lot.LOT_END)].itertuples():
        audit_points.append(dict(LOT_NO=int(lot.LOT_NO),EVENT_KIND='소둔로 감지',EQP_TAG=CAL_TAG,ID=t.WO_NO,TIME=t.DETC_DT,DETAIL=t.MNT_TYPE))
    for e in iq_events.loc[iq_events.EVT_DT.between(lot.LOT_START,lot.LOT_END)].itertuples():
        audit_points.append(dict(LOT_NO=int(lot.LOT_NO),EVENT_KIND='강종변경 기록',EQP_TAG='CR1',ID=e.EVT_ID,TIME=e.EVT_DT,DETAIL=f'{e.BEF_VAL}→{e.AFT_VAL}'))
def audit_union_minutes(intervals):
    merged=[]
    for a,b in sorted(intervals):
        if b<=a:continue
        if merged and a<=merged[-1][1]:merged[-1][1]=max(b,merged[-1][1])
        else:merged.append([a,b])
    return sum((b-a).total_seconds()/60 for a,b in merged)
audit_rows=[]
for lot in tl_lots.itertuples():
    hits=audit_overlap.loc[audit_overlap.KIND_A.eq('LOT')&audit_overlap.ID_A.eq(str(int(lot.LOT_NO)))&audit_overlap.MINUTES.gt(0)]
    work=hits.loc[hits.EQP_B.eq(CAL_TAG)&hits.KIND_B.isin(['BM','CBM','TBM'])]
    detect=audit_work.loc[audit_work.DETC_DT.between(lot.LOT_START,lot.LOT_END)]
    audit_rows.append(dict(LOT_NO=int(lot.LOT_NO),GRADE=lot.GRADE,START=lot.LOT_START,END=lot.LOT_END,
        CAL01_WORK=work.ID_B.nunique(),CAL01_WORK_UNION_MIN=audit_union_minutes(list(zip(work.START,work.END))),
        CAL01_DETECT=len(detect),CAL01_PART_WO=audit_parts.loc[audit_parts.WO_NO.isin(work.ID_B),'WO_NO'].nunique(),
        DETECT_LINKED_PART_WO=audit_parts.loc[audit_parts.WO_NO.isin(detect.WO_NO),'WO_NO'].nunique(),
        WELD_COIL=hits.loc[hits.KIND_B.eq('WLD'),'ID_B'].nunique(),PLAN=hits.loc[hits.KIND_B.eq('계획수리'),'ID_B'].nunique()))
audit_lots=pd.DataFrame(audit_rows)
audit_work['TEMP_LOTS']=audit_work.WO_NO.map(audit_overlap.loc[audit_overlap.KIND_A.eq('LOT')&audit_overlap.MINUTES.gt(0)].groupby('ID_B').ID_A.nunique()).fillna(0).astype(int)
audit_work['WELD_COILS']=audit_work.WO_NO.map(audit_overlap.loc[audit_overlap.KIND_B.eq('WLD')&audit_overlap.MINUTES.gt(0)].groupby('ID_A').ID_B.nunique()).fillna(0).astype(int)
assert audit_lots.CAL01_WORK.sum()==2 and audit_lots.WELD_COIL.sum()==82
assert set(audit_lots.loc[audit_lots.CAL01_WORK.gt(0),'LOT_NO'])=={240117}
audit_temp_work=audit_overlap.loc[audit_overlap.KIND_A.eq('LOT')&audit_overlap.KIND_B.isin(['BM','CBM','TBM'])&audit_overlap.MINUTES.gt(0)].merge(
    audit_work[['WO_NO','REMARK']],left_on='ID_B',right_on='WO_NO',validate='many_to_one')


<a id="analysis-overlap"></a>

### 소둔로 작업·부품·관측의 연결

#### 소둔로 작업과 관측의 연결

정비는 설비마스터의 **연속 소둔로 P1-CR1-CAL01**에 등록된 44건(BM 12·CBM 10·TBM 22)만 사용한다. 설비마스터에는 CAL01의 자식 설비가 등록돼 있지 않다. 부품교체는 이 44건의 WO_NO로 연결한 9행·8작업이다.

이전에 '다른 CR1 작업'으로 표시했던 PKL·TCM·SPM·펌프·팬 등은 같은 냉연 라인의 다른 설비 작업이었다. 시간상 겹친다는 이유로 소둔로 작업에 포함할 근거가 없어 이번 분석에서 제외했다. 용접기 자체 정비도 제외한다. 용접 CSV는 소둔로 관측·작업과 같은 시각의 계측 자료로만 대조한다.

##### CSV 사이의 연결 기준

- 설비마스터 EQP_TAG → 정비이력 EQP_TAG: **소둔로 소속을 확정**한다.
- 정비이력 WO_NO → 부품교체이력 WO_NO → 부품마스터 PART_CD: **어떤 작업에 어떤 부품·수량·사유가 연결되는지 확정**한다.
- 정비 착수·완료 ↔ 온도 LOT 관측: 실제 공통 시간만 구한다. 감지는 별도 시점이다.
- 강종변경·계획수리: CR1 캘린더의 시간 맥락이다. 6월 SCOPE는 '연속소둔로 노내 정비', 11월은 '냉연 라인 전 설비 정기수리'다. 11월 수리 안에서도 작업은 CAL01 5건만 선택한다.
- 용접 COIL ↔ 온도 LOT: 동일 시각에 계측된 관계다. 소재 추적 키가 없어 동일 소재 또는 원인 관계로 연결하지 않는다.

##### 그림 1B: 사건이 있는 두 시간창을 신호와 함께 보기

**240117 / 11월 13일:** 온도 관측 안에 TBM 01859(07:40–09:35)·CBM 01854(08:54–12:44)가 있다. 작업끼리 겹친 시간은 41분, 두 작업 구간의 합집합은 304분이다. 초기 TC-Z3~TC-Z6 승온은 작업 착수 전부터 관측된다. TBM 감지는 07:13, CBM 감지는 전날 15:00이다. 용접 31 COIL은 시간상 동시 관측이다. 두 작업에 연결된 부품교체 이력은 없다.

**240133 / 12월 28일:** 온도 관측은 14:57–16:44:24, CBM 02105 감지는 15:54, 착수는 16:54, 완료는 23:33이다. 감지는 LOT 안이지만 실제 작업은 관측 종료 9.6분 뒤다. 이 CBM에 가스 분석계 셀 INS-006 3개·예방 교체 이력이 연결된다. 가스 분석계 셀과 CP-Z4가 같은 센서라는 식별 정보는 없다. CP와 냉각 신호는 작업 전 관측을 확인하기 위한 것이며 교체 효과를 입증하지 않는다. 작업 후 온도가 없어 전후 효과는 평가할 수 없다.

##### 그림 1C: 계획수리 → 소둔로 작업 → 부품 → 다음 LOT

6/10–6/14 수리 기간에 소둔로 TBM 6건이 있다. 6/10 WO-2024-00916에 오일씰 SEL-006 2개(수명도래)와 O링 SEL-009 1개(손상)가 동시에 연결된다. REMARK는 기록 없음이다. 이는 같은 WO 연결이며 두 부품의 정확한 교체 시각이나 상호 원인을 뜻하지 않는다.

수리 종료 후 64분에 첫 온도 LOT 240061(SPHC)이 관측된다. 수리 전 마지막 240058은 DP590이므로 수리 효과의 직접 전후 비교가 아니다. 240061의 초기 승온은 다른 SPHC LOT와 비교한다. 6월 수리 안에는 온도·용접 관측이 없다.

##### 관측 겹침의 의미

겹침은 두 기록 구간에 공통 시간이 있다는 뜻이다. 작업 길이는 정지시간 DOWN_MIN과 다르다. 부품은 WO 연결이며 정확한 교체 시각을 임의로 부여하지 않는다. 0건인 LOT도 아래 대조표에 남긴다.


##### 온도 관측과 실제 겹친 소둔로 작업 전부

| ID_A | KIND_B | ID_B | START | END | MINUTES | REMARK |
| --- | --- | --- | --- | --- | --- | --- |
| 240117 | CBM | WO-2024-01854 | 11/13 08:54:00 | 11/13 12:44:00 | 230.0 | 마모량 초과 |
| 240117 | TBM | WO-2024-01859 | 11/13 07:40:00 | 11/13 09:35:00 | 115.0 | 계획 점검 수행 |

##### 전체 14 LOT 대조

| LOT_NO | GRADE | START | END | 소둔로 작업 | CAL01_WORK_UNION_MIN | 소둔로 감지 | 작업 연결 교체 WO | 감지 연결 교체 WO | 동시 용접 COIL | PLAN |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 240001 | SGCC | 01/02 03:36:00 | 01/02 15:04:30 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240024 | SPFH590 | 03/07 21:22:00 | 03/08 06:29:48 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240037 | SPCE | 04/13 23:16:00 | 04/14 09:27:19 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240038 | SPCEN | 04/16 11:35:00 | 04/16 13:39:56 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240044 | SPHC | 05/03 18:50:00 | 05/03 20:04:38 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240058 | DP590 | 06/09 05:26:00 | 06/09 09:02:53 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240061 | SPHC | 06/15 01:04:00 | 06/15 04:49:23 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240069 | SPHC | 07/07 16:57:00 | 07/08 06:29:34 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240091 | SPHC | 09/08 22:03:00 | 09/09 08:44:12 | 0 | 0.0 | 0 | 0 | 0 | 0 | 0 |
| 240108 | SPFH590 | 10/18 04:05:00 | 10/18 11:50:53 | 0 | 0.0 | 0 | 0 | 0 | 12 | 0 |
| 240114 | SPCEN | 11/04 15:40:00 | 11/04 20:59:02 | 0 | 0.0 | 0 | 0 | 0 | 11 | 0 |
| 240117 | SPCD | 11/13 07:08:00 | 11/13 14:32:19 | 2 | 304.0 | 1 | 0 | 0 | 31 | 0 |
| 240121 | SPCE | 11/24 07:31:00 | 11/24 18:22:12 | 0 | 0.0 | 0 | 0 | 0 | 28 | 0 |
| 240133 | SPCE | 12/28 14:57:00 | 12/28 16:44:24 | 0 | 0.0 | 1 | 0 | 1 | 0 | 0 |

##### 소둔로 작업과 연결된 모든 부품 이력

| WO_NO | MNT_TYPE | DETC_DT | STRT_DT | END_DT | PART_CD | PART_NM | QTY | CHG_RSN | FAIL_MODE | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00508 | BM | 03/24 14:59:00 | 03/24 15:56:00 | 03/24 23:36:00 | SEL-006 | 오일씰 150x190x18 | 2 | 손상 | ELE | 인버터 알람 |
| WO-2024-00597 | TBM | 04/09 14:55:00 | 04/09 17:00:00 | 04/09 19:09:00 | SEL-009 | O링 세트 | 3 | 수명도래 | — |   정기점검검 - 이상없음  |
| WO-2024-00776 | CBM | 05/14 07:17:00 | 05/19 12:14:00 | 05/19 16:05:00 | INS-006 | 가스 분석계 셀 | 3 | 수명도래 | ETC | 재점검 필요 |
| WO-2024-00916 | TBM | 06/10 06:23:00 | 06/10 08:45:00 | 06/10 10:48:00 | SEL-006 | 오일씰 150x190x18 | 2 | 수명도래 | — | — |
| WO-2024-00916 | TBM | 06/10 06:23:00 | 06/10 08:45:00 | 06/10 10:48:00 | SEL-009 | O링 세트 | 1 | 손상 | — | — |
| WO-2024-01094 | BM | 07/08 08:49:00 | 07/08 11:11:00 | 07/08 19:04:00 | SEL-009 | O링 세트 | 3 | 예방 | ELE | 1호기 재발, 절연 저하 |
| WO-2024-01178 | BM | 07/24 23:33:00 | 07/25 01:19:00 | 07/25 10:55:00 | INS-006 | 가스 분석계 셀 | 1 | 손상 | ELE | — |
| WO-2024-01346 | BM | 08/27 07:37:00 | 08/27 11:03:00 | 08/28 04:27:00 | SEL-006 | 오일씰 150x190x18 | 1 | 손상 | ELE | 1호기 재발, 모터 과열 |
| WO-2024-02105 | CBM | 12/28 15:54:00 | 12/28 16:54:00 | 12/28 23:33:00 | INS-006 | 가스 분석계 셀 | 3 | 예방 | WER | 마모량 초과 |

[소둔로 연결 관계](../figures/05_all_data_timeseries_jinny_01/05_사건연결/02_관계사례/01B_소둔로_관측_정비_겹침.png)

[소둔로 연결 관계](../figures/05_all_data_timeseries_jinny_01/05_사건연결/02_관계사례/01C_계획수리_작업_부품_LOT_연결.png)


In [ ]:
# %% 그림 1B: 소둔로의 관측·감지·작업·교체 연결을 사건별로 보여준다.
from matplotlib.patches import ConnectionPatch
ann_red='#c52e43';ann_blue='#2379ae';ann_purple='#7951ad';ann_teal='#147a83'
ann_sensor_colors=['#007eb6','#e68117','#14856d','#ba416c']
ann_case_evidence=[]

def ann_style(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#b8c5cf')
    ax.grid(axis='both',color='#e7edf2',lw=.65);ax.set_axisbelow(True)
    ax.tick_params(labelsize=9,color='#b8c5cf')

def ann_temp_line(ax,lot,sensors):
    raw=temp.loc[temp.LOT_NO.eq(lot)].copy()
    minute=raw.set_index('MEAS_DT')[sensors].resample('1min').median()
    counts=raw.set_index('MEAS_DT')[sensors[0]].resample('1min').count()
    minute=minute.loc[counts.gt(0)]
    for sensor,color in zip(sensors,ann_sensor_colors):ax.plot(minute.index,minute[sensor],color=color,lw=1.4,label=sensor)
    ax.set_ylabel('CP' if sensors[0].startswith('CP') else '온도 [℃]',fontsize=10)
    ax.legend(loc='upper right',ncol=min(len(sensors),2),fontsize=8,frameon=False)
    return raw

def ann_box(ax,stamp,value,text,offset=(12,12)):
    ax.scatter(stamp,value,s=35,facecolors='white',edgecolors=ann_red,zorder=9)
    return ax.annotate(text,(stamp,value),xytext=offset,textcoords='offset points',fontsize=9,color=ann_red,
        ha='left' if offset[0]>=0 else 'right',va='bottom' if offset[1]>=0 else 'top',
        bbox=dict(boxstyle='round,pad=.3',fc='white',ec=ann_red,lw=.8),
        arrowprops=dict(arrowstyle='-',color=ann_red,lw=.8),zorder=8)

fig=plt.figure(figsize=(23,11))
grid=fig.add_gridspec(4,2,height_ratios=[.45,1.6,1.05,1.15],left=.08,right=.98,top=.82,bottom=.08,hspace=.48,wspace=.30)
left=[fig.add_subplot(grid[r,0]) for r in range(4)]
right=[fig.add_subplot(grid[r,1]) for r in range(4)]
for col,lot_id,start,end in [(left,240117,pd.Timestamp('2024-11-13 07:00'),pd.Timestamp('2024-11-13 14:45')),
    (right,240133,pd.Timestamp('2024-12-28 14:40'),pd.Timestamp('2024-12-28 23:45'))]:
    info=tl_lots.set_index('LOT_NO').loc[lot_id]
    states=iq_states.loc[(iq_states.START<end)&(iq_states.END>start)]
    for state in states.itertuples():
        a,b=max(start,state.START),min(end,state.END)
        col[0].plot([a,b],[0,0],lw=18,color='#dce7ee',solid_capstyle='butt')
        if (b-a)>pd.Timedelta(minutes=20):col[0].text(a+(b-a)/2,0,state.GRADE,ha='center',va='center',fontsize=10,color='#33566f')
    col[0].set_yticks([0],['강종 기록']);col[0].set_ylim(-.8,.8)
    for ax in col:
        ax.set_xlim(start,end);ann_style(ax)
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1 if lot_id==240117 else 2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        if ax is not col[-1]:ax.tick_params(axis='x',labelbottom=False)
    for ax in col[1:3]:
        ax.axvline(info.LOT_START,color='#a5b9c4',lw=.8,ls=':')
        ax.axvline(info.LOT_END,color='#a5b9c4',lw=.8,ls=':')
    col[0].set_title(f'LOT {lot_id} · {info.GRADE} · {start:%Y/%m/%d}',loc='left',fontsize=13,weight='bold',pad=14,color='#17344d')

# 240117: 신호 변화가 작업 경계보다 먼저인지, 작업끼리 겹치는지 확인.
raw=ann_temp_line(left[1],240117,['TC-Z3','TC-Z4','TC-Z5','TC-Z6'])
left[1].set_title('가열·균열 [TC-Z3~TC-Z6]',loc='left',fontsize=10,pad=8)
work117=audit_work.loc[audit_work.WO_NO.isin(['WO-2024-01859','WO-2024-01854'])].sort_values('STRT_DT')
both_start,both_end=max(work117.STRT_DT),min(work117.END_DT)
for ax in left[1:]:ax.axvspan(both_start,both_end,color=ann_red,alpha=.10,zorder=0)
first_minute=raw.set_index('MEAS_DT')['TC-Z3'].resample('1min').median().dropna()
ann_box(left[1],first_minute.index[0],first_minute.iloc[0],'TC-Z3 초기 승온\nTBM 착수 전부터 관측',(14,12))
coils=tl_wld.loc[(tl_wld.WLD_START>=raw.MEAS_DT.min())&(tl_wld.WLD_START<=raw.MEAS_DT.max())]
edges=pd.date_range('2024-11-13 07:00','2024-11-13 14:50',freq='10min')
counts,_=np.histogram(mdates.date2num(coils.WLD_START),bins=mdates.date2num(edges))
left[2].bar(edges[:-1],counts,width=10/1440*.85,align='edge',color='#de971b',alpha=.8)
left[2].set_ylabel('COIL 수',fontsize=10)
left[2].set_title('WLD01 관측 · 10분별 시작 COIL 수 · 31 COIL',loc='left',fontsize=10,pad=8)
left[2].yaxis.set_major_locator(plt.MaxNLocator(integer=True,nbins=3))
for y,task,color in zip([1,0],work117.itertuples(),[ann_blue,ann_red]):
    left[3].plot([task.STRT_DT,task.END_DT],[y,y],color=color,lw=3,marker='x',ms=7,mew=1.6)
left[3].scatter(pd.Timestamp('2024-11-13 07:13'),1,marker='D',s=34,facecolors='white',edgecolors=ann_blue,zorder=6)
left[3].set_yticks([1,0],['TBM 01859','CBM 01854'],fontsize=9)
left[3].set_ylim(-.7,1.8)
left[3].text(both_start+(both_end-both_start)/2,1.60,'41분 겹침',ha='center',va='center',color=ann_red,fontsize=10,
    bbox=dict(fc='white',ec='#eac1c6',boxstyle='round,pad=.25'))
left[3].set_title('CAL01 작업 · × 착수/완료 · ◇ 감지',loc='left',fontsize=10,pad=8)

# 240133: 감지→관측 종료→착수→부품 WO 연결. 없는 작업 후 온도를 만들지 않는다.
raw133=ann_temp_line(right[1],240133,['CP-Z4'])
right[1].set_title('가열·균열 [CP-Z4]',loc='left',fontsize=10,pad=8)
ann_temp_line(right[2],240133,['TC-OUT1','TC-OUT2'])
right[2].set_title('냉각 [TC-OUT1·TC-OUT2]',loc='left',fontsize=10,pad=8)
task=audit_work.set_index('WO_NO').loc['WO-2024-02105']
lot_end=raw133.MEAS_DT.max();gap=(task.STRT_DT-lot_end).total_seconds()/60
for ax in right[1:]:
    ax.axvline(task.DETC_DT,color=ann_red,ls=':',lw=1)
    ax.axvspan(task.STRT_DT,task.END_DT,color=ann_purple,alpha=.07,zorder=0)
right[1].text(.59,.40,'작업 후 온도 관측 없음',transform=right[1].transAxes,ha='center',color='#6e7f8a',fontsize=11)
right[3].plot([task.STRT_DT,task.END_DT],[0,0],color=ann_red,lw=3,marker='x',ms=7,mew=1.6)
right[3].plot([raw133.MEAS_DT.min(),lot_end],[1,1],color=ann_teal,lw=4)
right[3].scatter(task.DETC_DT,0,marker='D',s=42,facecolors='white',edgecolors=ann_red,zorder=6)
ann_box(right[3],task.DETC_DT,0,'15:54 감지',(-10,-13))
right[3].set_yticks([1,0],['LOT 240133','CBM 02105'],fontsize=9)
right[3].set_ylim(-1.25,1.8)
right[3].text(task.STRT_DT+(task.END_DT-task.STRT_DT)/2,.46,'↳ 가스 분석계 셀 INS-006 · 3개 · 예방',ha='center',color=ann_purple,fontsize=9.5,
    bbox=dict(fc='white',ec=ann_purple,boxstyle='round,pad=.25',lw=.6))
right[3].text(lot_end+(task.STRT_DT-lot_end)/2,1.55,f'관측 종료 → {gap:g}분 뒤 착수',ha='center',fontsize=9,color=ann_red,
    bbox=dict(fc='white',ec='none',pad=2))
right[3].set_title('CAL01 감지·작업과 작업번호로 연결된 교체 기록',loc='left',fontsize=10,pad=8)
fig.suptitle('그림 1B. 소둔로에서 무엇이 함께 기록됐는가',x=.025,y=.98,ha='left',fontsize=18,weight='bold',color='#17344d')
fig.text(.025,.925,'11/13: 온도·용접 관측과 작업 2건  |  12/28: 관측 중 감지 → 관측 종료 뒤 작업·교체',fontsize=11,color='#456176')
fig.savefig(IQ_FIG/'01B_소둔로_관측_정비_겹침.png',dpi=160,facecolor='white');plt.show()
assert len(coils)==31 and (both_end-both_start).total_seconds()/60==41 and np.isclose(gap,9.6)
ann_case_evidence.extend([dict(CASE='240117',WORK_COUNT=2,WELD_COILS=31,WORK_MUTUAL_MIN=41),
    dict(CASE='240133',WORK_COUNT=0,DETECT_COUNT=1,WORK_AFTER_OBSERVATION_MIN=gap,PART='INS-006',PART_QTY=3)])

# 그림 1C: 달력 시간과 정확한 작업번호 연결을 한 화면에서 읽는다.
plan=tl_plans.loc[tl_plans.PLAN_ID.eq('PLAN-2024-02')].iloc[0]
start,end=pd.Timestamp('2024-06-08'),pd.Timestamp('2024-06-16')
tasks=audit_work.loc[(audit_work.STRT_DT<plan.END_EXCLUSIVE)&(audit_work.END_DT>=plan.STRT_DT)].sort_values('STRT_DT')
assert len(tasks)==6 and tasks.MNT_TYPE.eq('TBM').all()
linked=audit_parts.loc[audit_parts.WO_NO.eq('WO-2024-00916')]
assert set(linked.PART_CD)=={'SEL-006','SEL-009'}
fig,ax=plt.subplots(figsize=(23,8))
fig.subplots_adjust(left=.095,right=.985,top=.76,bottom=.12)
ax.set_xlim(start,end);ax.set_ylim(-2.7,2.7)
states=iq_states.loc[(iq_states.START<end)&(iq_states.END>start)]
labels=[]
for state in states.itertuples():
    a,b=max(start,state.START),min(end,state.END)
    ax.plot([a,b],[2.15,2.15],color='#dce7ee',lw=21,solid_capstyle='butt')
    labels.append(ax.text(a+(b-a)/2,2.15,state.GRADE,ha='center',va='center',color='#33566f',fontsize=10))
for change in iq_events.loc[iq_events.EVT_DT.between(start,end)].itertuples():
    ax.scatter(change.EVT_DT,2.42,marker='v',color='#447ba3',s=23)
for i,task in enumerate(tasks.itertuples()):
    ax.plot([task.STRT_DT,task.END_DT],[1.25,1.25],color=ann_blue,lw=3,marker='x',ms=6)
    stamp=task.STRT_DT+(task.END_DT-task.STRT_DT)/2
    labels.append(ax.text(stamp,1.60 if i%2==0 else .99,f'TBM {task.WO_NO[-5:]}',ha='center',va='center',fontsize=9,color=ann_blue))
work=audit_work.set_index('WO_NO').loc['WO-2024-00916']
mid=work.STRT_DT+(work.END_DT-work.STRT_DT)/2
ax.plot([mid,mid],[1.25,-.37],color=ann_purple,ls=':',lw=1.2)
for yy,p in zip([.30,-.37],linked.itertuples()):
    ax.scatter(mid,yy,color=ann_purple,s=30,marker='D',zorder=6)
    labels.append(ax.annotate(f'{p.PART_NM} · {p.PART_CD} · {p.QTY}개 · {p.CHG_RSN}',
        (mid,yy),xytext=(14,0),textcoords='offset points',ha='left',va='center',fontsize=10,color=ann_purple,
        bbox=dict(boxstyle='round,pad=.3',fc='white',ec=ann_purple,lw=.7)))
ax.text(.012,.55,'같은 WO 00916',transform=ax.transAxes,color=ann_purple,fontsize=9)
for lot_id in [240058,240061]:
    lot=tl_lots.set_index('LOT_NO').loc[lot_id]
    ax.plot([lot.LOT_START,lot.LOT_END],[-1.25,-1.25],color=ann_teal,lw=4,marker='o',ms=3)
    if lot_id==240058:
        labels.append(ax.annotate(f'LOT {lot_id} · {lot.GRADE}\n{lot.LOT_START:%m/%d %H:%M}–{lot.LOT_END:%H:%M}',
            (lot.LOT_END,-1.25),xytext=(10,0),textcoords='offset points',va='center',fontsize=10,color=ann_teal))
    else:
        gap=(lot.LOT_START-plan.END_EXCLUSIVE).total_seconds()/60
        labels.append(ax.annotate(f'LOT {lot_id} · {lot.GRADE}\n수리 종료 후 {gap:g}분 · 초기 승온 확인',
            (lot.LOT_START,-1.25),xytext=(0,21),textcoords='offset points',ha='center',va='bottom',fontsize=10,color=ann_red,
            bbox=dict(boxstyle='round,pad=.3',fc='white',ec=ann_red,lw=.7),arrowprops=dict(arrowstyle='-',color=ann_red)))
ax.fill_between([plan.STRT_DT,plan.END_EXCLUSIVE],[-2.25]*2,[-1.95]*2,facecolor='#dceef5',edgecolor='#89b1c3',hatch='///',lw=.5)
ax.text(plan.STRT_DT+(plan.END_EXCLUSIVE-plan.STRT_DT)/2,-2.1,'06/10–06/14 · 연속소둔로 노내 정비',ha='center',va='center',fontsize=10,color='#28667f',
    bbox=dict(fc='white',ec='none',pad=2))
ax.axvline(plan.END_EXCLUSIVE,color='#82aebf',ls=':',lw=.9)
ax.set_yticks([2.15,1.25,0,-1.25,-2.1],['강종 기록','CAL01 작업','작업번호 연결 부품','온도 LOT','CR1 계획수리'])
ax.xaxis.set_major_locator(mdates.DayLocator());ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ann_style(ax)
fig.suptitle('그림 1C. 6월 계획수리와 소둔로 작업·부품·다음 LOT의 연결',x=.025,y=.98,ha='left',fontsize=18,weight='bold',color='#17344d')
fig.text(.025,.91,'WO 00916 → 오일씰 + O링 · 수리 중 강종 기록 DP590 → SPCC → DP590 · 수리 후 첫 관측 240061(SPHC)',fontsize=11,color='#456176')
fig.savefig(IQ_FIG/'01C_계획수리_작업_부품_LOT_연결.png',dpi=160,facecolor='white');plt.show()
ann_case_evidence.append(dict(CASE='PLAN-2024-02',WORK_COUNT=6,LINKED_WO='WO-2024-00916',PART_ROWS=2,NEXT_LOT=240061,NEXT_LOT_GAP_MIN=64))


### 그림 1B·1C에서 확인한 연결

- **240117:** 초기 승온은 작업 착수 전부터 있다. 소둔로 TBM·CBM은 41분 겹치며 같은 시간창에 용접 31 COIL이 관측됐다.
- **240133:** 관측 중 감지 → 관측 종료 9.6분 뒤 CBM 착수 → 같은 WO의 가스 분석계 셀 교체 이력이 연결된다. 작업 후 온도는 없다.
- **6월 수리:** CAL01 TBM 6건 → WO 00916에 오일씰·O링 연결 → 수리 종료 64분 뒤 첫 LOT 240061 관측이다. 다른 강종의 전후 LOT로 수리 효과를 단정하지 않는다.

CSV의 작업번호로 연결된 관계와 같은 시각에 관측된 관계를 구분한다. 전체 14 LOT의 0건 결과와 정확한 REMARK는 앞의 마크다운에 보존한다.


In [ ]:
# %% 그림 1: CSV 색상, 실제 시간 연결, 읽을 수 있는 정비 구간.
# 색은 데이터 출처를 나타내며 겹침 여부는 별도 연결선으로 나타낸다.
fig, ax = plt.subplots(figsize=(24, 11.8))
fig.subplots_adjust(left=.105, right=.987, top=.845, bottom=.17)
Y = {"LOT": 9.1, "GRADE": 8.15, "WLD": 7.2, "BM": 6.15, "CBM": 5.25, "TBM": 4.35}
part_codes = sorted(tl_parts["PART_CD"].unique())
part_y = {code: 3.25 - .78 * i for i, code in enumerate(part_codes)}
plan_y = .25
source_rows = [(Y["LOT"], TL_COLORS["LOT"]), (Y["GRADE"], TL_COLORS["GRADE"]),
               (Y["WLD"], TL_COLORS["WLD"])]
source_rows += [(Y[k], TL_COLORS[k]) for k in ["BM", "CBM", "TBM"]]
source_rows += [(part_y[k], TL_PART_COLORS[k]) for k in part_codes]
source_rows += [(plan_y, "#3386a5")]
for yy, color in source_rows:
    ax.axhspan(yy - .34, yy + .34, color=color, alpha=.045, lw=0, zorder=0)
for p in tl_plans.itertuples():
    ax.axvspan(p.STRT_DT, p.END_EXCLUSIVE, color=TL_COLORS["PLAN"], zorder=0)
    ax.plot([p.STRT_DT, p.END_EXCLUSIVE], [plan_y] * 2, color="#3386a5", lw=9, solid_capstyle="butt")
    ax.text(p.STRT_DT + (p.END_EXCLUSIVE - p.STRT_DT) / 2, plan_y - .25,
            f"{p.STRT_DT:%m/%d}–{p.END_DT:%m/%d}\n{p.SCOPE}", ha="center", va="top",
            fontsize=9, color="#28667f", linespacing=1.45)

ax.vlines(tl_grades["EVT_DT"], Y["GRADE"] - .075, Y["GRADE"] + .075,
          color=TL_COLORS["GRADE"], alpha=.35, lw=.8)
for lot in tl_lots.itertuples():
    ax.plot([lot.LOT_START, lot.LOT_END], [Y["LOT"]] * 2, color=TL_COLORS["LOT"], lw=5)
    ax.scatter(lot.LOT_START, Y["LOT"], color=TL_COLORS["LOT"], s=23, zorder=5)
    if tl_grades["EVT_DT"].eq(lot.LOT_START).any():
        ax.plot([lot.LOT_START] * 2, [Y["GRADE"], Y["LOT"]], color=TL_COLORS["GRADE"], ls=":", lw=.8)
        ax.scatter(lot.LOT_START, Y["GRADE"], marker="D", color=TL_COLORS["GRADE"], s=23)
    hit = tl_overlap.loc[tl_overlap["종류_A"].eq("LOT") & tl_overlap["ID_A"].eq(str(int(lot.LOT_NO))) &
                         tl_overlap["종류_B"].eq("WLD")]
    if len(hit):
        when = hit["겹침_시작"].min()
        ax.plot([when] * 2, [Y["LOT"], Y["WLD"]], color=TL_COLORS["OVERLAP"], lw=1.6, ls=(0, (3, 2)), zorder=3)
        ax.scatter(lot.LOT_START, Y["LOT"], s=105, facecolors="none", edgecolors=TL_COLORS["OVERLAP"], lw=1.6, zorder=6)
        ax.text(when, Y["WLD"] - .29, f"{hit['ID_B'].nunique()} COIL", ha="center", va="top", fontsize=8,
                color=TL_COLORS["OVERLAP"], bbox=dict(fc="white", ec="none", pad=1.5), zorder=7)
ax.scatter(tl_wld_day["DATE"], [Y["WLD"]] * len(tl_wld_day),
           s=15 + 40 * tl_wld_day["N_COIL"] / tl_wld_day["N_COIL"].max(),
           color=TL_COLORS["WLD"], alpha=.75, edgecolors="white", lw=.4, zorder=4)

for kind in ["BM", "CBM", "TBM"]:
    subset = tl_maint.loc[tl_maint["MNT_TYPE"].eq(kind)].sort_values("STRT_DT")
    # 연간 축에서는 한 작업당 × 하나. 실제 작업 길이는 선으로 유지한다.
    # 겹치는 마커만 같은 항목 안의 세부 행으로 옮긴다. 감지 시각은 설명 표에만 기록한다.
    offsets = tl_stack(list(zip(subset["STRT_DT"], subset["END_DT"])), min_days=2.8)
    for task, dy in zip(subset.itertuples(), offsets):
        yy = Y[kind] + dy
        ax.plot([task.STRT_DT, task.END_DT], [yy] * 2, lw=2.8, color=TL_COLORS[kind], zorder=4)
        ax.scatter(task.STRT_DT + (task.END_DT - task.STRT_DT) / 2, yy,
                   marker="x", s=35, linewidths=1.6, color=TL_COLORS[kind], zorder=5)
for code, yy in part_y.items():
    for task in tl_parts.loc[tl_parts["PART_CD"].eq(code)].itertuples():
        ax.scatter(task.END_DT, yy, marker="D", s=43, color=TL_PART_COLORS[code], zorder=5)
        ax.plot([task.END_DT] * 2, [Y[task.MNT_TYPE], yy], color=TL_COLORS["PART"],
                lw=.9, alpha=.5, ls=(0, (1, 3)), zorder=2)
# 정비–용접 / 정비–정비의 실제 교집합. 같은 날짜라는 이유만으로 연결하지 않는다.
for pair in tl_overlap.itertuples(index=False):
    a, aid, b, bid, t0, t1, mins = pair
    if a in ["BM", "CBM", "TBM"] and b in ["WLD", "BM", "CBM", "TBM"]:
        xx = t0 + (t1 - t0) / 2
        ax.plot([xx] * 2, [Y[a], Y[b]], color=TL_COLORS["OVERLAP"], lw=.65, alpha=.23, zorder=1)
    if a == "LOT" and b in ["BM", "CBM", "TBM"]:
        ax.plot([t0] * 2, [Y["LOT"], Y[b]], color=TL_COLORS["OVERLAP"], lw=1.2, ls="--", zorder=2)

labels = [f"센서 LOT · {len(tl_lots)}개", f"강종변경 · {len(tl_grades)}건",
          f"용접 · {len(tl_wld):,} COIL\n{len(tl_wld_day)}일 관측"]
labels += [f"{kind} {TL_KIND_NAMES[kind]}\n{tl_maint['MNT_TYPE'].eq(kind).sum()}건" for kind in ["BM", "CBM", "TBM"]]
labels += [f"{tl_parts.loc[tl_parts.PART_CD.eq(code), 'PART_NM'].iloc[0]}\n{code} · {tl_parts.PART_CD.eq(code).sum()}건" for code in part_codes]
labels += ["CR1 계획수리 · 2회"]
positions = list(Y.values()) + list(part_y.values()) + [plan_y]
ax.set_yticks(positions, labels)
ax.set_xlim(YEAR_START - pd.Timedelta(days=4), YEAR_END)
ax.set_ylim(-1.05, 11.1)
ax.set_xticks(pd.date_range(YEAR_START, periods=12, freq="MS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_xlabel("", fontsize=11, labelpad=10)
tl_style(ax)
ax.tick_params(axis="y", labelsize=10, pad=9)
for label, (_, color) in zip(ax.get_yticklabels(), source_rows):
    label.set_color(color)
    label.set_bbox(dict(boxstyle="round,pad=.35", fc=color, ec="none", alpha=.065))
lot_label_artists = tl_place_lot_labels(ax, list(tl_lots.itertuples()), Y["LOT"])
fig.suptitle("그림 1. 2024년 온도·용접·소둔로 정비·부품교체", x=.018, y=.98,
             ha="left", fontsize=20, fontweight="bold", color="#17344d")
fig.text(.018, .929, "센서 14 LOT · 98.9시간   |   소둔로 정비 44건   |   부품교체 9건   |   LOT–용접 겹침 4 LOT   |   LOT–정비 작업 겹침 1 LOT",
         fontsize=11, color="#36566e")
source_handles = [Patch(fc=TL_COLORS[k], label=label) for k, label in
                  [("LOT", "온도 CSV"), ("GRADE", "조업이벤트 CSV"), ("WLD", "용접 CSV"),
                   ("BM", "정비이력 CSV"), ("PART", "부품교체 CSV"), ("PLAN", "정기수리 CSV")]]
source_legend = fig.legend(handles=source_handles, loc="lower center", bbox_to_anchor=(.55, .085),
           ncol=6, fontsize=9.5, frameon=False, columnspacing=2.8)
fig.text(.105, .058, "× 정비 작업 구간     ◆ 교체 연결 작업 완료     붉은 연결 = 실제 시간 겹침     보라 연결 = 같은 작업번호",
         fontsize=10, color="#435b6c")
tl_check_text(fig, [lot_label_artists, ax.get_yticklabels(), [source_legend, ax.xaxis.label] + list(fig.texts)])
fig.savefig(Path(ANNUAL_DIR) / "01_연간_통합_타임라인.png", dpi=180, facecolor="white")
fig.savefig(Path(ANNUAL_DIR) / "01_연간_통합_타임라인.svg", facecolor="white")
plt.show()

**결과 — 연간 LOT 관측과 작업 대조**

| 행 | Focus | LOT | 강종 | 관측_시작 | 관측_종료 | 관측_분 | 정비_작업 | 정비_감지 | WLD_COIL수 | Focus_이유 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 |  | 240001 | SGCC | 2024-01-02 03:36:00 | 2024-01-02 15:04:30 | 688.50 |  |  | 0 |  |
| 1 |  | 240024 | SPFH590 | 2024-03-07 21:22:00 | 2024-03-08 06:29:48 | 547.80 |  |  | 0 |  |
| 2 |  | 240037 | SPCE | 2024-04-13 23:16:00 | 2024-04-14 09:27:19 | 611.32 |  |  | 0 |  |
| 3 |  | 240038 | SPCEN | 2024-04-16 11:35:00 | 2024-04-16 13:39:56 | 124.93 |  |  | 0 |  |
| 4 |  | 240044 | SPHC | 2024-05-03 18:50:00 | 2024-05-03 20:04:38 | 74.63 |  |  | 0 |  |
| 5 | F1 | 240058 | DP590 | 2024-06-09 05:26:00 | 2024-06-09 09:02:53 | 216.88 |  |  | 0 | 수리 전 마지막 관측 |
| 6 | F2 | 240061 | SPHC | 2024-06-15 01:04:00 | 2024-06-15 04:49:23 | 225.38 |  |  | 0 | 수리 후 첫 관측 |
| 7 |  | 240069 | SPHC | 2024-07-07 16:57:00 | 2024-07-08 06:29:34 | 812.57 |  |  | 0 |  |
| 8 |  | 240091 | SPHC | 2024-09-08 22:03:00 | 2024-09-09 08:44:12 | 641.20 |  |  | 0 |  |
| 9 |  | 240108 | SPFH590 | 2024-10-18 04:05:00 | 2024-10-18 11:50:53 | 465.88 |  |  | 12 |  |
| 10 | F3 | 240114 | SPCEN | 2024-11-04 15:40:00 | 2024-11-04 20:59:02 | 319.03 |  |  | 11 | 수리 전 마지막 관측 |
| 11 | F4 | 240117 | SPCD | 2024-11-13 07:08:00 | 2024-11-13 14:32:19 | 444.32 | WO-2024-01859, WO-2024-01854 | WO-2024-01859 | 31 | 수리 후 첫 관측 / 정비 작업 겹침 / 정비 감지 포함 |
| 12 |  | 240121 | SPCE | 2024-11-24 07:31:00 | 2024-11-24 18:22:12 | 651.20 |  |  | 28 |  |
| 13 | F5 | 240133 | SPCE | 2024-12-28 14:57:00 | 2024-12-28 16:44:24 | 107.40 |  | WO-2024-02105 | 0 | 정비 감지 포함 |

### 그림 1 확대: 관측·작업 구간과 용접 기록을 어떻게 읽는가

정비는 **CAL01 연속 소둔로(`P1-CR1-CAL01`)**만 포함한다. 계획수리는 **CR1 라인** 캘린더이며 용접은 **WLD01** 데이터다.
청록 선은 LOT 온도 CSV의 최초~최종 관측 구간이고, 선 옆에 기간과 길이를 표시한다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다.
×–×는 정비 착수~완료, 보라 ◆는 부품교체에 연결된 작업의 완료다.

72시간을 넘는 넓은 창에서는 용접을 **일별 관측 COIL 수**로 보여준다. 막대 높이가 COIL 수이며 폭은 하루 집계 폭이다.
짧은 확대에서는 **COIL별 최초~최종 기록 구간**을 유지한다. COIL당 약 2.2분인 기록을 여러 날 축에 그리면 바코드처럼 압축되므로 표시 방식을 나눴다.
날짜 눈금은 수리·강종변경·관측을 같은 달력에서 맞추기 위해 유지하고, 중복되는 “실제 날짜/시각” 축 제목은 제거했다.

정확한 시각·REMARK·교체사유·해석은 아래 사건별 마크다운 셀에서 읽는다. 각 셀의 그림 링크로 해당 확대 그림을 열 수 있다.


이전에 그림 전체에 칠했던 파란 음영은 **CR1 계획수리 기간**이었다. 지금은 **계획수리 전용 행**에 빗금·청록선·날짜로 표시한다. 강종 행의 푸른 띠는 마지막 강종변경 기록을 유지한 상태다.


In [ ]:
# %% 그림 1 상세: 핵심 시간 관계만 그리고, 설명·원문은 마크다운에 보존한다.
def tl_interval_note(ax, start, end, yy, text, color):
    """구간의 실제 길이를 유지하면서 설명을 비어 있는 쪽에 배치한다."""
    ax.figure.canvas.draw()
    panel=ax.get_window_extent()
    for x,offset,align in [(end,7,"left"),(start,-7,"right")]:
        label=ax.annotate(text,(x,yy),xytext=(offset,0),textcoords="offset points",
            ha=align,va="center",fontsize=9,color=color,linespacing=1.25,
            bbox=dict(fc="white",ec="none",alpha=.88,pad=2),zorder=9)
        ax.figure.canvas.draw()
        box=label.get_window_extent()
        if box.x0>=panel.x0 and box.x1<=panel.x1:return label
        label.remove()
    return ax.text(.015,yy-.14,text,transform=ax.get_yaxis_transform(),ha="left",va="top",
        fontsize=8.5,color=color,bbox=dict(fc="white",ec="none",alpha=.88,pad=2),zorder=9)

def tl_draw_detail(start, end, title, filename):
    local_lots = tl_lots.loc[(tl_lots.LOT_START <= end) & (tl_lots.LOT_END >= start)]
    local_maint = iq_work.loc[((iq_work.STRT_DT <= end) & (iq_work.END_DT >= start)) | iq_work.DETC_DT.between(start,end)]
    assert local_maint.EQP_TAG.eq("P1-CR1-CAL01").all(), "CAL01 이외 정비가 섞였습니다."
    local_parts = tl_parts.loc[(tl_parts.STRT_DT <= end) & (tl_parts.END_DT >= start)]
    local_coils = tl_wld.loc[(tl_wld.WLD_START <= end) & (tl_wld.WLD_END >= start)]
    local_plans = tl_plans.loc[(tl_plans.STRT_DT < end) & (tl_plans.END_EXCLUSIVE > start)]
    states = iq_states.loc[(iq_states.START < end) & (iq_states.END > start)]
    aggregate_weld=(end-start)>pd.Timedelta(hours=72)
    rows = [dict(key="GRADE",kind="GRADE",label="강종 기록",start=start,end=end)]
    for lot in local_lots.itertuples():
        rows.append(dict(key=str(int(lot.LOT_NO)),kind="LOT",label=f"LOT {int(lot.LOT_NO)} · {lot.GRADE}\n온도 관측",start=lot.LOT_START,end=lot.LOT_END))
    if local_lots.empty:
        rows.append(dict(key="NO_TEMP",kind="LOT",label="온도 관측",start=start,end=end))
    rows.append(dict(key="WLD",kind="WLD",label=f"WLD01 · {len(local_coils)} COIL\n"+("일별 관측 COIL 수" if aggregate_weld else "COIL별 관측 구간"),start=start,end=end))
    for task in local_maint.itertuples():
        rows.append(dict(key=task.WO_NO,kind=task.MNT_TYPE,label=f"{task.MNT_TYPE} · {task.WO_NO[-5:]}",start=task.STRT_DT,end=task.END_DT))
    for task in local_parts.itertuples():
        rows.append(dict(key=task.WO_NO+"/"+task.PART_CD,kind="PART",label=f"{task.PART_NM}\n{task.PART_CD}",start=task.STRT_DT,end=task.END_DT,code=task.PART_CD))
    for plan in local_plans.itertuples():
        rows.append(dict(key=plan.PLAN_ID,kind="PLAN",label=f"CR1 계획수리 · {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}",start=plan.STRT_DT,end=plan.END_EXCLUSIVE))
    n=len(rows)
    sizes=np.array([1.9 if r["kind"]=="WLD" and aggregate_weld else 1 for r in rows])
    total_height=sizes.sum()
    height=2.4+.62*total_height
    fig,ax=plt.subplots(figsize=(22,height))
    fig.subplots_adjust(left=.12,right=.985,top=1-1.05/height,bottom=.95/height)
    ax.set_xlim(start,end)
    ax.set_ylim(-.6,total_height-.4)
    centers=total_height-np.cumsum(sizes)+sizes/2-.5
    row_y={r["key"]:float(y) for r,y in zip(rows,centers)}
    grade_labels=[]
    interval_labels=[]
    daily_labels=[]
    wld_size=1.9 if aggregate_weld else 1
    for row in rows:
        yy,kind=row_y[row["key"]],row["kind"]
        color="#3386a5" if kind=="PLAN" else TL_PART_COLORS.get(row.get("code"),TL_COLORS[kind])
        ax.axhspan(yy-.4*(wld_size if kind=="WLD" else 1),yy+.4*(wld_size if kind=="WLD" else 1),color="#f5f7f9",lw=0,zorder=0)
        if kind=="GRADE":
            for i,s in enumerate(states.itertuples()):
                a,b=max(s.START,start),min(s.END,end)
                ax.plot([a,b],[yy]*2,color="#d2e1ee" if i%2==0 else "#b8d0e3",lw=23,solid_capstyle="butt",zorder=2)
                grade_labels.append(ax.text(a+(b-a)/2,yy,s.GRADE,ha="center",va="center",fontsize=10,color="#254e6b",zorder=3))
            for change in iq_events.loc[iq_events.EVT_DT.between(start,end)].itertuples():
                ax.scatter(change.EVT_DT,yy+.26,marker="v",s=28,color=TL_COLORS["GRADE"],zorder=4)
        elif row["key"]=="NO_TEMP" or (kind=="WLD" and local_coils.empty):
            ax.text(start+(end-start)/2,yy,"CSV 관측 없음",ha="center",va="center",fontsize=11,color="#6e8290")
        elif kind=="WLD":
            if aggregate_weld:
                # 같은 날짜 축에 일별 COIL 수를 독립 눈금으로 표시한다.
                observed=wld.loc[(wld.MEAS_DT>=start)&(wld.MEAS_DT<end)]
                first=observed.groupby("COIL_ID").MEAS_DT.min()
                daily=first.dt.floor("D").value_counts().sort_index()
                mini=ax.inset_axes([mdates.date2num(start),yy-.65,
                    mdates.date2num(end)-mdates.date2num(start),1.3],transform=ax.transData,zorder=3)
                mini.patch.set_alpha(0)
                peak=int(daily.max()) if len(daily) else 1
                for day,count in daily.items():
                    a,b=max(day,start),min(day+pd.Timedelta(days=1),end)
                    center=a+(b-a)/2
                    width=(b-a).total_seconds()/86400*.65
                    mini.bar(center,count,width=width,color=color,alpha=.85,zorder=2)
                    daily_labels.append(mini.text(center,count+peak*.055,str(count),ha="center",va="bottom",fontsize=9,color=color))
                mini.set_xlim(start,end);mini.set_ylim(0,peak*1.35)
                mini.set_yticks([0,peak]);mini.tick_params(axis="y",length=0,labelsize=8,pad=-18,labelcolor="#826d52")
                mini.xaxis.set_visible(False)
                mini.spines[:].set_visible(False)
                mini._daily_coil_counts=daily
            else:
                for coil in local_coils.itertuples():
                    ax.plot([coil.WLD_START,coil.WLD_END],[yy]*2,color=color,lw=5,solid_capstyle="butt",zorder=3)
        elif kind in ["BM","CBM","TBM"]:
            ax.plot([row["start"],row["end"]],[yy]*2,color=color,lw=2.5,marker="x",ms=6,mew=1.8,zorder=4)
            minutes=(row["end"]-row["start"]).total_seconds()/60
            interval_labels.append(tl_interval_note(ax,row["start"],row["end"],yy,f"작업 {minutes:g}분",color))
        elif kind=="PART":
            ax.plot([row["start"],row["end"]],[yy]*2,color=color,lw=4,alpha=.4)
            ax.scatter(row["end"],yy,marker="D",s=40,color=color,zorder=7)
            if row["key"].split("/")[0] in row_y:
                ax.plot([row["end"]]*2,[row_y[row["key"].split("/")[0]],yy],color=color,ls=":",lw=1,zorder=2)
        elif kind=="LOT":
            ax.plot([row["start"],row["end"]],[yy]*2,color=color,lw=4,marker="o",ms=3,mec="white",mew=.5,zorder=3)
            minutes=(row["end"]-row["start"]).total_seconds()/60
            text=f"{row['start']:%m/%d %H:%M}–{row['end']:%H:%M} · {minutes:.1f}분"
            if row['start'].date()!=row['end'].date():
                text=f"{row['start']:%m/%d %H:%M}–{row['end']:%m/%d %H:%M} · {minutes:.1f}분"
            count=tl_overlap.loc[tl_overlap.ID_A.eq(row["key"])&tl_overlap["종류_B"].eq("WLD"),"ID_B"].nunique()
            if aggregate_weld and count:text+=f"\n용접 {count} COIL과 시간 겹침"
            interval_labels.append(tl_interval_note(ax,row["start"],row["end"],yy,text,color))
        else:
            # 계획수리는 전용 행 안에서만 표시하여 관측·강종 배경과 섞이지 않는다.
            a,b=max(start,row["start"]),min(end,row["end"])
            ax.fill_between([a,b],[yy-.36]*2,[yy+.36]*2,facecolor="#e0eef4",edgecolor="#90b5c7",hatch="///",lw=.4,zorder=1)
            ax.plot([row["start"],row["end"]],[yy-.18]*2,color=color,lw=6,solid_capstyle="butt",zorder=3)
            plan=local_plans.loc[local_plans.PLAN_ID.eq(row["key"])].iloc[0]
            interval_labels.append(ax.text(a+(b-a)/2,yy+.17,f"CR1 계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}",
                ha="center",va="center",fontsize=10,color="#28667f",zorder=6))
    if filename=="gap_202402_03.png":
        yy=row_y["240024"]
        ax.fill_between([iq_gap_start,iq_gap_end],[yy-.32]*2,[yy+.32]*2,
                        facecolor="#e9eef2",edgecolor="#cdd6de",hatch="///",alpha=.6,lw=0,zorder=1)
        ax.text(iq_gap_start+(iq_gap_end-iq_gap_start)/2,yy,
                "온도 CSV 관측 없음 · 02/01–03/07 21:22",ha="center",va="center",fontsize=11,
                color="#586d7c",bbox=dict(fc="white",ec="none",pad=3),zorder=6)
    # 계획수리 날짜는 전용 행에 표시하고, 실제 관측/작업의 교집합만 붉게 강조한다.
    hits=tl_overlap.loc[tl_overlap.ID_A.isin(row_y) & tl_overlap.ID_B.isin(row_y) & tl_overlap["종류_B"].ne("계획수리")]
    overlap_labels=[]
    for pair in hits.itertuples(index=False):
        ya,yb=row_y[pair[1]],row_y[pair[3]]
        mid=pair[4]+(pair[5]-pair[4])/2
        for yy in [ya,yb]: ax.plot([pair[4],pair[5]],[yy]*2,color=TL_COLORS["OVERLAP"],lw=4,zorder=5)
        ax.plot([mid]*2,[ya,yb],color=TL_COLORS["OVERLAP"],ls=":",lw=1,alpha=.6)
        if pair[0] in ["BM","CBM","TBM"] and pair[2] in ["BM","CBM","TBM"]:
            overlap_labels.append(ax.text(mid,(ya+yb)/2,f"{pair[6]:g}분 겹침",ha="center",va="center",fontsize=10,color=TL_COLORS["OVERLAP"],zorder=8,bbox=dict(boxstyle="round,pad=.3",fc="white",ec="#ecc2c8")))
    wh=tl_overlap.loc[tl_overlap.ID_A.isin(row_y) & tl_overlap["종류_B"].eq("WLD")]
    for (kind,event_id),group in wh.groupby(["종류_A","ID_A"]):
        for pair in group.itertuples(index=False):
            for yy in ([row_y[event_id]] if aggregate_weld else [row_y[event_id],row_y["WLD"]]):
                ax.plot([pair[4],pair[5]],[yy]*2,color=TL_COLORS["OVERLAP"],lw=4,zorder=5)
        when=group["겹침_시작"].min()
        if not aggregate_weld:
            ax.plot([when]*2,[row_y[event_id],row_y["WLD"]],color=TL_COLORS["OVERLAP"],ls=":",lw=.9,alpha=.55)
    ax.set_yticks(list(row_y.values()),[r["label"] for r in rows])
    span_h=(end-start).total_seconds()/3600
    locator=mdates.HourLocator(interval=next(v for v in [1,2,3,4,6,8,12] if v>=span_h/7)) if span_h<=72 else mdates.DayLocator(interval=max(1,int(np.ceil(span_h/24/7))))
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d" if aggregate_weld else "%m/%d\n%H:%M"))
    ax.set_xlabel("")
    tl_style(ax)
    ax.tick_params(axis="y",labelsize=10,pad=10)
    for label,row in zip(ax.get_yticklabels(),rows):
        label.set_color("#3386a5" if row["kind"]=="PLAN" else TL_COLORS[row["kind"]])
    fig.suptitle(title,x=.018,y=1-.18/height,ha="left",fontsize=17,fontweight="bold",color="#17344d")
    period=f"{start:%Y/%m/%d}–{end:%m/%d}" if aggregate_weld else f"{start:%Y/%m/%d %H:%M}–{end:%m/%d %H:%M}"
    fig.text(.018,1-.66/height,period+" · 정비: CAL01 연속 소둔로 · 계획수리: CR1",fontsize=10.5,color="#435b6c")
    fig.text(.12,.14/height,"청록 선: LOT 온도 관측    ×–× CAL01 정비 작업    ◆ 교체 연결 작업 완료    붉은색 시간 겹침    ▼ 강종변경 기록",fontsize=9.5,color="#435b6c")
    tl_check_text(fig,[grade_labels,overlap_labels,interval_labels,daily_labels,ax.get_yticklabels()])
    fig.savefig(TL_DETAIL_DIR/tl_detail_filename(filename),dpi=160,facecolor="white")
    plt.show()

for p in tl_plans.itertuples():
    tl_draw_detail(p.STRT_DT-pd.Timedelta(days=2),p.END_EXCLUSIVE+pd.Timedelta(days=3),
                   f"그림 1 확대. {p.STRT_DT:%m월} 계획수리 · 강종변경과 개별 정비",f"plan_{p.PLAN_ID}.png")
detail_lots=sorted(set(tl_focus) | set(tl_lot_table.loc[tl_lot_table["WLD_COIL수"].gt(0),"LOT"]))
for lot_id in detail_lots:
    lot=tl_lots.loc[tl_lots.LOT_NO.eq(lot_id)].iloc[0]
    tl_draw_detail(lot.LOT_START-pd.Timedelta(hours=6),lot.LOT_END+pd.Timedelta(hours=12),
                   f"그림 1 확대. LOT {lot_id} · {lot.GRADE} · 관측과 작업",f"lot_{lot_id}.png")
tl_draw_detail(pd.Timestamp("2024-11-09 08:00"),pd.Timestamp("2024-11-09 20:00"),"그림 1 확대. 11/09 TBM 두 작업의 107분 겹침","maintenance_20241109.png")
for task_id in sorted(tl_part_overlap.WO_NO.unique()):
    task=tl_maint.loc[tl_maint.WO_NO.eq(task_id)].iloc[0]
    tl_draw_detail(task.STRT_DT-pd.Timedelta(hours=1),task.END_DT+pd.Timedelta(hours=1),
                   f"그림 1 확대. {task_id} · 부품 연결 작업과 용접",f"parts_{task_id}.png")
tl_draw_detail(iq_gap_start,pd.Timestamp("2024-03-08 12:00"),"질문 2. 2월~3월 초 · 강종변경과 정비는 있고 센서 관측은 없는 구간","gap_202402_03.png")
for task in iq_bm.loc[iq_bm.START_GRADE.eq("SPCE")].itertuples():
    tl_draw_detail(task.STRT_DT.normalize()-pd.Timedelta(days=2),task.END_DT.normalize()+pd.Timedelta(days=3),
                   f"질문 3. SPCE 기록 중 BM · {task.STRT_DT:%m/%d} · 센서 관측 확인",f"bm_{task.WO_NO}.png")

# 정비 REMARK·부품 원문은 아래 마크다운 셀에 보존한다.

<a id="context-plan_PLAN-2024-02"></a>

### 그림 1 확대. 06월 계획수리 · 강종변경과 개별 정비


![그림 1 확대. 06월 계획수리 · 강종변경과 개별 정비](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/01_계획수리_PLAN-2024-02.png)

기간: 2024/06/08 00:00–2024/06/18 00:00.

**관측:** 온도 2 LOT, 용접 291 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240058 | DP590 | 06/09 05:26:00 | 06/09 09:02:53 | 216.88 |
| 240061 | SPHC | 06/15 01:04:00 | 06/15 04:49:23 | 225.38 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0117 | 06/09 05:26:00 | SGCC | DP590 |
| EVT-2024-0120 | 06/11 08:00:00 | DP590 | SPCC |
| EVT-2024-0121 | 06/13 10:34:00 | SPCC | DP590 |
| EVT-2024-0122 | 06/15 01:04:00 | DP590 | SPHC |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00916 | P1-CR1-CAL01 | TBM | 06/10 06:23:00 | 06/10 08:45:00 | 06/10 10:48:00 | DP590 | DP590 | — | — |
| WO-2024-00931 | P1-CR1-CAL01 | TBM | 06/11 14:51:00 | 06/11 18:40:00 | 06/11 23:37:00 | SPCC | SPCC | WER | 마모량 초과 |
| WO-2024-00940 | P1-CR1-CAL01 | TBM | 06/12 06:09:00 | 06/12 09:51:00 | 06/12 15:23:00 | SPCC | SPCC | — | 수리기간 중 점검 |
| WO-2024-00952 | P1-CR1-CAL01 | TBM | 06/12 23:51:00 | 06/13 02:51:00 | 06/13 05:16:00 | SPCC | SPCC | — |   정기수리 부속작업  |
| WO-2024-00960 | P1-CR1-CAL01 | TBM | 06/13 14:22:00 | 06/13 18:19:00 | 06/13 22:29:00 | DP590 | DP590 | — | 1호기 재발, SD작업 수행 |
| WO-2024-00972 | P1-CR1-CAL01 | TBM | 06/14 14:09:00 | 06/14 15:35:00 | 06/14 18:03:00 | DP590 | DP590 | — | 1호기 재발, SD작업 수행 |

#### 부품교체

| WO_NO | PART_CD | PART_NM | QTY | CHG_RSN |
| --- | --- | --- | --- | --- |
| WO-2024-00916 | SEL-006 | 오일씰 150x190x18 | 2 | 수명도래 |
| WO-2024-00916 | SEL-009 | O링 세트 | 1 | 손상 |

교체 자체의 시각은 미제공이며, 그림은 연결 작업 구간과 완료 시각을 표시한다.

#### 이 6월 그림에서 확인할 결론

6/10–6/14 계획수리 창에 CAL01 작업 6건이 있다. 이 창의 온도 CSV는 0행, 용접 CSV는 0행이다. 관측 미제공의 이유와 실제 가동 상태는 수집/운전 기록으로 확인해야 한다.

| LOT | 온도 관측 종료 | 그날 첫 용접 기록 | 첫 용접 − 온도 종료(분) | 온도와 겹친 COIL 수 |
| --- | --- | --- | --- | --- |
| 240058 | 06/09 09:02:53 | 06/09 09:19:19 | 16.44 | 0 |
| 240061 | 06/15 04:49:23 | 06/15 09:13:51 | 264.47 | 0 |

두 LOT는 용접과 같은 날 관측됐지만 실제 관측 구간은 겹치지 않는다. 위 시간 차이는 다음 센서 기록까지의 간격이며 소재의 이동시간을 뜻하지 않는다.

수리 전 마지막 온도 관측 240058(DP590) → 수리 중 CAL01 TBM → 수리 후 첫 온도 관측 240061(SPHC)의 시간 순서를 확인한다. 전후 강종이 다르므로 수리 효과의 직접 비교 조건은 갖춰지지 않았다. 240061의 온도 특성은 다른 SPHC LOT의 공정별 신호와 비교한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

##### 일별 용접 관측 COIL 수

| 날짜 | COIL 수 |
| --- | --- |
| 06/09 00:00:00 | 87 |
| 06/15 00:00:00 | 73 |
| 06/16 00:00:00 | 62 |
| 06/17 00:00:00 | 69 |

주황 막대 높이는 각 COIL의 화면 내 첫 기록 날짜 기준 관측 COIL 수다. 막대 폭은 하루 집계 폭이며 개별 용접 지속시간을 뜻하지 않는다. 하루 경계를 넘는 COIL은 한 번 집계한다. COIL별 정확한 구간과 겹침은 LOT 확대 그림·시간 대응 표에서 확인한다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-plan_PLAN-2024-04"></a>

### 그림 1 확대. 11월 계획수리 · 강종변경과 개별 정비


![그림 1 확대. 11월 계획수리 · 강종변경과 개별 정비](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/01_계획수리_PLAN-2024-04.png)

기간: 2024/11/03 00:00–2024/11/14 00:00.

**관측:** 온도 2 LOT, 용접 166 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240114 | SPCEN | 11/04 15:40:00 | 11/04 20:59:02 | 319.03 |
| 240117 | SPCD | 11/13 07:08:00 | 11/13 14:32:19 | 444.32 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0220 | 11/04 15:40:00 | SPHC | SPCEN |
| EVT-2024-0223 | 11/07 14:08:00 | SPCEN | SPCE |
| EVT-2024-0226 | 11/10 17:02:00 | SPCE | SPFH590 |
| EVT-2024-0229 | 11/13 07:08:00 | SPFH590 | SPCD |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01775 | P1-CR1-CAL01 | TBM | 11/06 07:49:00 | 11/06 10:40:00 | 11/06 12:23:00 | SPCEN | SPCEN | — | 정기수리 실시 |
| WO-2024-01794 | P1-CR1-CAL01 | TBM | 11/07 09:20:00 | 11/07 12:53:00 | 11/07 14:14:00 | SPCEN | SPCEN | — | 정기수리 부속작업 |
| WO-2024-01823 | P1-CR1-CAL01 | TBM | 11/09 06:05:00 | 11/09 09:58:00 | 11/09 13:02:00 | SPCE | SPCE | — | 1호기 재발, 정기수리 실시 |
| WO-2024-01826 | P1-CR1-CAL01 | TBM | 11/09 07:39:00 | 11/09 11:15:00 | 11/09 18:20:00 | SPCE | SPCE | — | 수리기간 중 점검 |
| WO-2024-01834 | P1-CR1-CAL01 | TBM | 11/10 07:00:00 | 11/10 10:40:00 | 11/10 12:52:00 | SPCE | SPCE | — | 정기수리 부속작업 |
| WO-2024-01859 | P1-CR1-CAL01 | TBM | 11/13 07:13:00 | 11/13 07:40:00 | 11/13 09:35:00 | SPCD | SPCD | — | 계획 점검 수행 |
| WO-2024-01854 | P1-CR1-CAL01 | CBM | 11/12 15:00:00 | 11/13 08:54:00 | 11/13 12:44:00 | SPFH590 | SPCD | WER | 마모량 초과 |

#### 실제 시간 겹침

| 종류_A | ID_A | 종류_B | ID_B | 겹침_시작 | 겹침_종료 | 겹침_분 |
| --- | --- | --- | --- | --- | --- | --- |
| LOT | 240117 | TBM | WO-2024-01859 | 11/13 07:40:00 | 11/13 09:35:00 | 115.0 |
| LOT | 240117 | CBM | WO-2024-01854 | 11/13 08:54:00 | 11/13 12:44:00 | 230.0 |
| TBM | WO-2024-01823 | TBM | WO-2024-01826 | 11/09 11:15:00 | 11/09 13:02:00 | 107.0 |
| TBM | WO-2024-01859 | CBM | WO-2024-01854 | 11/13 08:54:00 | 11/13 09:35:00 | 41.0 |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| CBM | WO-2024-01854 | 21 | 11/13 09:04:01 | 11/13 12:29:13 |
| LOT | 240114 | 11 | 11/04 15:40:24 | 11/04 17:19:59 |
| LOT | 240117 | 31 | 11/13 09:04:01 | 11/13 14:26:50 |
| TBM | WO-2024-01859 | 8 | 11/13 09:04:01 | 11/13 09:34:13 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

##### 일별 용접 관측 COIL 수

| 날짜 | COIL 수 |
| --- | --- |
| 11/04 00:00:00 | 30 |
| 11/11 00:00:00 | 41 |
| 11/12 00:00:00 | 47 |
| 11/13 00:00:00 | 48 |

주황 막대 높이는 각 COIL의 화면 내 첫 기록 날짜 기준 관측 COIL 수다. 막대 폭은 하루 집계 폭이며 개별 용접 지속시간을 뜻하지 않는다. 하루 경계를 넘는 COIL은 한 번 집계한다. COIL별 정확한 구간과 겹침은 LOT 확대 그림·시간 대응 표에서 확인한다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240058"></a>

### 그림 1 확대. LOT 240058 · DP590 · 관측과 작업


![그림 1 확대. LOT 240058 · DP590 · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240058.png)

기간: 2024/06/08 23:26–2024/06/09 21:02.

**관측:** 온도 1 LOT, 용접 87 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240058 | DP590 | 06/09 05:26:00 | 06/09 09:02:53 | 216.88 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0117 | 06/09 05:26:00 | SGCC | DP590 |

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240061"></a>

### 그림 1 확대. LOT 240061 · SPHC · 관측과 작업


![그림 1 확대. LOT 240061 · SPHC · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240061.png)

기간: 2024/06/14 19:04–2024/06/15 16:49.

**관측:** 온도 1 LOT, 용접 63 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240061 | SPHC | 06/15 01:04:00 | 06/15 04:49:23 | 225.38 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0122 | 06/15 01:04:00 | DP590 | SPHC |

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240108"></a>

### 그림 1 확대. LOT 240108 · SPFH590 · 관측과 작업


![그림 1 확대. LOT 240108 · SPFH590 · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240108.png)

기간: 2024/10/17 22:05–2024/10/18 23:50.

**관측:** 온도 1 LOT, 용접 41 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240108 | SPFH590 | 10/18 04:05:00 | 10/18 11:50:53 | 465.88 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0210 | 10/18 04:05:00 | SPCE | SPFH590 |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01678 | P1-CR1-CAL01 | CBM | 10/18 15:50:00 | 10/26 23:53:00 | 10/27 02:42:00 | SPFH590 | SGCC | ETC | 원인 미상 |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| LOT | 240108 | 12 | 10/18 09:11:29 | 10/18 11:50:53 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240114"></a>

### 그림 1 확대. LOT 240114 · SPCEN · 관측과 작업


![그림 1 확대. LOT 240114 · SPCEN · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240114.png)

기간: 2024/11/04 09:40–2024/11/05 08:59.

**관측:** 온도 1 LOT, 용접 30 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240114 | SPCEN | 11/04 15:40:00 | 11/04 20:59:02 | 319.03 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0220 | 11/04 15:40:00 | SPHC | SPCEN |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| LOT | 240114 | 11 | 11/04 15:40:24 | 11/04 17:19:59 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240117"></a>

### 그림 1 확대. LOT 240117 · SPCD · 관측과 작업


![그림 1 확대. LOT 240117 · SPCD · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240117.png)

기간: 2024/11/13 01:08–2024/11/14 02:32.

**관측:** 온도 1 LOT, 용접 48 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240117 | SPCD | 11/13 07:08:00 | 11/13 14:32:19 | 444.32 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0229 | 11/13 07:08:00 | SPFH590 | SPCD |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01859 | P1-CR1-CAL01 | TBM | 11/13 07:13:00 | 11/13 07:40:00 | 11/13 09:35:00 | SPCD | SPCD | — | 계획 점검 수행 |
| WO-2024-01854 | P1-CR1-CAL01 | CBM | 11/12 15:00:00 | 11/13 08:54:00 | 11/13 12:44:00 | SPFH590 | SPCD | WER | 마모량 초과 |

#### 실제 시간 겹침

| 종류_A | ID_A | 종류_B | ID_B | 겹침_시작 | 겹침_종료 | 겹침_분 |
| --- | --- | --- | --- | --- | --- | --- |
| LOT | 240117 | TBM | WO-2024-01859 | 11/13 07:40:00 | 11/13 09:35:00 | 115.0 |
| LOT | 240117 | CBM | WO-2024-01854 | 11/13 08:54:00 | 11/13 12:44:00 | 230.0 |
| TBM | WO-2024-01859 | CBM | WO-2024-01854 | 11/13 08:54:00 | 11/13 09:35:00 | 41.0 |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| CBM | WO-2024-01854 | 21 | 11/13 09:04:01 | 11/13 12:29:13 |
| LOT | 240117 | 31 | 11/13 09:04:01 | 11/13 14:26:50 |
| TBM | WO-2024-01859 | 8 | 11/13 09:04:01 | 11/13 09:34:13 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240121"></a>

### 그림 1 확대. LOT 240121 · SPCE · 관측과 작업


![그림 1 확대. LOT 240121 · SPCE · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240121.png)

기간: 2024/11/24 01:31–2024/11/25 06:22.

**관측:** 온도 1 LOT, 용접 28 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240121 | SPCE | 11/24 07:31:00 | 11/24 18:22:12 | 651.2 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0238 | 11/24 07:31:00 | SPCD | SPCE |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| LOT | 240121 | 28 | 11/24 09:04:58 | 11/24 14:58:43 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-lot_240133"></a>

### 그림 1 확대. LOT 240133 · SPCE · 관측과 작업


![그림 1 확대. LOT 240133 · SPCE · 관측과 작업](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/02_LOT_240133.png)

기간: 2024/12/28 08:57–2024/12/29 04:44.

**관측:** 온도 1 LOT, 용접 0 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240133 | SPCE | 12/28 14:57:00 | 12/28 16:44:24 | 107.4 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0262 | 12/28 14:57:00 | SPCC | SPCE |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-02105 | P1-CR1-CAL01 | CBM | 12/28 15:54:00 | 12/28 16:54:00 | 12/28 23:33:00 | SPCE | SPCE | WER | 마모량 초과 |

#### 부품교체

| WO_NO | PART_CD | PART_NM | QTY | CHG_RSN |
| --- | --- | --- | --- | --- |
| WO-2024-02105 | INS-006 | 가스 분석계 셀 | 3 | 예방 |

교체 자체의 시각은 미제공이며, 그림은 연결 작업 구간과 완료 시각을 표시한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-maintenance_20241109"></a>

### 그림 1 확대. 11/09 TBM 두 작업의 107분 겹침


![그림 1 확대. 11/09 TBM 두 작업의 107분 겹침](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/03_TBM_작업겹침_20241109.png)

기간: 2024/11/09 08:00–2024/11/09 20:00.

**관측:** 온도 0 LOT, 용접 0 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01823 | P1-CR1-CAL01 | TBM | 11/09 06:05:00 | 11/09 09:58:00 | 11/09 13:02:00 | SPCE | SPCE | — | 1호기 재발, 정기수리 실시 |
| WO-2024-01826 | P1-CR1-CAL01 | TBM | 11/09 07:39:00 | 11/09 11:15:00 | 11/09 18:20:00 | SPCE | SPCE | — | 수리기간 중 점검 |

#### 실제 시간 겹침

| 종류_A | ID_A | 종류_B | ID_B | 겹침_시작 | 겹침_종료 | 겹침_분 |
| --- | --- | --- | --- | --- | --- | --- |
| TBM | WO-2024-01823 | TBM | WO-2024-01826 | 11/09 11:15:00 | 11/09 13:02:00 | 107.0 |

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-parts_WO-2024-00508"></a>

### 그림 1 확대. WO-2024-00508 · 부품 연결 작업과 용접


![그림 1 확대. WO-2024-00508 · 부품 연결 작업과 용접](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/04_부품연결_WO-2024-00508.png)

기간: 2024/03/24 14:56–2024/03/25 00:36.

**관측:** 온도 0 LOT, 용접 14 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00508 | P1-CR1-CAL01 | BM | 03/24 14:59:00 | 03/24 15:56:00 | 03/24 23:36:00 | SGCC | SGCC | ELE | 인버터 알람 |

#### 부품교체

| WO_NO | PART_CD | PART_NM | QTY | CHG_RSN |
| --- | --- | --- | --- | --- |
| WO-2024-00508 | SEL-006 | 오일씰 150x190x18 | 2 | 손상 |

교체 자체의 시각은 미제공이며, 그림은 연결 작업 구간과 완료 시각을 표시한다.

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| BM | WO-2024-00508 | 11 | 03/24 16:19:59 | 03/24 17:47:02 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-parts_WO-2024-00597"></a>

### 그림 1 확대. WO-2024-00597 · 부품 연결 작업과 용접


![그림 1 확대. WO-2024-00597 · 부품 연결 작업과 용접](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/04_부품연결_WO-2024-00597.png)

기간: 2024/04/09 16:00–2024/04/09 20:09.

**관측:** 온도 0 LOT, 용접 12 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00597 | P1-CR1-CAL01 | TBM | 04/09 14:55:00 | 04/09 17:00:00 | 04/09 19:09:00 | SGCC | SGCC | — |   정기점검검 - 이상없음  |

#### 부품교체

| WO_NO | PART_CD | PART_NM | QTY | CHG_RSN |
| --- | --- | --- | --- | --- |
| WO-2024-00597 | SEL-009 | O링 세트 | 3 | 수명도래 |

교체 자체의 시각은 미제공이며, 그림은 연결 작업 구간과 완료 시각을 표시한다.

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| TBM | WO-2024-00597 | 5 | 04/09 17:02:22 | 04/09 17:32:13 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-parts_WO-2024-01178"></a>

### 그림 1 확대. WO-2024-01178 · 부품 연결 작업과 용접


![그림 1 확대. WO-2024-01178 · 부품 연결 작업과 용접](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/04_부품연결_WO-2024-01178.png)

기간: 2024/07/25 00:19–2024/07/25 11:55.

**관측:** 온도 0 LOT, 용접 16 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01178 | P1-CR1-CAL01 | BM | 07/24 23:33:00 | 07/25 01:19:00 | 07/25 10:55:00 | DP590 | DP590 | ELE | — |

#### 부품교체

| WO_NO | PART_CD | PART_NM | QTY | CHG_RSN |
| --- | --- | --- | --- | --- |
| WO-2024-01178 | INS-006 | 가스 분석계 셀 | 1 | 손상 |

교체 자체의 시각은 미제공이며, 그림은 연결 작업 구간과 완료 시각을 표시한다.

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| BM | WO-2024-01178 | 9 | 07/25 09:30:27 | 07/25 10:34:14 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

주황 선 하나는 COIL별 최초~최종 센서 기록 구간이다. COIL 사이의 빈 시간을 이어 그리지 않는다. COIL 내부 SEQ 기록 사이도 연속 전력 계측이나 아크가 켜진 시간으로 확정할 수 없다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-gap_202402_03"></a>

### 질문 2. 2월~3월 초 · 강종변경과 정비는 있고 센서 관측은 없는 구간


![질문 2. 2월~3월 초 · 강종변경과 정비는 있고 센서 관측은 없는 구간](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/05_관측공백_202402_03.png)

기간: 2024/02/01 00:00–2024/03/08 12:00.

**관측:** 온도 1 LOT, 용접 0 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 온도 관측

| LOT | 강종 | 관측 시작 | 관측 종료 | 길이(분) |
| --- | --- | --- | --- | --- |
| 240024 | SPFH590 | 03/07 21:22:00 | 03/08 06:29:48 | 547.8 |

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0020 | 02/02 07:17:00 | SGCC | SPCEN |
| EVT-2024-0023 | 02/04 14:42:00 | SPCEN | SPFH590 |
| EVT-2024-0024 | 02/06 06:51:00 | SPFH590 | SPHC |
| EVT-2024-0025 | 02/08 03:09:00 | SPHC | SPCEN |
| EVT-2024-0026 | 02/11 22:58:00 | SPCEN | SGCC |
| EVT-2024-0028 | 02/13 00:18:00 | SGCC | SPCE |
| EVT-2024-0030 | 02/16 18:39:00 | SPCE | SPCEN |
| EVT-2024-0034 | 02/19 03:51:00 | SPCEN | SPCD |
| EVT-2024-0037 | 02/22 15:42:00 | SPCD | SGCC |
| EVT-2024-0038 | 02/25 15:59:00 | SGCC | SPCD |
| EVT-2024-0041 | 02/28 06:51:00 | SPCD | SPHC |
| EVT-2024-0044 | 03/02 10:09:00 | SPHC | SPCE |
| EVT-2024-0046 | 03/04 06:30:00 | SPCE | SPCC |
| EVT-2024-0051 | 03/07 21:22:00 | SPCC | SPFH590 |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00211 | P1-CR1-CAL01 | BM | 02/09 15:31:00 | 02/09 15:52:00 | 02/09 23:22:00 | SPCEN | SPCEN | — | 1호기 재발, 원인 미상 |
| WO-2024-00220 | P1-CR1-CAL01 | TBM | 02/11 07:13:00 | 02/11 07:14:00 | 02/11 10:32:00 | SPCEN | SPCEN | — | 정기점검검 - 이상없음 |

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

##### 일별 용접 관측 COIL 수

| 날짜 | COIL 수 |
| --- | --- |

주황 막대 높이는 각 COIL의 화면 내 첫 기록 날짜 기준 관측 COIL 수다. 막대 폭은 하루 집계 폭이며 개별 용접 지속시간을 뜻하지 않는다. 하루 경계를 넘는 COIL은 한 번 집계한다. COIL별 정확한 구간과 겹침은 LOT 확대 그림·시간 대응 표에서 확인한다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-bm_WO-2024-01281"></a>

### 질문 3. SPCE 기록 중 BM · 08/14 · 센서 관측 확인


![질문 3. SPCE 기록 중 BM · 08/14 · 센서 관측 확인](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/06_BM_WO-2024-01281.png)

기간: 2024/08/12 00:00–2024/08/17 00:00.

**관측:** 온도 0 LOT, 용접 0 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0160 | 08/12 02:21:00 | SGCC | SPCE |
| EVT-2024-0163 | 08/14 12:05:00 | SPCE | SPHC |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01281 | P1-CR1-CAL01 | BM | 08/14 06:43:00 | 08/14 09:00:00 | 08/14 12:27:00 | SPCE | SPCE | WER | 마모량 초과 |

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

##### 일별 용접 관측 COIL 수

| 날짜 | COIL 수 |
| --- | --- |

주황 막대 높이는 각 COIL의 화면 내 첫 기록 날짜 기준 관측 COIL 수다. 막대 폭은 하루 집계 폭이며 개별 용접 지속시간을 뜻하지 않는다. 하루 경계를 넘는 COIL은 한 번 집계한다. COIL별 정확한 구간과 겹침은 LOT 확대 그림·시간 대응 표에서 확인한다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


<a id="context-bm_WO-2024-01646"></a>

### 질문 3. SPCE 기록 중 BM · 10/12 · 센서 관측 확인


![질문 3. SPCE 기록 중 BM · 10/12 · 센서 관측 확인](../figures/05_all_data_timeseries_jinny_01/05_사건연결/01_기간확대/06_BM_WO-2024-01646.png)

기간: 2024/10/10 00:00–2024/10/15 00:00.

**관측:** 온도 0 LOT, 용접 74 COIL. 센서 기록이 없다는 사실만으로 설비 정지를 판단할 수 없다.

**설비 범위:** 정비는 `EQP_TAG=P1-CR1-CAL01`(연속 소둔로)만 포함한다. 계획수리는 CR1 라인 캘린더, 용접은 WLD01 CSV다. 다른 설비의 TBM은 포함하지 않는다.

#### 강종변경 원문

| 이벤트 | 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| EVT-2024-0203 | 10/11 17:43:00 | SPCC | SPCE |
| EVT-2024-0207 | 10/14 06:46:00 | SPCE | SPCC |

#### 정비 기간·REMARK

| 작업번호 | 설비 | 정비 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01634 | P1-CR1-CAL01 | TBM | 10/10 14:37:00 | 10/10 16:59:00 | 10/10 18:18:00 | SPCC | SPCC | — | 점검 이상없음 |
| WO-2024-01646 | P1-CR1-CAL01 | BM | 10/12 15:06:00 | 10/12 16:25:00 | 10/12 18:44:00 | SPCE | SPCE | — | 현상 재현 안됨 |

#### 용접 관측과 시간 대응

| 종류_A | ID_A | COIL수 | 최초교집합 | 최종교집합 |
| --- | --- | --- | --- | --- |
| BM | WO-2024-01646 | 5 | 10/12 16:27:13 | 10/12 17:27:41 |

최초~최종 교집합 사이 전체가 연속 용접 시간은 아니다. COIL별 시간 대응은 메모리의 `tl_overlap`에서 계산하며 아래 마크다운 표로 확인한다.

#### 표시를 읽는 방법

청록 선은 LOT 온도 CSV의 최초~최종 기록 구간이다. 전체 생산 기간이나 온도 크기를 뜻하지 않는다. 선 옆에 관측 시각과 길이를 표기한다. ×–×는 CAL01 정비 착수~완료이며, 작업 길이는 설비 정지시간과 구분한다.

##### 일별 용접 관측 COIL 수

| 날짜 | COIL 수 |
| --- | --- |
| 10/10 00:00:00 | 32 |
| 10/12 00:00:00 | 42 |

주황 막대 높이는 각 COIL의 화면 내 첫 기록 날짜 기준 관측 COIL 수다. 막대 폭은 하루 집계 폭이며 개별 용접 지속시간을 뜻하지 않는다. 하루 경계를 넘는 COIL은 한 번 집계한다. COIL별 정확한 구간과 겹침은 LOT 확대 그림·시간 대응 표에서 확인한다.

가로 날짜 눈금은 서로 다른 날의 수리·관측·강종변경을 같은 달력에서 맞추기 위한 것이다. LOT 내부의 신호 비교에서는 경과 분을 사용한다. 중복된 축 제목은 생략했다.

**해석 범위:** 강종 띠는 마지막 강종변경 기록을 다음 변경까지 유지한 표시다. 계획수리 안의 변경 기록은 생산 재개·실제 소재 통과의 증거가 아니다. 동일 시간 관측도 LOT–COIL 직접 연결이나 고장 원인을 뜻하지 않는다.


### 정비·부품 원문은 마크다운으로

정비 44건과 교체 9건의 정확한 기간·감지·강종 기록·REMARK·교체사유를 이미지 표 대신 아래에 보존한다.
공란은 `기록 없음`으로 표시하고 원문 오탈자는 그대로 둔다.


<a id="analysis-records"></a>

### 정비 REMARK와 부품교체 원문

##### 1분기 정비 6건

| 작업번호 | 유형 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00060 | TBM | 01/11 15:19:00 | 01/11 18:43:00 | 01/11 20:32:00 | SGCC | SGCC | — | 기록 없음 |
| WO-2024-00211 | BM | 02/09 15:31:00 | 02/09 15:52:00 | 02/09 23:22:00 | SPCEN | SPCEN | — | 1호기 재발, 원인 미상 |
| WO-2024-00220 | TBM | 02/11 07:13:00 | 02/11 07:14:00 | 02/11 10:32:00 | SPCEN | SPCEN | — | 정기점검검 - 이상없음 |
| WO-2024-00378 | TBM | 03/10 15:10:00 | 03/10 15:46:00 | 03/10 18:36:00 | SPCEN | SPCEN | — | 1호기 재발, 정기점검 - 이상없음 |
| WO-2024-00385 | CBM | 03/11 15:01:00 | 03/18 23:13:00 | 03/19 04:59:00 | SPCEN | SPCC | WER | 마모 진행 |
| WO-2024-00508 | BM | 03/24 14:59:00 | 03/24 15:56:00 | 03/24 23:36:00 | SGCC | SGCC | ELE | 인버터 알람 |

##### 2분기 정비 14건

| 작업번호 | 유형 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00597 | TBM | 04/09 14:55:00 | 04/09 17:00:00 | 04/09 19:09:00 | SGCC | SGCC | — | 정기점검검 - 이상없음 |
| WO-2024-00627 | CBM | 04/14 15:04:00 | 04/23 15:27:00 | 04/23 18:29:00 | SPCE | DP590 | — | 확인 중 |
| WO-2024-00658 | CBM | 04/20 23:47:00 | 04/23 20:04:00 | 04/23 21:44:00 | SPCD | DP590 | MAL | 설비 트립 |
| WO-2024-00668 | CBM | 04/22 22:50:00 | 04/24 06:55:00 | 04/24 16:38:00 | DP590 | DP590 | ETC | 현상 재현 안됨 |
| WO-2024-00717 | BM | 05/03 23:46:00 | 05/04 04:50:00 | 05/04 08:15:00 | SPHC | SPHC | MAL | 동작 불량 |
| WO-2024-00747 | TBM | 05/08 07:53:00 | 05/08 11:38:00 | 05/08 12:30:00 | SPFH590 | SPFH590 | ELE | 기록 없음 |
| WO-2024-00776 | CBM | 05/14 07:17:00 | 05/19 12:14:00 | 05/19 16:05:00 | SPCC | SGCC | ETC | 재점검 필요 |
| WO-2024-00833 | BM | 05/25 15:55:00 | 05/25 19:06:00 | 05/26 05:01:00 | SPFH590 | SPFH590 | ETC | 1호기 재발, 확인 중 |
| WO-2024-00916 | TBM | 06/10 06:23:00 | 06/10 08:45:00 | 06/10 10:48:00 | DP590 | DP590 | — | 기록 없음 |
| WO-2024-00931 | TBM | 06/11 14:51:00 | 06/11 18:40:00 | 06/11 23:37:00 | SPCC | SPCC | WER | 마모량 초과 |
| WO-2024-00940 | TBM | 06/12 06:09:00 | 06/12 09:51:00 | 06/12 15:23:00 | SPCC | SPCC | — | 수리기간 중 점검 |
| WO-2024-00952 | TBM | 06/12 23:51:00 | 06/13 02:51:00 | 06/13 05:16:00 | SPCC | SPCC | — | 정기수리 부속작업 |
| WO-2024-00960 | TBM | 06/13 14:22:00 | 06/13 18:19:00 | 06/13 22:29:00 | DP590 | DP590 | — | 1호기 재발, SD작업 수행 |
| WO-2024-00972 | TBM | 06/14 14:09:00 | 06/14 15:35:00 | 06/14 18:03:00 | DP590 | DP590 | — | 1호기 재발, SD작업 수행 |

##### 3분기 정비 11건

| 작업번호 | 유형 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01094 | BM | 07/08 08:49:00 | 07/08 11:11:00 | 07/08 19:04:00 | SPHC | SPHC | ELE | 1호기 재발, 절연 저하 |
| WO-2024-01111 | TBM | 07/10 22:24:00 | 07/10 23:15:00 | 07/11 00:05:00 | DP590 | DP590 | — | 기록 없음 |
| WO-2024-01178 | BM | 07/24 23:33:00 | 07/25 01:19:00 | 07/25 10:55:00 | DP590 | DP590 | ELE | 기록 없음 |
| WO-2024-01259 | TBM | 08/09 22:35:00 | 08/10 01:48:00 | 08/10 06:20:00 | SPFH590 | SPFH590 | — | 점검 이상없음 |
| WO-2024-01266 | BM | 08/11 09:57:00 | 08/11 14:06:00 | 08/11 17:07:00 | SGCC | SGCC | MAL | 오작동 발생 |
| WO-2024-01281 | BM | 08/14 06:43:00 | 08/14 09:00:00 | 08/14 12:27:00 | SPCE | SPCE | WER | 마모량 초과 |
| WO-2024-01346 | BM | 08/27 07:37:00 | 08/27 11:03:00 | 08/28 04:27:00 | DP590 | DP590 | ELE | 1호기 재발, 모터 과열 |
| WO-2024-01419 | BM | 09/11 12:55:00 | 09/11 17:01:00 | 09/11 20:05:00 | SPHC | SGCC | ELE | 절연 저하 |
| WO-2024-01421 | TBM | 09/11 23:31:00 | 09/12 01:37:00 | 09/12 06:12:00 | SGCC | SGCC | — | 정기점검 - 이상없음 |
| WO-2024-01536 | BM | 09/22 15:25:00 | 09/22 19:36:00 | 09/22 20:42:00 | DP590 | DP590 | ETC | 1호기 재발, 확인 중 |
| WO-2024-01467 | CBM | 09/17 07:21:00 | 09/23 21:35:00 | 09/24 05:26:00 | SPHC | SPCD | — | 재점검 필요 |

##### 4분기 정비 13건

| 작업번호 | 유형 | 감지 | 착수 | 완료 | 감지 강종기록 | 착수 강종기록 | 고장형태 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01634 | TBM | 10/10 14:37:00 | 10/10 16:59:00 | 10/10 18:18:00 | SPCC | SPCC | — | 점검 이상없음 |
| WO-2024-01646 | BM | 10/12 15:06:00 | 10/12 16:25:00 | 10/12 18:44:00 | SPCE | SPCE | — | 현상 재현 안됨 |
| WO-2024-01678 | CBM | 10/18 15:50:00 | 10/26 23:53:00 | 10/27 02:42:00 | SPFH590 | SGCC | ETC | 원인 미상 |
| WO-2024-01775 | TBM | 11/06 07:49:00 | 11/06 10:40:00 | 11/06 12:23:00 | SPCEN | SPCEN | — | 정기수리 실시 |
| WO-2024-01794 | TBM | 11/07 09:20:00 | 11/07 12:53:00 | 11/07 14:14:00 | SPCEN | SPCEN | — | 정기수리 부속작업 |
| WO-2024-01823 | TBM | 11/09 06:05:00 | 11/09 09:58:00 | 11/09 13:02:00 | SPCE | SPCE | — | 1호기 재발, 정기수리 실시 |
| WO-2024-01826 | TBM | 11/09 07:39:00 | 11/09 11:15:00 | 11/09 18:20:00 | SPCE | SPCE | — | 수리기간 중 점검 |
| WO-2024-01834 | TBM | 11/10 07:00:00 | 11/10 10:40:00 | 11/10 12:52:00 | SPCE | SPCE | — | 정기수리 부속작업 |
| WO-2024-01859 | TBM | 11/13 07:13:00 | 11/13 07:40:00 | 11/13 09:35:00 | SPCD | SPCD | — | 계획 점검 수행 |
| WO-2024-01854 | CBM | 11/12 15:00:00 | 11/13 08:54:00 | 11/13 12:44:00 | SPFH590 | SPCD | WER | 마모량 초과 |
| WO-2024-01922 | CBM | 11/25 14:19:00 | 11/30 04:19:00 | 11/30 07:12:00 | SPCE | SPCD | MAL | 1호기 재발, 설비 트립 |
| WO-2024-01999 | TBM | 12/09 23:43:00 | 12/10 01:50:00 | 12/10 05:55:00 | SPCC | SPCC | — | 기록 없음 |
| WO-2024-02105 | CBM | 12/28 15:54:00 | 12/28 16:54:00 | 12/28 23:33:00 | SPCE | SPCE | WER | 마모량 초과 |

##### 부품 명칭과 교체 내용

가스 분석계 셀은 계장 부품, 오일씰·O링 세트는 실링(밀봉) 부품으로 등록되어 있다.
가스 종류·셀 설치 위치·치수 단위는 원본에 명시되지 않는다.
개별 교체 시각도 없으므로 연결 작업의 시간 구간과 완료 시각을 사용한다.
6/10 오일씰·O링은 같은 작업번호의 두 교체 기록이며 개별 교체의 동시성은 알 수 없다.

| 부품코드 | 품명 | 분류 | 적용설비 |
| --- | --- | --- | --- |
| SEL-006 | 오일씰 150x190x18 | 실링 | 공통 |
| SEL-009 | O링 세트 | 실링 | 공통 |
| INS-006 | 가스 분석계 셀 | 계장 | 연속로 |

| 연결 작업번호 | 부품코드 | 품명 | 수량 | 교체사유 | 연결 작업 착수 | 연결 작업 완료 | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00508 | SEL-006 | 오일씰 150x190x18 | 2 | 손상 | 03/24 15:56:00 | 03/24 23:36:00 | 인버터 알람 |
| WO-2024-00597 | SEL-009 | O링 세트 | 3 | 수명도래 | 04/09 17:00:00 | 04/09 19:09:00 | 정기점검검 - 이상없음 |
| WO-2024-00776 | INS-006 | 가스 분석계 셀 | 3 | 수명도래 | 05/19 12:14:00 | 05/19 16:05:00 | 재점검 필요 |
| WO-2024-00916 | SEL-006 | 오일씰 150x190x18 | 2 | 수명도래 | 06/10 08:45:00 | 06/10 10:48:00 | 기록 없음 |
| WO-2024-00916 | SEL-009 | O링 세트 | 1 | 손상 | 06/10 08:45:00 | 06/10 10:48:00 | 기록 없음 |
| WO-2024-01094 | SEL-009 | O링 세트 | 3 | 예방 | 07/08 11:11:00 | 07/08 19:04:00 | 1호기 재발, 절연 저하 |
| WO-2024-01178 | INS-006 | 가스 분석계 셀 | 1 | 손상 | 07/25 01:19:00 | 07/25 10:55:00 | 기록 없음 |
| WO-2024-01346 | SEL-006 | 오일씰 150x190x18 | 1 | 손상 | 08/27 11:03:00 | 08/28 04:27:00 | 1호기 재발, 모터 과열 |
| WO-2024-02105 | INS-006 | 가스 분석계 셀 | 3 | 예방 | 12/28 16:54:00 | 12/28 23:33:00 | 마모량 초과 |


In [ ]:
# %% 질문 1 그림: 모든 강종·모든 BM을 동일하게 표시한다.
fig,axes=plt.subplots(1,2,figsize=(21,7),gridspec_kw={'width_ratios':[1,2.8]})
fig.subplots_adjust(left=.07,right=.98,top=.80,bottom=.12,wspace=.22)
grades=iq_grade_summary.sort_values(['BM_착수건수','GRADE'],ascending=[False,True]).GRADE.tolist()
for i,g in enumerate(grades):
    n=int(iq_bm.START_GRADE.eq(g).sum()); yy=len(grades)-i-1
    color='#bd2c45' if g=='SPCE' else '#487dad'
    axes[0].barh(yy,n,height=.55,color=color)
    axes[0].text(n+.06,yy,str(n),va='center',fontsize=11,color=color)
    for task in iq_bm.loc[iq_bm.START_GRADE.eq(g)].itertuples():
        axes[1].scatter(task.STRT_DT,yy,marker='x',s=55,color=color)
        axes[1].annotate(task.WO_NO[-5:],(task.STRT_DT,yy),xytext=(0,11),textcoords='offset points',ha='center',fontsize=9,color=color)
for ax in axes:
    ax.set_ylim(-.7,len(grades)-.1);ax.set_yticks(range(len(grades)),grades[::-1]);tl_style(ax)
axes[0].set_xlim(0,3.6);axes[0].set_xticks(range(4));axes[0].set_xlabel('사후정비 [건]')
axes[0].set_title('전체 8강종 비교',loc='left',pad=15,fontsize=12)
axes[1].set_title('전체 BM 12건의 착수일 · 작업번호 끝 5자리',loc='left',pad=15,fontsize=12)
axes[1].set_xlim(YEAR_START,YEAR_END);axes[1].set_xticks(pd.date_range(YEAR_START,periods=12,freq='MS'))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m월'))
fig.suptitle('질문 1. SPCE에 사후정비가 집중됐다는 근거는 부족하다',x=.025,y=.98,ha='left',fontsize=17,weight='bold',color='#17344d')
fig.text(.025,.91,'SPCE 2/12건 · SGCC·DP590 각각 3건 · SPCE 온도 LOT와 BM 작업 겹침 0건',fontsize=11,color='#456176')
fig.savefig(IQ_FIG/'01D_SPCE_BM_강종기록_시간대응.png',dpi=160,facecolor='white');plt.show()


<a id="analysis-questions"></a>

### 그림 1의 결론과 확인할 질문


[그림 1의 결론과 확인할 질문](../figures/05_all_data_timeseries_jinny_01/05_사건연결/02_관계사례/01D_SPCE_BM_강종기록_시간대응.png)

#### SPCE는 확인할 후보이며, 사후정비 집중은 아직 확인되지 않았다

- **확인한 사실:** 착수 시각의 강종 기록 기준 SPCE BM은 8/14와 10/12의 2건이다. SGCC·DP590은 각각 3건이다. 건수는 생산량·가동시간을 보정하지 않은 값이다.
- **온도로 검증 가능한가:** SPCE 관측 LOT 240037(4/13), 240121(11/24), 240133(12/28)은 두 BM 작업과 겹치지 않는다. 8월에는 온도 CSV 기록 자체가 없다. 세 LOT의 온도 차이를 이 BM들의 전조·결과로 연결할 수 없다.
- **다음 확인:** 해당 두 BM 창의 강종변경·REMARK·용접 관측을 먼저 본다. 원인 비교에는 같은 강종의 정비 직전/중/직후 온도와 생산량·가동시간·설정 이력이 추가로 필요하다.
- **연결 시각의 영향:** 9/11 BM은 감지 때 SPHC, 착수 때 SGCC다. 그림에서 가까운 LOT 라벨로 강종을 붙이지 않는다.

#### 정기수리 안의 강종변경은 무엇인가

| PLAN_ID | 변경 시각 | 이전 | 이후 |
| --- | --- | --- | --- |
| PLAN-2024-02 | 06/11 08:00:00 | DP590 | SPCC |
| PLAN-2024-02 | 06/13 10:34:00 | SPCC | DP590 |
| PLAN-2024-04 | 11/07 14:08:00 | SPCEN | SPCE |
| PLAN-2024-04 | 11/10 17:02:00 | SPCE | SPFH590 |

6월에는 DP590→SPCC→DP590, 11월에는 SPCEN→SPCE→SPFH590 기록이 수리 안에 존재한다.
두 수리 구간 안에는 CAL01 온도와 WLD01 용접 관측이 없다. 이벤트 기록의 시각·상태 정의를 확인하기 전 생산 중 강종전환으로 해석하지 않는다.
6월 수리 전 LOT 240058은 DP590, 수리 후 240061은 SPHC라서 직접적인 같은 강종 전후 비교가 아니다.
11월 전 240114는 SPCEN, 후 240117은 SPCD이며 역시 강종 조건이 다르다.

#### 2월~3월 초 공백은 무엇부터 확인할까

2/1 00:00~3/7 21:22 직전에는 온도 0행·용접 0행, 강종변경 13건, CAL01 정비 2건(BM 1·TBM 1)이 있다.
2/9 BM의 당시 강종 기록은 SPCEN이며 REMARK는 `1호기 재발, 원인 미상`이다.
온도·용접 파일의 추출/표본 선정 범위, 수집 로그, 이벤트 기록 시각의 의미를 먼저 확인한다.
현재 CSV만으로 수집 누락·발췌 제공·실제 미가동 중 어느 이유인지 결정할 수 없다. 이 공백을 온도 0이나 보간된 신호로 채우지 않는다.

#### 그림 1에서 내릴 수 있는 결론

1. 센서 온도는 연속 연간 측정이 아니라 14개의 관측 창(합계 98.9시간)이다. 관측 창 밖은 온도 기반 진단이 불가하다.
2. 실제 LOT–정비 작업 겹침은 LOT 240117의 CBM·TBM뿐이며 두 작업끼리는 41분 겹친다. LOT 240133은 감지만 관측 중이고 작업은 종료 후다.
3. 용접과 실제 관측 시간이 겹친 LOT는 240108·240114·240117·240121의 네 개다. 이들은 공동 시간 그래프를 볼 우선 후보다.
4. 정기수리·부품교체는 맥락과 시간 관계를 제공한다. 강종·설정·관측 범위가 달라 수리 효과나 강종별 고장률을 그림 1만으로 판단할 수 없다.

#### 분석 진행 순서

연간 관계 → 질문별 창(수리·BM·공백) → 강종별 공정 비교 → 개별 LOT·TC/OP/CP → 필요한 원본 초 단위 확대 순서로 진행한다.
아래 LOT 탐색 기록에는 무엇을 봤고, 어느 센서·시간에서 단서를 찾았으며, 어떤 비교를 더 해야 하는지를 남긴다.


### 그림 1 이후의 확인 경로

**사실:** SPCE 기록 중 BM 착수는 2건이며, 센서가 있는 SPCE 3 LOT와 겹치지 않는다.
2~3월 초에는 온도·용접 관측이 모두 없고 이벤트·정비는 있다. 두 정기수리 안에도 강종변경 기록이 있다.
센서 미관측의 이유·수리 중 강종변경의 업무 의미는 원본 CSV로 확정할 수 없다.

질문별 확대 → 같은 강종·공정 비교 → 개별 LOT의 TC/OP/CP → 필요한 초 단위 구간 순으로 본다.
아래 공정 분석 뒤의 **LOT 탐색 기록**에서 후보를 고른 근거와 다음 확인 항목을 확인한다.
그림 목차와 분석 해석은 이 노트북의 [LOT 탐색 기록과 공정별 목차](#analysis-journey)에 보존한다.


# 2. 데이터 품질과 관측 가능 범위

## CAL01 질문

- LOT 내부 sampling interval은 안정적인가?
- 결측은 어떤 LOT/센서에 집중되는가?
- 데이터 수집문제를 공정 이상으로 오판할 가능성이 있는가?

## WLD01 질문

- 모든 COIL이 39개 SEQ를 가지는가?
- PWR=0은 어느 COIL에 있는가?
- 관측일은 연중 얼마나 제한적인가?

In [ ]:
# CAL01 interval
temp_q = temp.sort_values(["LOT_NO", "MEAS_DT"]).copy()
temp_q["DT_SEC"] = (
    temp_q.groupby("LOT_NO")["MEAS_DT"]
          .diff()
          .dt.total_seconds()
)

interval_summary = (
    temp_q.groupby("LOT_NO")
          .agg(
              N=("MEAS_DT", "size"),
              MEDIAN_DT_SEC=("DT_SEC", "median"),
              MAX_DT_SEC=("DT_SEC", "max"),
              IRREGULAR_RATE=("DT_SEC", lambda s: (s.dropna() > 1).mean() * 100)
          )
          .reset_index()
)

**결과 — LOT별 수집 간격**

| 행 | LOT_NO | N | MEDIAN_DT_SEC | MAX_DT_SEC | IRREGULAR_RATE |
| --- | --- | --- | --- | --- | --- |
| 0 | 240001 | 40402 | 1.0 | 3.0 | 2.247469 |
| 1 | 240024 | 31729 | 1.0 | 11.0 | 3.558371 |
| 2 | 240037 | 35348 | 1.0 | 3.0 | 3.751379 |
| 3 | 240038 | 7342 | 1.0 | 10.0 | 1.961586 |
| 4 | 240044 | 4334 | 1.0 | 2.0 | 3.346411 |
| 5 | 240058 | 12559 | 1.0 | 3.0 | 3.583373 |
| 6 | 240061 | 13299 | 1.0 | 2.0 | 1.691984 |
| 7 | 240069 | 46836 | 1.0 | 11.0 | 4.043984 |
| 8 | 240091 | 37633 | 1.0 | 6.0 | 2.221514 |
| 9 | 240108 | 26982 | 1.0 | 3.0 | 3.591416 |
| 10 | 240114 | 18763 | 1.0 | 6.0 | 2.004051 |
| 11 | 240117 | 25708 | 1.0 | 3.0 | 3.683822 |
| 12 | 240121 | 38237 | 1.0 | 3.0 | 2.183806 |
| 13 | 240133 | 6218 | 1.0 | 3.0 | 3.635194 |

In [ ]:
# WLD01 구조 품질
wld_seq = (
    wld.groupby("COIL_ID")["SEQ_NO"]
       .agg(["count", "nunique", "min", "max"])
)

wld_quality = pd.Series({
    "행 수": len(wld),
    "COIL 수": wld["COIL_ID"].nunique(),
    "실제 관측일": wld["MEAS_DT"].dt.date.nunique(),
    "결측 총합": int(wld.isna().sum().sum()),
    "완전 중복행": int(wld.duplicated().sum()),
    "39 SEQ 완전 COIL": int(
        (
            (wld_seq["count"] == 39)
            & (wld_seq["nunique"] == 39)
            & (wld_seq["min"] == 1)
            & (wld_seq["max"] == 39)
        ).sum()
    )
})

**결과 — WLD01 수집 구조**

| 행 | 값 |
| --- | --- |
| 행 수 | 140985 |
| COIL 수 | 3615 |
| 실제 관측일 | 77 |
| 결측 총합 | 0 |
| 완전 중복행 | 0 |
| 39 SEQ 완전 COIL | 3615 |

## 그림 2. 어디까지 관측됐는지 먼저 확인

2A는 월별 강종변경 기록과 실제 온도 LOT 수를 분리한다. 변경 기록은 생산량이 아니다. 2B는 14 LOT 각각의 관측일·길이·기록 간격이다. 센서가 없는 시간에 발생한 작업은 온도로 검증할 수 없다는 범위를 먼저 정한다.

In [ ]:
# 그림 2 준비: 조업에 기록된 강종 전체와 실제 센서 관측을 분리한다.
from pathlib import Path
import textwrap
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

RT_ROOT = Path(OUT_DIR)
RT_PROCESS_DIR = Path(GRADE_DIR) / "02_센서별"
RT_PROCESS_DIR.mkdir(parents=True, exist_ok=True)
RT_JOINT_DIR = Path(EVENT_DIR) / "03_온도용접대조"
RT_JOINT_DIR.mkdir(parents=True, exist_ok=True)
RT_COLORS = ["#087eaa", "#e57c12", "#16856b", "#b93b64", "#7841bc"]
RT_YEAR0, RT_YEAR1 = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
RT_TEMP_SIGNALS = [c for c in temp if c.startswith("TC-") or c == "CLN-IN"]
RT_SIGNALS = [c for c in temp if c not in ["MEAS_DT", "LOT_NO"]]
rt_all_grades = sorted(set(grade_evt["BEF_VAL"].dropna()) | set(grade_evt["AFT_VAL"].dropna()) |
                       set(lot_info["GRADE"].dropna()))
rt_lot_color = {}
for grade, frame in lot_info.groupby("GRADE"):
    rt_lot_color.update({int(lot): RT_COLORS[i % len(RT_COLORS)]
                         for i, lot in enumerate(frame.sort_values("LOT_START")["LOT_NO"])})

rt_observation_rows = []
rt_gaps = []
for lot, frame in temp.sort_values(["LOT_NO", "MEAS_DT"]).groupby("LOT_NO"):
    info = lot_info.loc[lot_info["LOT_NO"].eq(lot)].iloc[0]
    dt = frame["MEAS_DT"].diff().dt.total_seconds()
    expected = int((info.LOT_END - info.LOT_START).total_seconds()) + 1
    missing_seconds = expected - frame["MEAS_DT"].nunique()
    rt_observation_rows.append({"LOT_NO": int(lot), "GRADE": info.GRADE,
        "START": info.LOT_START, "END": info.LOT_END, "DURATION_MIN": info.DURATION_MIN,
        "N_ROWS": len(frame), "EXPECTED_1S": expected, "MISSING_1S": missing_seconds,
        "DT_MEDIAN_S": dt.median(), "DT_MAX_S": dt.max(),
        "MISSING_CELLS": int(frame[RT_SIGNALS].isna().sum().sum()),
        "SAMPLE_COVERAGE_PCT": 100 * frame["MEAS_DT"].nunique() / expected})
    for idx in frame.index[dt.gt(1)]:
        pos = frame.index.get_loc(idx)
        rt_gaps.append({"LOT_NO": int(lot), "GAP_START": frame.iloc[pos - 1].MEAS_DT,
                        "GAP_END": frame.loc[idx, "MEAS_DT"], "INTERVAL_S": float(dt.loc[idx])})
rt_observation = pd.DataFrame(rt_observation_rows)
rt_gaps = pd.DataFrame(rt_gaps)
rt_grade_rows = []
for grade in rt_all_grades:
    observed = rt_observation.loc[rt_observation["GRADE"].eq(grade)]
    rt_grade_rows.append({"GRADE": grade, "GRADE_EVENTS": int(grade_evt["AFT_VAL"].eq(grade).sum()),
        "SENSOR_LOTS": len(observed), "OBS_MIN": observed["DURATION_MIN"].sum(),
        "OBS_HOURS": observed["DURATION_MIN"].sum() / 60,
        "SENSOR_ROWS": int(observed["N_ROWS"].sum()),
        "MISSING_1S": int(observed["MISSING_1S"].sum()),
        "LOT_LIST": ", ".join(observed["LOT_NO"].astype(str))})
rt_grade_table = pd.DataFrame(rt_grade_rows)
grade_lot = (lot_info.groupby("GRADE")["LOT_NO"].agg(N_LOT="nunique",
              LOT_LIST=lambda s: ", ".join(str(int(x)) for x in s)).sort_values("N_LOT", ascending=False))

def rt_style(ax, time=True):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#acb9c6")
    ax.tick_params(labelsize=9, color="#acb9c6")
    ax.grid(color="#e5ecf2", lw=.65, alpha=.9)
    ax.set_axisbelow(True)

def rt_finish(fig, filename, show=True, dpi=160):
    """저장 전 제목·본문 경계와 명시한 라벨 그룹의 충돌을 확인한다."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for group in getattr(fig, "_collision_groups", []):
        boxes = [artist.get_window_extent(renderer).expanded(1.015, 1.015) for artist in group]
        if any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:]):
            raise RuntimeError(f"라벨 충돌: {filename}")
    common_text = list(fig.texts) + [ax.xaxis.label for ax in fig.axes
                                    if ax.axison and ax.xaxis.label.get_text()]
    boxes = [artist.get_window_extent(renderer) for artist in common_text]
    if any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:]):
        raise RuntimeError(f"제목·설명·시간축 라벨 충돌: {filename}")
    for artist in fig.texts:
        b = artist.get_window_extent(renderer)
        if b.x0 < -1 or b.x1 > fig.bbox.x1 + 1 or b.y0 < -1 or b.y1 > fig.bbox.y1 + 1:
            raise RuntimeError(f"그림 경계를 넘는 설명: {filename}")
    # 범례/제목 경계와 빨간 메모가 패널 밖으로 나가는지 확인한다.
    for legend in fig.legends:
        bounds = legend.get_window_extent(renderer)
        if any(bounds.overlaps(t.get_window_extent(renderer)) for t in fig.texts):
            raise RuntimeError(f"범례와 제목·설명 충돌: {filename}")
        if bounds.x0 < 0 or bounds.x1 > fig.bbox.x1 or bounds.y0 < 0 or bounds.y1 > fig.bbox.y1:
            raise RuntimeError(f"범례 경계 초과: {filename}")
    for ax in fig.axes:
        if hasattr(ax,"_review_note"):
            note = ax._review_note.get_bbox_patch().get_window_extent(renderer) if ax._review_note.get_bbox_patch() is not None else ax._review_note.get_window_extent(renderer)
            legend = ax.get_legend()
            if legend is not None and note.overlaps(legend.get_window_extent(renderer)):
                raise RuntimeError(f'빨간 상자와 범례 충돌: {filename}')
            panel = ax.get_window_extent(renderer)
            if note.x0 < panel.x0 or note.x1 > panel.x1 or note.y0 < panel.y0 or note.y1 > panel.y1:
                raise RuntimeError(f"빨간 확인 설명 경계 초과: {filename}")
            if hasattr(ax,'_review_xy'):
                point=ax.transData.transform(ax._review_xy)
                dx=max(note.x0-point[0],0,point[0]-note.x1)
                dy=max(note.y0-point[1],0,point[1]-note.y1)
                if np.hypot(dx,dy)>76*fig.dpi/72:
                    raise RuntimeError(f"확인 상자의 연결선이 너무 깁니다: {filename}")
    target = Path(filename)
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=dpi, facecolor="white")
    if show:
        plt.show()
    else:
        plt.close(fig)

def rt_header(fig, title, subtitle):
    fig.suptitle(title, x=.025, y=.98, ha="left", fontsize=16, fontweight="bold", color="#17344d")
    fig.text(.025, .925, subtitle, ha="left", fontsize=10, color="#456176")

def rt_minute_line(ax, lot, sensor, start=None, end=None, envelope=True, lw=1.5):
    frame = rt_profiles[int(lot)]
    if start is not None:
        frame = frame.loc[frame.index >= start]
    if end is not None:
        frame = frame.loc[frame.index <= end]
    color = rt_lot_color[int(lot)]
    if envelope:
        ax.fill_between(frame.index.to_numpy(float), frame[(sensor, "min")].to_numpy(float),
                        frame[(sensor, "max")].to_numpy(float), color=color, alpha=.12, lw=0)
    ax.plot(frame.index, frame[(sensor, "median")], color=color, lw=lw, label=str(int(lot)))

def rt_lot_legend(fig, lots, y=.875):
    handles = []
    for row in lot_info.loc[lot_info["LOT_NO"].isin(lots)].itertuples():
        handles.append(Line2D([], [], color=rt_lot_color[int(row.LOT_NO)], lw=2,
            label=f"LOT {int(row.LOT_NO)} · {row.DURATION_MIN:.1f}분"))
    legend = fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(.975, y),
               ncol=min(len(lots), 2), fontsize=9, frameon=False, columnspacing=1.6)
    legend._legend_box.align = "right"

**결과 — 강종별 기록과 센서 관측 범위**

| 행 | GRADE | GRADE_EVENTS | SENSOR_LOTS | OBS_MIN | SENSOR_ROWS | MISSING_1S | LOT_LIST |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | DP590 | 12 | 1 | 216.883333 | 12559 | 455 | 240058 |
| 1 | SGCC | 15 | 1 | 688.500000 | 40402 | 909 | 240001 |
| 2 | SPCC | 16 | 0 | 0.000000 | 0 | 0 |  |
| 3 | SPCD | 16 | 1 | 444.316667 | 25708 | 952 | 240117 |
| 4 | SPCE | 23 | 3 | 1369.916667 | 79803 | 2395 | 240037, 240121, 240133 |
| 5 | SPCEN | 15 | 2 | 443.966667 | 26105 | 535 | 240038, 240114 |
| 6 | SPFH590 | 17 | 2 | 1013.683333 | 58711 | 2112 | 240024, 240108 |
| 7 | SPHC | 20 | 4 | 1753.783333 | 102102 | 3129 | 240044, 240061, 240069, 240091 |

In [ ]:
# 그림 2A: 월별 기록과 실제 온도 관측을 같은 크기의 두 격자로 분리한다.
fig,axes=plt.subplots(1,2,figsize=(21,8))
fig.subplots_adjust(left=.075,right=.97,top=.78,bottom=.12,wspace=.19)
coverage_matrices=[]
for ax,frame,datecol,title,kind in [
    (axes[0],grade_evt,'EVT_DT','강종변경 기록 [건]','event'),
    (axes[1],rt_observation,'START','온도 관측 [LOT]','sensor')]:
    matrix=(frame.assign(MONTH=frame[datecol].dt.month).groupby(['GRADE' if kind=='sensor' else 'AFT_VAL','MONTH']).size()
        .unstack(fill_value=0).reindex(index=rt_all_grades,columns=range(1,13),fill_value=0))
    coverage_matrices.append(matrix)
    ax.imshow(matrix.values,aspect='auto',cmap='Blues',vmin=0,vmax=max(4,matrix.values.max()))
    for i,g in enumerate(rt_all_grades):
        for j in range(12):
            n=int(matrix.iloc[i,j]);text_value=str(n) if n else '·'
            ax.text(j,i,text_value,ha='center',va='center',fontsize=13,
                color='white' if n>=max(4,matrix.values.max())*.55 else '#17344d')
    ax.set_xticks(range(12),[f'{m:02}월' for m in range(1,13)])
    ax.set_yticks(range(len(rt_all_grades)),rt_all_grades,fontsize=11)
    ax.set_xticks(np.arange(-.5,12,1),minor=True);ax.set_yticks(np.arange(-.5,len(rt_all_grades),1),minor=True)
    ax.grid(which='minor',color='white',lw=2);ax.tick_params(which='both',length=0)
    ax.set_title(title,loc='left',fontsize=13,pad=15)
    for spine in ax.spines.values():spine.set_visible(False)
rt_header(fig,'그림 2A. 강종 기록이 있어도 온도 관측은 없을 수 있다',
    '강종변경 134건 / 8종 · 온도 14 LOT / 7종 · SPCC 온도 관측 없음 · 2월·8월 온도 관측 없음')
rt_finish(fig,Path(OBS_DIR)/'02A_강종별_조업기록_센서관측.png')

# 그림 2B: 각 LOT의 실제 길이, 1초 기록 커버리지, 최대 간격 구간을 함께 본다.
ordered = rt_observation.sort_values(["GRADE", "START"]).reset_index(drop=True)
n = len(ordered)
fig, axes = plt.subplots(1, 3, figsize=(22, 12), gridspec_kw={"width_ratios": [2.5, 2, 2.2]})
fig.subplots_adjust(left=.16, right=.98, top=.81, bottom=.13, wspace=.18)
gap_annotations = []
for i, lot in ordered.iterrows():
    yy = n - i - 1
    color = rt_lot_color[int(lot.LOT_NO)]
    axes[0].barh(yy, lot.DURATION_MIN, color=color, height=.45)
    axes[0].text(lot.DURATION_MIN + 9, yy, f"{lot.DURATION_MIN:.1f}분", fontsize=9, va="center")
    axes[1].barh(yy, lot.N_ROWS, color="#d1e6f1", height=.45)
    axes[1].barh(yy, lot.MISSING_1S, left=lot.N_ROWS, color="#d68564", height=.45)
    axes[1].text(0, yy + .3, f"{lot.N_ROWS:,}행 · 누락 {lot.MISSING_1S:,}초 · {lot.SAMPLE_COVERAGE_PCT:.2f}%",
                 fontsize=8.1, color="#456176")
    raw = temp.loc[temp["LOT_NO"].eq(lot.LOT_NO)].sort_values("MEAS_DT")
    dt = raw["MEAS_DT"].diff().dt.total_seconds()
    idx = dt.idxmax()
    loc = raw.index.get_loc(idx)
    anchor = raw.iloc[loc - 1].MEAS_DT
    subset = raw.loc[raw["MEAS_DT"].between(anchor - pd.Timedelta(seconds=5), anchor + pd.Timedelta(seconds=16))]
    minutes = (subset["MEAS_DT"] - anchor).dt.total_seconds() / 60
    axes[2].scatter(minutes, [yy] * len(subset), marker="|", s=60, lw=1, color=color)
    axes[2].plot([0, lot.DT_MAX_S / 60], [yy - .17] * 2, color="#bd6147", lw=1.3)
    gap_annotations.append(axes[2].text(15.8 / 60, yy + .23, f"{anchor:%m/%d %H:%M:%S} · 간격 {lot.DT_MAX_S / 60:.3f}분",
                                       ha="right", fontsize=8.3, color="#8f503d"))
labels = [f"{r.GRADE} · LOT {r.LOT_NO}\n{r.START:%m/%d %H:%M} ~ {r.END:%m/%d %H:%M}" for r in ordered.itertuples()]
axes[0].set_yticks(range(n), labels[::-1], fontsize=8.6)
for ax in axes:
    ax.set_ylim(-.6, n - .2)
    rt_style(ax)
for ax in axes[1:]:
    ax.set_yticks([])
axes[0].set_xlim(0, ordered["DURATION_MIN"].max() * 1.18)
axes[0].set_title("각 LOT의 관측 길이", pad=18, fontsize=11)
axes[0].set_xlabel("LOT 시작 후 경과시간 [분]")
axes[1].set_title("1초 격자 기준 기록 / 미관측", pad=18, fontsize=11)
axes[1].set_xlabel("기록된 시각 수 + 누락된 1초 시각 수")
axes[1].ticklabel_format(axis="x", style="plain")
axes[2].set_title("LOT별 최대 간격 전후의 원본 기록", pad=18, fontsize=10)
axes[2].set_xlim(-5.5 / 60, 17 / 60)
axes[2].set_xticks(np.array([-5, 0, 5, 10, 15]) / 60)
axes[2].set_xlabel("최대 간격 시작 시각 대비 경과시간 [분]")
rt_header(fig, "그림 2B. 강종별 · LOT별 관측 길이와 기록 간격",
          "관측 길이와 최대 간격은 분 · 가운데는 1초 격자 기준 기록 수")
fig._collision_groups = [gap_annotations]
rt_finish(fig, Path(OBS_DIR) / "02B_강종별_LOT별_관측시간_초단위.png")

두 격자의 숫자는 각각 **변경 기록 건수**와 **온도 LOT 수**다. 점(·)은 해당 파일에서 0건이다. 생산량이나 가동률을 비교하는 그림이 아니다. 아래 그림 2B에서 각 LOT의 날짜와 관측 길이를 확인한다.


## 그림 3. 같은 강종의 공정·센서를 실제 경과 분으로 비교

위 패널명은 `공정 [센서]`, y축은 관측량·단위다. 빨간 실제 관측점에서 짧은 확인 상자로 연결한다. LOT 색상 범례는 오른쪽에 있다. OP는 %, CP는 무차원이다. 선은 관측 종료까지만 그리며, 회색선은 모든 비교 LOT가 관측된 공통 경과 분의 끝이다. 온도 ℃는 기존 신호 해석을 따른다.

공정 범위는 **세정 [CLN-IN]**, **건조 [TC-Z1·TC-Z2]**, **가열·균열 [TC-Z3~TC-Z6]**, **과시효 [TC-Z7~TC-Z9]**, **냉각 [TC-OUT1·TC-OUT2]**다. 요약 제목에는 중앙값 여부를 표시하고 개별 센서 제목에는 실제 센서명을 적는다. 세정은 CLN-IN 단일 신호이므로 TC 구간을 만들지 않는다.


In [ ]:
PROCESS_GROUPS = {
    "세정": ["CLN-IN"],
    "건조": ["TC-Z1", "OP-Z1", "TC-Z2", "OP-Z2"],
    "가열·균열": [
        "TC-Z3", "OP-Z3",
        "TC-Z4", "OP-Z4", "CP-Z4",
        "TC-Z5", "OP-Z5",
        "TC-Z6", "OP-Z6"
    ],
    "과시효": ["TC-Z7", "OP-Z7", "TC-Z8", "OP-Z8", "TC-Z9"],
    "냉각": ["TC-OUT1", "TC-OUT2"]
}

TC_GROUPS = {
    "세정": ["CLN-IN"],
    "건조": ["TC-Z1", "TC-Z2"],
    "가열·균열": ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"],
    "과시효": ["TC-Z7", "TC-Z8", "TC-Z9"],
    "냉각": ["TC-OUT1", "TC-OUT2"]
}

# 표시 이름에 실제 열을 명시한다. 세정에는 원자료에 없는 TC 번호를 만들지 않는다.
RT_PROCESS_RANGES={
    '세정':'CLN-IN',
    '건조':'TC-Z1·TC-Z2',
    '가열·균열':'TC-Z3~TC-Z6',
    '과시효':'TC-Z7~TC-Z9',
    '냉각':'TC-OUT1·TC-OUT2'}

def rt_process_label(process,statistic=False):
    suffix=' 중앙값' if statistic and len(TC_GROUPS[process])>1 else ''
    return f'{process} [{RT_PROCESS_RANGES[process]}{suffix}]'

In [ ]:
# 시간은 LOT 시작 후 실제 초/분이다. 진행률·리샘플링에 의한 시간 늘이기는 사용하지 않는다.
temp_flow = temp.merge(lot_info[["LOT_NO", "LOT_START", "LOT_END", "DURATION_MIN", "GRADE"]],
                       on="LOT_NO", how="left", validate="many_to_one")
temp_flow["ELAPSED_SEC"] = (temp_flow["MEAS_DT"] - temp_flow["LOT_START"]).dt.total_seconds()
temp_flow["ELAPSED_MIN"] = temp_flow["ELAPSED_SEC"] / 60
temp_flow["MINUTE"] = np.floor(temp_flow["ELAPSED_MIN"]).astype(int)
rt_profiles = {}
rt_process_profiles = {}
for lot, frame in temp_flow.groupby("LOT_NO"):
    grid = pd.RangeIndex(0, int(frame["MINUTE"].max()) + 1, name="MINUTE")
    rt_profiles[int(lot)] = frame.groupby("MINUTE")[RT_SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    # 공정 중앙값은 지도용으로만 사용한다. 개별 TC·OP·CP는 아래 별도 패널에 모두 보존한다.
    summary = pd.DataFrame({name: frame[sensors].median(axis=1)
                            for name, sensors in TC_GROUPS.items()})
    summary["MINUTE"] = frame["MINUTE"]
    rt_process_profiles[int(lot)] = summary.groupby("MINUTE").agg(["median", "min", "max"]).reindex(grid)
rt_common_minutes = (lot_info.groupby("GRADE")["DURATION_MIN"].min().apply(np.floor).astype(int).to_dict())
eligible_grades = lot_info["GRADE"].dropna().value_counts().index.tolist()

def rt_events_for_lot(lot_id):
    info = lot_info.loc[lot_info["LOT_NO"].eq(lot_id)].iloc[0]
    records = []
    tasks = cal_maint.loc[(cal_maint["STRT_DT"] <= info.LOT_END) &
                          (cal_maint["END_DT"] >= info.LOT_START)]
    for task in tasks.itertuples():
        a0, a1 = max(info.LOT_START, task.STRT_DT), min(info.LOT_END, task.END_DT)
        records.append({"kind": task.MNT_TYPE, "id": task.WO_NO,
                        "start": (a0 - info.LOT_START).total_seconds() / 60,
                        "end": (a1 - info.LOT_START).total_seconds() / 60})
    return records

In [ ]:
# %% 확인 포인트: 실제 관측에서 검토할 LOT·센서·경과 분을 정한다.
# 탐색 위치이며 고장 라벨이나 관리 한계가 아니다. 선정 이유는 아래 마크다운에 기록한다.
RT_RED = "#cc2535"
rt_review_rows = []

def rt_review_add(lot, sensor, start, end, short, why, follow, method="minute"):
    info = lot_info.set_index("LOT_NO").loc[lot]
    if method in ["raw_max", "raw_extreme"]:
        raw = temp_flow.loc[temp_flow.LOT_NO.eq(lot) & temp_flow.ELAPSED_MIN.between(start, end)]
        center = temp_flow.loc[temp_flow.LOT_NO.eq(lot),sensor].median()
        point = raw.loc[raw[sensor].idxmax() if method == "raw_max" else (raw[sensor]-center).abs().idxmax()]
        minute, value = float(point.ELAPSED_MIN), float(point[sensor])
    else:
        values = rt_profiles[lot][(sensor, "median")].loc[start:end-(1 if method=="minute" else 0)].dropna()
        if values.empty:
            return
        peers = lot_info.loc[lot_info.GRADE.eq(info.GRADE) & lot_info.LOT_NO.ne(lot), "LOT_NO"]
        if len(peers):
            reference = pd.concat([rt_profiles[int(p)][(sensor,"median")] for p in peers], axis=1).median(axis=1)
            delta = (values - reference.reindex(values.index)).abs()
            minute = float(delta.idxmax()) if delta.notna().any() else float(values.index[0])
        else:
            minute = float(values.index[-1])
        value = float(values.loc[minute])
    process = next(p for p, sensors in PROCESS_GROUPS.items() if sensor in sensors)
    rt_review_rows.append(dict(GRADE=info.GRADE, LOT_NO=lot, PROCESS=process, SENSOR=sensor,
        START_MIN=float(start), END_MIN=float(end), POINT_MIN=minute, VALUE=value,
        SHORT=short, WHY=why, FOLLOW=follow, METHOD=method))

# 앞선 원본 판독에서 확인한 단서. 주기 차이를 지속적인 수준 이상으로 해석하지 않는다.
rt_review_add(240061, "TC-Z3", 0, 5, "초기 승온 확인",
    "같은 강종 다른 3 LOT의 중앙값과 처음 5분 평균 차이가 약 42.8℃*다.",
    "TC-Z3와 OP-Z3의 초기 상승을 비교하고 수리 후 첫 관측이라는 날짜 맥락을 확인한다.")
rt_review_add(240061, "TC-OUT1", 0, 5, "초기 냉각값 비교",
    "같은 SPHC 다른 3 LOT의 중앙값과 처음 5분 평균 차이가 약 29.6℃*다.",
    "승온 구간과 함께 보고 냉각 신호의 초기조건과 이후 주기를 확인한다.")
rt_review_add(240037, "TC-Z9", 29, 34, "피크 시점 비교",
    "SPCE 공통 107분에서 차이가 큰 구간이다. 전체 관측에서는 240121과 약 197분 주기 및 범위가 비슷하다.",
    "전체 주기 그림에서 위상 차이를 먼저 확인하고 TC-Z7·Z8과 비교한다.")
rt_review_add(240133, "TC-OUT1", 77, 82, "냉각 위상 비교",
    "공통 107분의 종료-초기 변화가 +11.44℃*로 다른 SPCE의 음수 변화와 방향이 다르다.",
    "TC-OUT2와 같은 시각의 움직임 및 주기를 확인한다. 107.4분의 짧은 관측 길이를 고려한다.")
rt_review_add(240121, "TC-OUT2", 0, 5, "초기 변동폭 확인",
    "공통 107분 TC-OUT2 IQR이 14.543℃*로 SPCE 3 LOT 중 가장 크다.",
    "초기 과도응답과 반복 냉각 진폭을 원본 기록에서 구분한다.")

# 두 LOT가 있는 강종: 가열/냉각 각각에서 같은 경과 분의 평균 차이가 가장 큰 연속 5분.
for grade in ["SPCEN", "SPFH590"]:
    lots = lot_info.loc[lot_info.GRADE.eq(grade), "LOT_NO"].astype(int).tolist()
    common = rt_common_minutes[grade]
    for process in ["가열·균열", "냉각"]:
        best = None
        for sensor in TC_GROUPS[process]:
            matrix = pd.concat([rt_profiles[l][(sensor,"median")].rename(l) for l in lots],axis=1).reindex(range(common))
            difference = (matrix[lots[0]] - matrix[lots[1]]).abs().rolling(5,min_periods=5).mean()
            if difference.notna().any() and (best is None or difference.max() > best[0]):
                best = (float(difference.max()), sensor, int(difference.idxmax())+1)
        difference, sensor, end = best
        # 두 LOT 비교는 대칭이다. 반복 검증이 되지 않았으므로 후기 LOT를 표시 대상으로 고정한다.
        rt_review_add(lots[-1], sensor, end-5, end, "두 LOT 차이 확인",
            f"{process} TC 중 두 LOT의 같은 경과 분 차이가 가장 큰 5분: 평균 절대 차이 {difference:.2f}℃*.",
            f"LOT {lots[0]}과 {lots[1]}의 {sensor} 및 대응 OP를 비교한다. 조건 차이와 주기 위상을 확인한다.")

# 한 LOT뿐인 강종: 반복 LOT 비교 대신 가열 TC의 가장 큰 5분 변화 위치를 탐색한다.
for grade in ["SGCC", "DP590", "SPCD"]:
    lot = int(lot_info.loc[lot_info.GRADE.eq(grade),"LOT_NO"].iloc[0])
    best = None
    for sensor in TC_GROUPS["가열·균열"]:
        change = rt_profiles[lot][(sensor,"median")].diff(5).abs()
        if change.notna().any() and (best is None or change.max() > best[0]):
            best = (float(change.max()), sensor, int(change.idxmax()))
    difference, sensor, end = best
    rt_review_add(lot, sensor, end-5, end, "5분 변화 확인",
        f"가열 TC 중 5분 전 대비 변화가 가장 큰 위치: {difference:.2f}℃*. 동일 강종 관측 LOT는 하나다.",
        f"{sensor}와 대응 OP의 변화 순서·반복성을 본다. 다른 LOT보다 특이하다는 판단은 할 수 없다.",method="change_5min")

raw61 = temp_flow.loc[temp_flow.LOT_NO.eq(240061)]
cp61 = raw61.loc[raw61["CP-Z4"].idxmax()]
rt_review_add(240061,"CP-Z4",cp61.ELAPSED_MIN-.5,cp61.ELAPSED_MIN+.5,"CP 최댓값·OP 확인",
    f"원본 CP-Z4 최댓값 {cp61['CP-Z4']:.4g}, 같은 시각 OP-Z3={cp61['OP-Z3']:.4g}%.",
    "133.1분 주변 TC-Z3·OP-Z3·CP-Z4의 변화 순서를 원본 기록에서 함께 본다. 모든 SPHC의 공통 74분 밖이다.",method="raw_max")

# 기존 CP 원본 극값 표시는 빨간 포인트로 보존한다. 각 LOT의 중앙값에서 가장 먼 원본 관측이다.
for grade in eligible_grades:
    raw = temp_flow.loc[temp_flow.GRADE.eq(grade)]
    center = raw.groupby("LOT_NO")["CP-Z4"].transform("median")
    departure = (raw["CP-Z4"]-center).abs()
    p = raw.loc[departure.idxmax()]
    median = center.loc[p.name]
    duration = float(lot_info.set_index("LOT_NO").loc[p.LOT_NO,"DURATION_MIN"])
    rt_review_add(int(p.LOT_NO),"CP-Z4",max(0,p.ELAPSED_MIN-.5),min(duration,p.ELAPSED_MIN+.5),
        "CP 원본 극값 확인",
        f"LOT 내 CP-Z4 중앙값 {median:.6g} 대비 원본 값 {p['CP-Z4']:.6g}, 차이 {p['CP-Z4']-median:+.6g}. 강종 안에서 이 차이가 가장 큰 관측이다.",
        "최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다.",
        method="raw_extreme")
    if grade == "DP590":
        rt_review_rows[-1]["WHY"] += " 이 LOT에서는 CP가 관측 후반에 하강하며 TC·OP도 종료 쪽에서 낮아지는 형태다."
        rt_review_rows[-1]["FOLLOW"] = "약 130분 이후 CP 하강과 TC·OP의 변화 순서를 확인한다. 종료 단계의 조건 전환인지 판단할 운전 상태 기록이 필요하다."

rt_review_points = pd.DataFrame(rt_review_rows)

def rt_points_for(lots, sensor):
    selected = rt_review_points.loc[rt_review_points.LOT_NO.isin(lots) & rt_review_points.SENSOR.eq(sensor)]
    if sensor == "CP-Z4" and len(selected):
        selected = selected.sort_values("METHOD",key=lambda x:x.ne("raw_extreme"))
    if len(selected) or not sensor.startswith("OP-"):
        return selected.head(1)
    tc = sensor.replace("OP-", "TC-", 1)
    sources = rt_review_points.loc[rt_review_points.LOT_NO.isin(lots) & rt_review_points.SENSOR.eq(tc)]
    if sensor == "OP-Z4":
        sources = rt_review_points.loc[rt_review_points.LOT_NO.isin(lots) & rt_review_points.SENSOR.eq("CP-Z4") & rt_review_points.METHOD.eq("raw_extreme")]
    elif sensor == "OP-Z3" and 240061 in lots:
        sources = rt_review_points.loc[rt_review_points.LOT_NO.eq(240061) & rt_review_points.SENSOR.eq("CP-Z4")]
    selected = sources.head(1).copy()
    if len(selected):
        p = selected.iloc[0]
        if p.METHOD in ["raw_max", "raw_extreme"]:
            value = temp_flow.loc[temp_flow.LOT_NO.eq(p.LOT_NO) & temp_flow.ELAPSED_MIN.eq(p.POINT_MIN),sensor].iloc[0]
        else:
            value = rt_profiles[int(p.LOT_NO)].loc[int(p.POINT_MIN),(sensor,"median")]
        selected.loc[:,"VALUE"] = float(value)
        selected.loc[:,"SHORT"] = f"{p.SENSOR} 확인 시점"
        selected.loc[:,"SENSOR"] = sensor
    return selected

def rt_sensor_label(sensor):
    # 센서 식별자는 패널 제목에, 관측량·단위는 y축에 둔다.
    return sensor

def rt_measure_label(sensor):
    if sensor in RT_TEMP_SIGNALS: return '온도 [℃]'
    if sensor.startswith('OP-'): return 'OP [%]'
    if sensor.startswith('CP-'): return 'CP'
    return sensor

def rt_focus(ax, points, process_profile=False, label=None):
    """점은 현재 패널의 실제 신호 값에 놓고, 짧은 연결선으로 설명한다."""
    if not len(points): return
    point=points.iloc[0];lot=int(point.LOT_NO)
    minute,value=float(point.POINT_MIN),float(point.VALUE)
    if process_profile:
        minute=int(np.floor(minute))
        value=rt_process_profiles[lot].loc[minute,(point.PROCESS,'median')]
    ax.axvspan(point.START_MIN,point.END_MIN,color=RT_RED,alpha=.07,lw=0,zorder=0)
    ax.scatter(minute,value,s=40,facecolors='white',edgecolors=RT_RED,lw=1.6,zorder=8)
    source=rt_process_label(point.PROCESS,statistic=True) if process_profile else point.SENSOR
    caption=label or f'LOT {lot} · {minute:.1f}분\n{source} · {point.SHORT}'
    return rt_point_box(ax,(minute,value),caption)


def rt_point_box(ax,xy,caption):
    """점 가까이의 빈 영역을 선택한다. 반대편 끝으로 설명선을 늘리지 않는다."""
    fig=ax.figure
    # 축 범위를 확정한 후 화면 거리로 배치한다.
    ax.get_xlim();ax.get_ylim();fig.canvas.draw()
    renderer=fig.canvas.get_renderer();panel=ax.get_window_extent(renderer)
    from matplotlib.transforms import Bbox
    point=ax.transData.transform(xy)
    samples=[]
    for line in ax.lines:
        try:
            pts=line.get_transform().transform(line.get_xydata())
            pts=pts[np.isfinite(pts).all(axis=1)]
            if len(pts): samples.append(pts[::max(1,len(pts)//2000)])
        except (TypeError,ValueError): pass
    samples=np.concatenate(samples) if samples else np.empty((0,2))
    legend=ax.get_legend()
    legend_box=legend.get_window_extent(renderer).expanded(1.03,1.08) if legend else None
    ghost=ax.text(0,0,caption,fontsize=8.5,linespacing=1.25)
    size=ghost.get_window_extent(renderer);ghost.remove()
    pad=5*fig.dpi/72;ww=size.width+2*pad;hh=size.height+2*pad
    options=[]
    # Text-box corners/edges are anchored just outside the marked point.
    for gap in [12,24,40,60]:
        g=gap*fig.dpi/72
        for sx,sy in [(1,1),(1,-1),(-1,1),(-1,-1),(1,0),(-1,0),(0,1),(0,-1)]:
            left=point[0]+g if sx==1 else point[0]-g-ww if sx==-1 else point[0]-ww/2
            bottom=point[1]+g if sy==1 else point[1]-g-hh if sy==-1 else point[1]-hh/2
            # Small clamping allows an edge point to retain a short leader.
            left=np.clip(left,panel.x0+7,panel.x1-ww-7)
            bottom=np.clip(bottom,panel.y0+7,panel.y1-hh-7)
            box=Bbox.from_bounds(left,bottom,ww,hh)
            if box.x1>panel.x1-2 or box.y1>panel.y1-2: continue
            dx=max(box.x0-point[0],0,point[0]-box.x1)
            dy=max(box.y0-point[1],0,point[1]-box.y1)
            distance=np.hypot(dx,dy)
            # Keep the point outside the label; cap the leader length at 72 pt.
            if distance<6 or distance>72*fig.dpi/72: continue
            if legend_box is not None and box.overlaps(legend_box): continue
            inside=(samples[:,0]>=box.x0-3)&(samples[:,0]<=box.x1+3)&(samples[:,1]>=box.y0-3)&(samples[:,1]<=box.y1+3)
            # Curve clearance first, then the distance to the point. All choices are local.
            penalty=inside.sum()*3+distance*.35
            options.append((penalty,left+pad,bottom+pad,distance))
    if not options:
        raise RuntimeError(f'확인 상자를 점 가까이 배치할 공간이 없습니다: {caption}')
    _,xx,yy,_=min(options,key=lambda item:item[0])
    position=ax.transAxes.inverted().transform((xx,yy))
    note=ax.annotate(caption,xy=xy,xytext=position,textcoords='axes fraction',
        ha='left',va='bottom',fontsize=8.5,color=RT_RED,linespacing=1.25,
        bbox=dict(boxstyle='round,pad=.35',fc='white',ec=RT_RED,lw=.75,alpha=.98),
        arrowprops=dict(arrowstyle='-',color=RT_RED,lw=.9,shrinkA=3,shrinkB=4),zorder=10)
    ax._review_note=note;ax._review_xy=xy
    return note


In [ ]:
# 원본 OP 결측과 1분 요약의 표시를 분리한다. 결측을 0이나 보간값으로 채우지 않는다.
rt_op_missing_rows = []
for lot_id, raw in temp_flow.sort_values(["LOT_NO", "MEAS_DT"]).groupby("LOT_NO"):
    for sensor in [s for s in RT_SIGNALS if s.startswith("OP-")]:
        missing = raw[sensor].isna()
        groups = missing.ne(missing.shift(fill_value=False)).cumsum()
        for _, run in raw.loc[missing].groupby(groups.loc[missing]):
            rt_op_missing_rows.append(dict(LOT_NO=int(lot_id), SENSOR=sensor,
                START_MIN=float(run.ELAPSED_MIN.iloc[0]), END_MIN=float(run.ELAPSED_MIN.iloc[-1]),
                START_DT=run.MEAS_DT.iloc[0], END_DT=run.MEAS_DT.iloc[-1], N_MISSING=len(run)))
rt_op_missing = pd.DataFrame(rt_op_missing_rows, columns=["LOT_NO", "SENSOR", "START_MIN",
    "END_MIN", "START_DT", "END_DT", "N_MISSING"])


def rt_mark_op_missing(ax, lot, sensor):
    """회색 빗금은 원본 NaN 구간, 아래 눈금은 짧은 결측의 위치다."""
    if not sensor.startswith("OP-"):
        return
    hits = rt_op_missing.loc[rt_op_missing.LOT_NO.eq(int(lot)) & rt_op_missing.SENSOR.eq(sensor)]
    duration = float(lot_info.set_index("LOT_NO").loc[int(lot), "DURATION_MIN"])
    for run in hits.itertuples():
        # 1초 표본이 차지하는 폭만 표시하며, 데이터 값이나 축 범위는 바꾸지 않는다.
        right = min(duration, run.END_MIN + 1 / 60)
        ax.axvspan(run.START_MIN, right, facecolor="#e5e7eb", edgecolor="#858e99",
                   hatch="////", alpha=.35, lw=.4, zorder=.4)
        ax.plot([(run.START_MIN + right) / 2] * 2, [.015, .075],
                transform=ax.get_xaxis_transform(), color="#737d89", lw=1.1, zorder=2)
    if len(hits):
        ax._op_missing_segments = getattr(ax, "_op_missing_segments", []) + list(hits.itertuples())


def rt_signal_points(lots, sensors):
    frames = [rt_points_for(lots, sensor) for sensor in sensors]
    frames = [frame for frame in frames if len(frame)]
    return pd.concat(frames, ignore_index=True).drop_duplicates(
        ["LOT_NO", "SENSOR", "START_MIN", "END_MIN", "METHOD"]) if frames else rt_review_points.iloc[:0]


def rt_review_caption(points):
    if not len(points):
        return None
    p = points.iloc[0]
    return f"LOT {int(p.LOT_NO)} · {p.POINT_MIN:.1f}분\n{p.SENSOR} · 비교 확인 시점"


def rt_signal_notes(lots, sensors, points, process_summary=False):
    """PNG에도 남는 표시 설명·원본 결측·비교 구간 선정 이유."""
    notes = ["선: 1분 중앙값 · 옅은 색 띠: 같은 1분의 최소~최대 (신뢰구간이나 정상 범위가 아님)"]
    if process_summary:
        notes.append("왼쪽은 같은 시각의 공정 센서 중앙값을 1분 요약한 값과 그 범위, 오른쪽은 개별 센서의 값과 범위다.")
    if len(points):
        notes.append("빨간 음영: 비교 확인 구간 · 빨간 원: 선정 시점 (이상·고장 판정이 아님)")
    op_sensors = [sensor for sensor in sensors if sensor.startswith("OP-")]
    if op_sensors:
        hits = rt_op_missing.loc[rt_op_missing.LOT_NO.isin(lots) & rt_op_missing.SENSOR.isin(op_sensors)]
        if len(hits):
            notes.append("회색 빗금·아래 회색 눈금: 원본 OP 결측 · 0% 측정값과 구분하며, 1분 중앙값은 그 분의 유효값으로 계산한다.")
            for (lot, sensor), frame in hits.groupby(["LOT_NO", "SENSOR"]):
                intervals = ", ".join(f"{r.START_MIN:.2f}–{r.END_MIN:.2f}분" for r in frame.itertuples())
                notes.append(f"원본 결측: LOT {int(lot)} · {sensor} · {intervals} · {int(frame.N_MISSING.sum())}행 / {len(frame)}구간")
        else:
            notes.append("이 그림의 OP 원본 결측: 없음 · OP 0%는 기록된 측정값이다.")
    if len(points):
        for (lot, start, end, why), frame in points.groupby(
                ["LOT_NO", "START_MIN", "END_MIN", "WHY"], sort=False):
            names = "·".join(frame.SENSOR.drop_duplicates())
            notes.append(f"비교 확인 구간 선정 이유 — LOT {int(lot)} · {names} · {start:g}–{end:g}분: {why}")
    lines = []
    for note in notes:
        lines.extend(textwrap.wrap(note, width=112, subsequent_indent="    ") or [""])
    return lines, .22 * len(lines) + .22


def rt_add_signal_notes(fig, lines, note_height, left=.075):
    height = fig.get_figheight()
    artists = []
    for i, line in enumerate(lines):
        artists.append(fig.text(left, (note_height - .10 - .22 * i) / height, line,
            ha="left", va="top", fontsize=8.8, color="#435b6c"))
    fig._signal_note_artists = artists


def rt_signal_header(fig, title, subtitle, lots):
    """아래 설명이 늘어나도 제목·범례의 물리적 여백을 유지한다."""
    rt_header(fig, title, subtitle)
    height = fig.get_figheight()
    fig._suptitle.set_y(1 - .28 / height)
    fig.texts[-1].set_y(1 - .82 / height)
    rt_lot_legend(fig, lots, y=1 - .95 / height)


<a id="op-missing-display"></a>

#### OP 결측값의 표시와 원본 기록

OP 패널의 **회색 빗금과 아래 회색 눈금**은 CSV에 값이 없는 원본 결측 위치다. 짧은 결측도 눈금으로 표시하며, 결측을 0이나 보간값으로 채우지 않는다. 실제 OP 0%는 그대로 유지한다.

1분 전체가 결측이면 선이 비고, 일부만 결측이면 남은 유효값으로 1분 중앙값을 계산한다. 아래 시간은 결측이 기록된 첫 시각과 마지막 시각이며, 표의 행 수는 원본 NaN 행 수다. 센서 기록 행 자체가 없는 시간과는 구분한다.

| LOT | 센서 | 첫 결측 시각 | 마지막 결측 시각 | 경과 분 | 원본 결측 행 수 |
| --- | --- | --- | --- | --- | --- |
| 240024 | OP-Z7 | 03/07 21:22:00 | 03/07 21:22:01 | 0.000–0.017 | 2 |
| 240024 | OP-Z8 | 03/07 21:22:00 | 03/07 21:22:00 | 0.000–0.000 | 1 |
| 240037 | OP-Z7 | 04/13 23:16:00 | 04/13 23:16:04 | 0.000–0.067 | 4 |
| 240037 | OP-Z8 | 04/13 23:16:00 | 04/13 23:16:04 | 0.000–0.067 | 4 |
| 240038 | OP-Z7 | 04/16 11:35:00 | 04/16 11:35:00 | 0.000–0.000 | 1 |
| 240038 | OP-Z8 | 04/16 11:35:00 | 04/16 11:35:01 | 0.000–0.017 | 2 |
| 240044 | OP-Z3 | 05/03 18:50:00 | 05/03 18:51:21 | 0.000–1.350 | 75 |
| 240044 | OP-Z7 | 05/03 18:50:00 | 05/03 18:50:08 | 0.000–0.133 | 8 |
| 240044 | OP-Z8 | 05/03 18:50:00 | 05/03 18:50:06 | 0.000–0.100 | 6 |
| 240058 | OP-Z7 | 06/09 05:26:00 | 06/09 05:26:01 | 0.000–0.017 | 2 |
| 240058 | OP-Z8 | 06/09 05:26:00 | 06/09 05:26:00 | 0.000–0.000 | 1 |
| 240061 | OP-Z3 | 06/15 01:04:00 | 06/15 01:07:03 | 0.000–3.050 | 179 |
| 240069 | OP-Z8 | 07/07 16:57:00 | 07/07 16:57:00 | 0.000–0.000 | 1 |
| 240091 | OP-Z7 | 09/08 22:03:00 | 09/08 22:03:00 | 0.000–0.000 | 1 |
| 240108 | OP-Z7 | 10/18 04:05:00 | 10/18 04:05:00 | 0.000–0.000 | 1 |
| 240108 | OP-Z8 | 10/18 04:05:00 | 10/18 04:05:01 | 0.000–0.017 | 2 |
| 240133 | OP-Z7 | 12/28 14:57:00 | 12/28 14:57:00 | 0.000–0.000 | 1 |

원본 OP 결측은 총 **291행**이다. 이 중 OP-Z3는 LOT 240044에서 75행, LOT 240061에서 179행이다. 공정별 온도 요약(03A)은 온도만 그리므로 OP 결측 표시는 해당 OP 센서 패널(03B·04A)에 둔다.


<a id="analysis-rules"></a>

### 빨간 확인 포인트와 표시 규칙

#### 공정·센서 범위

| 공정 | 표시·계산에 사용한 온도 신호 |
|---|---|
| 세정 | CLN-IN 단일 센서 |
| 건조 | TC-Z1·TC-Z2 중앙값 |
| 가열·균열 | TC-Z3~TC-Z6 중앙값 |
| 과시효 | TC-Z7~TC-Z9 중앙값 |
| 냉각 | TC-OUT1·TC-OUT2 중앙값 |

공정 중앙값은 먼저 같은 시각의 위 센서들을 집계하고, 이후 같은 1분 안에서 중앙값을 구한다. 개별 센서 그림은 해당 센서만의 **1분 중앙값 선**과 **같은 1분의 최소~최대 띠**다. 공정 요약의 띠는 같은 시각의 공정 센서 중앙값을 먼저 구한 뒤, 그 값의 1분 내 최소~최대로 계산한다. 띠는 신뢰구간이나 정상 범위가 아니다. 원본 초 해상도의 확대는 별도로 명시한다.

#### 빨간 비교 확인 구간: 왜, 어디를 확인하는가

빨간 테두리 점은 실제 관측 위치이며 연결선이 짧은 확인 상자로 이어진다. 옅은 빨간 구간은 **비교 확인 구간**, 빨간 원은 그 안의 **선정 시점**이다. 선정 이유는 각 공정 요약·센서별·LOT별 그림 아래에도 적는다.
LOT의 선 색상은 유지하고, 빨간색은 확인 포인트에 사용한다. 색상 범례는 오른쪽에 배치했다.
이는 관측 탐색 위치이며 고장·관리 한계 초과 판정이 아니다.

- **SPHC:** 240061의 초기 승온과 냉각 초기값, 133.1분의 CP 최댓값 및 같은 시각 OP를 함께 본다. 240069 CP-Z4의 574.48분 원본 값 0.909111도 별도 확대한다.
- **SPCE:** 240037 TC-Z9의 피크 시점, 240133 냉각 방향, 240121 냉각 변동폭을 비교한다. 전체 주기와 위상을 먼저 확인한다. 240037 CP-Z4의 41.98분 원본 값 0.60676은 원본 해상도로 확대한다.
- **SPCEN·SPFH590:** 각 두 LOT의 가열/냉각에서 같은 경과 분 차이가 가장 큰 5분을 표시했다. 후기 LOT는 위치 표시 대상이며 이상 LOT로 판정한 것이 아니다.
- **SGCC·DP590·SPCD:** 반복 LOT가 없으므로 가열 TC의 가장 큰 5분 변화 위치만 표시했다. 강종 간 또는 다른 LOT 대비 이상 판정은 할 수 없다.
- **각 강종의 CP:** LOT 안의 중앙값과 차이가 가장 큰 원본 관측도 표시했다. DP590 240058에서는 관측 후반의 CP 하강과 TC·OP의 동반 하강을 확인한다. 1분 중앙값에 가려진 짧은 값은 원본 해상도에서 확인한다.

##### 그래프 밖에서 알아둘 표시 규칙

- 경과시간은 **분**으로 통일한다. 원본 1초 기록의 해상도를 유지한 확대도 가로축은 분이다. 연간/정비 대응 그림은 실제 날짜·시각을 사용한다.
- **OP는 %**, **CP는 무차원**이다. CP 축에 단위 괄호를 붙이지 않는다. CP-Z4M의 신호 정의는 아직 확인되지 않았다.
- TC의 ℃*는 기존 분석 단위 가정이다. 선은 1분 중앙값, 띠는 같은 분의 최소~최대이며 CP 원본 최댓값은 띠 안의 원본 값을 표시한다.
- 관측이 끝난 LOT의 선은 연장하지 않는다. 결측 분을 채우지 않으며, 회색 점선은 모든 LOT를 같은 길이로 비교할 수 있는 공통 경과 분의 끝이다.
- 그림 3A 왼쪽은 제목에 명시한 센서들의 공정 중앙값이고 빨간 음영은 확인할 시간창이다. 오른쪽 확대는 해당 개별 센서이며 빨간 점도 그 센서의 값이다. 집계 중앙값을 개별 TC 값으로 표시하지 않는다.
- 가열 단서의 대응 OP 패널에서는 같은 시점을 함께 표시한다. OP-Z4에는 해당 LOT의 CP 극값 시점을, SPHC 240061 OP-Z3에는 133.1분의 CP 관측 시점을 표시한다. 동시 관측은 인과관계나 제어 이상을 뜻하지 않는다.
- 그림 4는 공통 시간의 중앙값·종료-초기 10분 차이·IQR을 비교한다. 파란 구간은 처음/마지막 10분, 빨간 구간은 다른 LOT 중앙값과 평균 차이가 큰 5분이다. 행마다 요약값 범위가 가장 큰 TC를 선택한다.

| 강종 | LOT | 공정·센서 | 경과 분 | 왜 확인하는가 | 어디를 볼 것인가 |
| --- | --- | --- | --- | --- | --- |
| SPHC | 240061 | 가열·균열 · TC-Z3 | 0–5 | 같은 강종 다른 3 LOT의 중앙값과 처음 5분 평균 차이가 약 42.8℃*다. | TC-Z3와 OP-Z3의 초기 상승을 비교하고 수리 후 첫 관측이라는 날짜 맥락을 확인한다. |
| SPHC | 240061 | 냉각 · TC-OUT1 | 0–5 | 같은 SPHC 다른 3 LOT의 중앙값과 처음 5분 평균 차이가 약 29.6℃*다. | 승온 구간과 함께 보고 냉각 신호의 초기조건과 이후 주기를 확인한다. |
| SPCE | 240037 | 과시효 · TC-Z9 | 29–34 | SPCE 공통 107분에서 차이가 큰 구간이다. 전체 관측에서는 240121과 약 197분 주기 및 범위가 비슷하다. | 전체 주기 그림에서 위상 차이를 먼저 확인하고 TC-Z7·Z8과 비교한다. |
| SPCE | 240133 | 냉각 · TC-OUT1 | 77–82 | 공통 107분의 종료-초기 변화가 +11.44℃*로 다른 SPCE의 음수 변화와 방향이 다르다. | TC-OUT2와 같은 시각의 움직임 및 주기를 확인한다. 107.4분의 짧은 관측 길이를 고려한다. |
| SPCE | 240121 | 냉각 · TC-OUT2 | 0–5 | 공통 107분 TC-OUT2 IQR이 14.543℃*로 SPCE 3 LOT 중 가장 크다. | 초기 과도응답과 반복 냉각 진폭을 원본 기록에서 구분한다. |
| SPCEN | 240114 | 가열·균열 · TC-Z3 | 31–36 | 가열·균열 TC 중 두 LOT의 같은 경과 분 차이가 가장 큰 5분: 평균 절대 차이 12.19℃*. | LOT 240038과 240114의 TC-Z3 및 대응 OP를 비교한다. 조건 차이와 주기 위상을 확인한다. |
| SPCEN | 240114 | 냉각 · TC-OUT1 | 81–86 | 냉각 TC 중 두 LOT의 같은 경과 분 차이가 가장 큰 5분: 평균 절대 차이 19.63℃*. | LOT 240038과 240114의 TC-OUT1 및 대응 OP를 비교한다. 조건 차이와 주기 위상을 확인한다. |
| SPFH590 | 240108 | 가열·균열 · TC-Z3 | 449–454 | 가열·균열 TC 중 두 LOT의 같은 경과 분 차이가 가장 큰 5분: 평균 절대 차이 14.12℃*. | LOT 240024과 240108의 TC-Z3 및 대응 OP를 비교한다. 조건 차이와 주기 위상을 확인한다. |
| SPFH590 | 240108 | 냉각 · TC-OUT1 | 195–200 | 냉각 TC 중 두 LOT의 같은 경과 분 차이가 가장 큰 5분: 평균 절대 차이 26.66℃*. | LOT 240024과 240108의 TC-OUT1 및 대응 OP를 비교한다. 조건 차이와 주기 위상을 확인한다. |
| SGCC | 240001 | 가열·균열 · TC-Z3 | 596–601 | 가열 TC 중 5분 전 대비 변화가 가장 큰 위치: 16.01℃*. 동일 강종 관측 LOT는 하나다. | TC-Z3와 대응 OP의 변화 순서·반복성을 본다. 다른 LOT보다 특이하다는 판단은 할 수 없다. |
| DP590 | 240058 | 가열·균열 · TC-Z3 | 117–122 | 가열 TC 중 5분 전 대비 변화가 가장 큰 위치: 11.05℃*. 동일 강종 관측 LOT는 하나다. | TC-Z3와 대응 OP의 변화 순서·반복성을 본다. 다른 LOT보다 특이하다는 판단은 할 수 없다. |
| SPCD | 240117 | 가열·균열 · TC-Z3 | 226–231 | 가열 TC 중 5분 전 대비 변화가 가장 큰 위치: 11.41℃*. 동일 강종 관측 LOT는 하나다. | TC-Z3와 대응 OP의 변화 순서·반복성을 본다. 다른 LOT보다 특이하다는 판단은 할 수 없다. |
| SPHC | 240061 | 가열·균열 · CP-Z4 | 132.6–133.6 | 원본 CP-Z4 최댓값 0.4709, 같은 시각 OP-Z3=0%. | 133.1분 주변 TC-Z3·OP-Z3·CP-Z4의 변화 순서를 원본 기록에서 함께 본다. 모든 SPHC의 공통 74분 밖이다. |
| SPHC | 240069 | 가열·균열 · CP-Z4 | 573.983–574.983 | LOT 내 CP-Z4 중앙값 0.450109 대비 원본 값 0.909111, 차이 +0.459002. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |
| SPCE | 240037 | 가열·균열 · CP-Z4 | 41.4833–42.4833 | LOT 내 CP-Z4 중앙값 0.450097 대비 원본 값 0.60676, 차이 +0.156663. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |
| SPFH590 | 240024 | 가열·균열 · CP-Z4 | 457.083–458.083 | LOT 내 CP-Z4 중앙값 0.450002 대비 원본 값 0.38364, 차이 -0.066362. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |
| SPCEN | 240038 | 가열·균열 · CP-Z4 | 122.533–123.533 | LOT 내 CP-Z4 중앙값 0.450046 대비 원본 값 0.505652, 차이 +0.055606. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |
| SGCC | 240001 | 가열·균열 · CP-Z4 | 542.767–543.767 | LOT 내 CP-Z4 중앙값 0.450053 대비 원본 값 0.432447, 차이 -0.017606. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |
| DP590 | 240058 | 가열·균열 · CP-Z4 | 216.083–216.883 | LOT 내 CP-Z4 중앙값 0.447995 대비 원본 값 0.0969111, 차이 -0.351084. 강종 안에서 이 차이가 가장 큰 관측이다. 이 LOT에서는 CP가 관측 후반에 하강하며 TC·OP도 종료 쪽에서 낮아지는 형태다. | 약 130분 이후 CP 하강과 TC·OP의 변화 순서를 확인한다. 종료 단계의 조건 전환인지 판단할 운전 상태 기록이 필요하다. |
| SPCD | 240117 | 가열·균열 · CP-Z4 | 356.333–357.333 | LOT 내 CP-Z4 중앙값 0.450003 대비 원본 값 0.367097, 차이 -0.082906. 강종 안에서 이 차이가 가장 큰 관측이다. | 최솟값/최댓값 띠와 원본 기록을 확인하고 같은 시점의 TC-Z4·OP-Z4를 비교한다. 짧은 급변과 초기조건을 구분한다. |


## 그림 3 읽는 순서

3A 왼쪽에서 센서 범위가 명시된 공정 중앙값을 비교하고, 빨간 비교 확인 구간을 바로 오른쪽의 개별 센서 확대에서 읽는다. 확대의 빨간 점은 제목에 적힌 해당 센서의 값이다. 3B에서는 개별 TC와 대응 OP/CP를 확인한다.
같은 강종이라도 관측 길이가 다르므로 **공통 경과시간 안의 비교**와 이후 단독 관측을 구분한다.
모든 7개 강종 × 5개 공정의 센서 그림은 아래 링크 표에서 열 수 있다.

[빨간 포인트의 선정 이유·확인할 센서·표시 규칙](#analysis-rules).

각 그림 아래에 선·색 띠의 의미와 비교 확인 구간의 선정 이유를 적었다. OP 패널의 회색 빗금·아래 회색 눈금은 원본 결측 위치다. OP 0% 측정값과 구분하며, 온도 패널에는 OP 결측을 칠하지 않는다.


In [ ]:
# 그림 3A: 요약 범위를 명시하고 확인할 개별 센서는 바로 옆에서 확대한다.
rt_detail_manifest=[]
rt_local_zoom_checks=[]
for grade in eligible_grades:
    info=lot_info.loc[lot_info.GRADE.eq(grade)].sort_values('LOT_START')
    lots=info.LOT_NO.astype(int).tolist()
    longest=info.DURATION_MIN.max();common=rt_common_minutes[grade]
    overview_points=rt_review_points.loc[rt_review_points.GRADE.eq(grade)&rt_review_points.SENSOR.isin(RT_TEMP_SIGNALS)].groupby('PROCESS',sort=False).head(1)
    overview_sensors=[s for group in TC_GROUPS.values() for s in group]
    note_lines,note_height=rt_signal_notes(lots,overview_sensors,overview_points,process_summary=True)
    height=15+note_height
    fig=plt.figure(figsize=(21,height))
    grid=fig.add_gridspec(6,2,width_ratios=[3.2,1.25],height_ratios=[1,1,1,1,1,.75],
        left=.065,right=.98,top=1-1.95/height,bottom=(1.05+note_height)/height,hspace=.65,wspace=.17)
    main_axes=[]
    for i,process in enumerate(TC_GROUPS):
        points=rt_review_points.loc[rt_review_points.GRADE.eq(grade)&rt_review_points.PROCESS.eq(process)&rt_review_points.SENSOR.isin(RT_TEMP_SIGNALS)].head(1)
        ax=fig.add_subplot(grid[i,0] if len(points) else grid[i,:]);main_axes.append(ax)
        for lot in lots:
            p=rt_process_profiles[lot];color=rt_lot_color[lot]
            ax.fill_between(p.index.to_numpy(float),p[(process,'min')].to_numpy(float),
                p[(process,'max')].to_numpy(float),color=color,alpha=.11,lw=0)
            ax.plot(p.index,p[(process,'median')],color=color,lw=1.5)
        ax.set_ylabel('온도 [℃]',fontsize=10)
        ax.set_title(rt_process_label(process,statistic=True),loc='left',fontsize=10.5,pad=8)
        ax.set_xlim(0,longest+max(2,longest*.015))
        if len(lots)>1: ax.axvline(common,color='#a8b7c3',ls='--',lw=.9)
        if len(points):
            point=points.iloc[0]
            # Overview marks a time window, not a particular TC value on an aggregate curve.
            ax.axvspan(point.START_MIN,point.END_MIN,color=RT_RED,alpha=.12,lw=0)
            ax.set_title(rt_process_label(process,statistic=True)+f'  |  비교 확인 구간 {point.START_MIN:g}–{point.END_MIN:g}분 → 확대',loc='left',fontsize=10,pad=8)
            zoom=fig.add_subplot(grid[i,1])
            lo=max(0,float(point.START_MIN)-5)
            hi=min(float(longest),max(lo+15,float(point.END_MIN)+5))
            for lot in lots: rt_minute_line(zoom,lot,point.SENSOR,start=lo,end=hi)
            zoom.set_xlim(lo,hi);zoom.margins(y=.25)
            zoom.set_ylabel('온도 [℃]',fontsize=9)
            zoom.set_title(f'{process} [{point.SENSOR}] · {lo:g}–{hi:g}분',loc='left',fontsize=9.5,pad=8)
            rt_focus(zoom,points,label=rt_review_caption(points))
            zoom.set_xlabel('경과시간 [분]',fontsize=9)
            zoom.xaxis.set_major_locator(plt.MaxNLocator(4))
            rt_style(zoom)
            rt_local_zoom_checks.append(dict(GRADE=grade,PROCESS=process,SENSOR=point.SENSOR,
                LOT_NO=int(point.LOT_NO),MINUTE=float(point.POINT_MIN),VALUE=float(point.VALUE),
                VIEW_START=lo,VIEW_END=hi,FIGURE='3A'))
        ax.set_xlabel('LOT 시작 후 경과시간 [분]',fontsize=9)
        rt_style(ax)
    event_ax=fig.add_subplot(grid[5,:])
    event_ticks=[];event_labels=[]
    for i,lot in enumerate(lots):
        yy=len(lots)-i-1;event_ticks.append(yy);event_labels.append(str(lot))
        duration=info.loc[info.LOT_NO.eq(lot),'DURATION_MIN'].iloc[0]
        event_ax.plot([0,duration],[yy]*2,color=rt_lot_color[lot],alpha=.25,lw=4)
        for event in rt_events_for_lot(lot):
            event_ax.plot([event['start'],event['end']],[yy]*2,marker='x',ms=5,
                color=TL_COLORS[event['kind']],lw=2.3)
    event_ax.set_yticks(event_ticks,event_labels,fontsize=8)
    event_ax.set_ylim(-.7,len(lots)-.3);event_ax.set_xlim(0,longest+max(2,longest*.015))
    event_ax.set_ylabel('LOT',fontsize=9)
    event_ax.set_title('温度 관측 길이 · ×–× CAL01 작업 겹침'.replace('温度','온도'),loc='left',fontsize=10,pad=7)
    event_ax.set_xlabel('LOT 시작 후 경과시간 [분]',fontsize=10);rt_style(event_ax)
    rt_signal_header(fig,f'그림 3A-{grade}. 공정별 온도 비교와 비교 확인 구간 확대',
        '왼쪽: 공정 센서 중앙값 · 오른쪽: 비교 확인 구간의 개별 센서',lots)
    rt_add_signal_notes(fig,note_lines,note_height,left=.065)
    rt_finish(fig,Path(GRADE_SUMMARY_DIR)/f'03A_{grade}_공정별_온도_시간흐름.png')

    # 그림 3B: 각 신호를 독립 축에서 확인한다.
    for process,sensors in PROCESS_GROUPS.items():
        if process=='가열·균열':
            sensors=[item for zone in range(3,7) for item in [f'TC-Z{zone}',f'OP-Z{zone}']]+['CP-Z4','CP-Z4M']
        sensors=[s for s in sensors if s in temp_flow]
        cols=2 if len(sensors)>1 else 1;nrows=int(np.ceil(len(sensors)/cols))
        selected_points=rt_signal_points(lots,sensors)
        note_lines,note_height=rt_signal_notes(lots,sensors,selected_points)
        height=4.6+2.3*nrows+note_height
        fig,axes=plt.subplots(nrows,cols,figsize=(19,height),sharex=True,squeeze=False)
        fig.subplots_adjust(left=.085,right=.97,top=1-1.65/height,
            bottom=(.7+note_height)/height,hspace=.48,wspace=.22)
        for ax,sensor in zip(axes.ravel(),sensors):
            for lot in lots:
                rt_minute_line(ax,lot,sensor)
                rt_mark_op_missing(ax,lot,sensor)
            ax.set_ylabel(rt_measure_label(sensor),fontsize=10)
            ax.set_title(f'{process} [{sensor}]',fontsize=10,loc='left',pad=8)
            if len(lots)>1:ax.axvline(common,color='#9aabb9',ls='--',lw=.8)
            rt_style(ax);ax.set_xlim(0,longest+2)
            points=rt_points_for(lots,sensor)
            rt_focus(ax,points,label=rt_review_caption(points))
        for ax in axes.ravel()[len(sensors):]:ax.axis('off')
        for ax in axes[-1]:
            if ax.axison:ax.set_xlabel('LOT 시작 후 경과시간 [분]')
        rt_signal_header(fig,f'그림 3B-{grade}. {rt_process_label(process)} · 센서별 비교',
            '선·색 띠·원본 OP 결측·비교 확인 구간의 설명은 그림 아래에 표시',lots)
        rt_add_signal_notes(fig,note_lines,note_height,left=.085)
        name=f"03B_{grade}_{process.replace('·','_')}_센서별.png"
        rt_finish(fig,RT_PROCESS_DIR/name,show=(grade=='SPHC' and process=='가열·균열'),dpi=150)
        rt_detail_manifest.append(dict(GRADE=grade,PROCESS=process,SIGNALS=', '.join(sensors),PNG=str((RT_PROCESS_DIR/name).as_posix())))

# 5. 같은 시간 길이에서 수준·변화량·변동폭 확인 — 그림 4

이전 열지도는 동일 강종의 다른 LOT 대비 Robust Z를 신호 전체의 Q75로 요약한 것이다.
그 요약만으로는 어떤 센서가 언제 달라졌는지, 왜 점수가 큰지 판단할 수 없다.

새 그림은 반복 LOT가 3개 이상인 SPHC·SPCE에서 다음을 나란히 보여준다.

- **수준:** 공통 시간 구간의 실제 중앙값.
- **변화량:** 같은 구간의 마지막 10분 중앙값 − 처음 10분 중앙값.
- **변동폭:** 같은 구간의 IQR(Q75−Q25). 분산과는 다르다.
- **시간 근거:** 요약값과 같은 센서의 실제 시계열, 처음/마지막 10분, 차이가 큰 5분 표시.

모든 통계는 강종별 최단 LOT 길이의 정수 분까지만 계산한다.
기존 Robust Z는 참고 표로만 보존하고, 그림에서는 실제 단위와 시간 근거를 사용한다.

In [ ]:
# 같은 강종은 같은 경과시간 길이에서만 통계를 비교한다.
ANALYSIS_SIGNALS = [c for c in RT_SIGNALS if pd.api.types.is_numeric_dtype(temp_flow[c])]
rows = []
for lot, frame in temp_flow.groupby("LOT_NO"):
    info = lot_info.loc[lot_info["LOT_NO"].eq(lot)].iloc[0]
    duration = rt_common_minutes[info.GRADE]
    frame = frame.loc[frame["ELAPSED_MIN"].ge(0) & frame["ELAPSED_MIN"].lt(duration)]
    for sensor in ANALYSIS_SIGNALS:
        valid = frame[["ELAPSED_MIN", sensor]].dropna()
        if valid.empty:
            continue
        first10 = valid.loc[valid["ELAPSED_MIN"] < min(10, duration), sensor].median()
        last10 = valid.loc[valid["ELAPSED_MIN"] >= max(duration - 10, 0), sensor].median()
        rows.append({"LOT_NO": int(lot), "GRADE": info.GRADE, "SENSOR": sensor,
                     "COMPARISON_END_MIN": duration, "N_RAW_SAMPLES": len(valid),
                     "LEVEL_MEDIAN": valid[sensor].median(), "TREND_DELTA": last10 - first10,
                     "VAR_IQR": valid[sensor].quantile(.75) - valid[sensor].quantile(.25),
                     "FIRST10": first10, "LAST10": last10})
feature = pd.DataFrame(rows)

**결과 — 동일 강종의 공통 비교 길이**

| 행 | GRADE | LOT_NO | COMPARISON_END_MIN |
| --- | --- | --- | --- |
| 0 | SGCC | 240001 | 688 |
| 22 | SPFH590 | 240024 | 465 |
| 44 | SPCE | 240037 | 107 |
| 66 | SPCEN | 240038 | 124 |
| 88 | SPHC | 240044 | 74 |
| 110 | DP590 | 240058 | 216 |
| 132 | SPHC | 240061 | 74 |
| 154 | SPHC | 240069 | 74 |
| 176 | SPHC | 240091 | 74 |
| 198 | SPFH590 | 240108 | 465 |
| 220 | SPCEN | 240114 | 124 |
| 242 | SPCD | 240117 | 444 |
| 264 | SPCE | 240121 | 107 |
| 286 | SPCE | 240133 | 107 |

## 참고 점수의 해석

동일 강종 다른 LOT만을 기준으로 계산한다. 비교 LOT가 2개 미만이면 계산하지 않는다.
작은 비교군이나 매우 작은 기준 척도는 큰 점수를 만들 수 있으므로,
점수는 고장 확률·규격 한계·효과 크기가 아니다. 그림 4의 실제 값과 시계열을 함께 확인한다.

In [ ]:
for metric in ["LEVEL_MEDIAN", "TREND_DELTA", "VAR_IQR"]:
    z_col = metric + "_RZ"
    feature[z_col] = np.nan

    for (grade, sensor), idx in feature.groupby(["GRADE", "SENSOR"]).groups.items():
        idx = list(idx)
        vals = feature.loc[idx, metric]

        for i in idx:
            peers = vals.drop(index=i)
            feature.loc[i, z_col] = robust_z(feature.loc[i, metric], peers.values)

feature["MAX_ABS_RZ"] = (
    feature[
        ["LEVEL_MEDIAN_RZ", "TREND_DELTA_RZ", "VAR_IQR_RZ"]
    ].abs().max(axis=1)
)

**결과 — 센서별 수준·변화량·변동폭과 참고 점수**

| 행 | LOT_NO | GRADE | SENSOR | COMPARISON_END_MIN | N_RAW_SAMPLES | LEVEL_MEDIAN | TREND_DELTA | VAR_IQR | FIRST10 | LAST10 | LEVEL_MEDIAN_RZ | TREND_DELTA_RZ | VAR_IQR_RZ | MAX_ABS_RZ |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 139 | 240061 | SPHC | TC-Z5 | 74 | 4369 | 841.894000 | 30.580000 | 15.841000 | 823.016000 | 853.596000 | -374.806084 | 99.441989 | 1504.596173 | 1504.596173 |
| 140 | 240061 | SPHC | TC-Z6 | 74 | 4369 | 842.054000 | 30.519000 | 15.893000 | 822.723000 | 853.242000 | -595.878861 | 164.467827 | 1173.538978 | 1173.538978 |
| 61 | 240037 | SPCE | TC-Z9 | 107 | 6208 | 335.481000 | -9.695000 | 8.602000 | 336.248000 | 326.553000 | 643.464185 | -1.388624 | 0.514112 | 643.464185 |
| 134 | 240061 | SPHC | TC-Z2 | 74 | 4369 | 98.549000 | 2.372900 | 1.082700 | 96.960400 | 99.333300 | -513.624713 | 5.710034 | 16.059773 | 513.624713 |
| 147 | 240061 | SPHC | TC-Z7 | 74 | 4369 | 323.380000 | 11.127000 | 5.870000 | 316.456000 | 327.583000 | -93.886181 | 219.447552 | 485.633347 | 485.633347 |
| 138 | 240061 | SPHC | TC-Z4 | 74 | 4369 | 841.886000 | 30.495000 | 16.131000 | 822.831000 | 853.326000 | -480.048563 | 40.235918 | 179.563159 | 480.048563 |
| 148 | 240061 | SPHC | TC-Z8 | 74 | 4369 | 323.272000 | 11.047000 | 5.897000 | 316.414000 | 327.461000 | -74.707881 | 169.542450 | 327.352849 | 327.352849 |
| 109 | 240044 | SPHC | TC-OUT2 | 74 | 4294 | 282.505000 | -12.071000 | 12.051000 | 289.325000 | 277.254000 | -1.443334 | -192.050920 | 5.944344 | 192.050920 |
| 65 | 240037 | SPCE | TC-OUT2 | 107 | 6206 | 282.102000 | -6.794500 | 11.844000 | 287.674000 | 280.879500 | 2.113581 | -189.082243 | -0.078228 | 189.082243 |
| 133 | 240061 | SPHC | TC-Z1 | 74 | 4369 | 98.361700 | 3.209600 | 1.385000 | 96.006400 | 99.216000 | -150.589430 | 11.158225 | 9.512678 | 150.589430 |
| 137 | 240061 | SPHC | TC-Z3 | 74 | 4369 | 842.097000 | 32.824000 | 16.181000 | 820.039000 | 852.863000 | -111.357817 | 30.278625 | 32.026918 | 111.357817 |
| 268 | 240121 | SPCE | OP-Z2 | 107 | 6298 | 24.045000 | -2.299050 | 2.158475 | 27.196500 | 24.897450 | 3.238460 | -1.366840 | -77.632485 | 77.632485 |
| 191 | 240091 | SPHC | TC-Z7 | 74 | 4332 | 332.951000 | 0.065000 | 0.110000 | 332.943000 | 333.008000 | 47.126376 | -0.407832 | -0.583034 | 47.126376 |
| 276 | 240121 | SPCE | OP-Z6 | 107 | 6298 | 73.334400 | 1.106450 | 1.984725 | 72.903950 | 74.010400 | 0.064510 | 0.344139 | -46.421531 | 46.421531 |
| 152 | 240061 | SPHC | TC-OUT1 | 74 | 4369 | 282.681000 | 25.405000 | 17.796000 | 265.181000 | 290.586000 | 0.590944 | 6.329900 | 39.604714 | 39.604714 |
| 279 | 240121 | SPCE | TC-Z7 | 107 | 6297 | 333.298000 | 0.035000 | 0.120000 | 333.314000 | 333.349000 | 37.422193 | 0.812387 | -1.069882 | 37.422193 |
| 167 | 240069 | SPHC | CP-Z4 | 74 | 4261 | 0.449834 | 0.000484 | 0.003294 | 0.449562 | 0.450046 | -35.410765 | 1.338605 | 2.840188 | 35.410765 |
| 192 | 240091 | SPHC | TC-Z8 | 74 | 4329 | 333.373000 | -0.013000 | 0.061000 | 333.376000 | 333.363000 | 34.109961 | -0.285361 | -0.909096 | 34.109961 |
| 145 | 240061 | SPHC | CP-Z4 | 74 | 4369 | 0.449893 | -0.000195 | 0.001976 | 0.449719 | 0.449524 | -25.462026 | -1.841879 | -7.873383 | 25.462026 |
| 157 | 240069 | SPHC | OP-Z1 | 74 | 4261 | 72.816100 | -0.747500 | 2.179900 | 72.991500 | 72.244000 | -1.916327 | 0.178805 | -22.439156 | 22.439156 |
| 175 | 240069 | SPHC | TC-OUT2 | 74 | 4261 | 283.637000 | 3.456000 | 11.215000 | 282.335000 | 285.791000 | 0.459677 | 22.354551 | 0.076533 | 22.354551 |
| 154 | 240069 | SPHC | CLN-IN | 74 | 4261 | 62.805000 | 2.359100 | 1.563400 | 60.998700 | 63.357800 | -1.980839 | 21.344275 | 4.504325 | 21.344275 |
| 306 | 240133 | SPCE | TC-OUT1 | 107 | 6193 | 287.755000 | 11.438500 | 18.351000 | 286.286500 | 297.725000 | 4.915475 | 19.283582 | 2.534374 | 19.283582 |
| 99 | 240044 | SPHC | OP-Z5 | 74 | 4295 | 54.164200 | 1.205100 | 2.168100 | 54.703500 | 55.908600 | 2.899573 | 0.348330 | 16.070134 | 16.070134 |
| 287 | 240133 | SPCE | TC-Z1 | 107 | 6193 | 99.940900 | -1.544200 | 0.316800 | 100.040500 | 98.496300 | -6.482320 | -15.757219 | 0.382163 | 15.757219 |
| 62 | 240037 | SPCE | OP-Z7 | 107 | 6204 | 65.974400 | 3.869100 | 3.549500 | 63.202800 | 67.071900 | -1.171549 | 15.542550 | 6.177581 | 15.542550 |
| 95 | 240044 | SPHC | TC-Z5 | 74 | 4295 | 859.708000 | -0.086000 | 0.383000 | 859.417000 | 859.331000 | 0.937707 | -0.421252 | 15.127864 | 15.127864 |
| 178 | 240091 | SPHC | TC-Z2 | 74 | 4332 | 100.114000 | 0.066000 | 0.232950 | 100.050000 | 100.116000 | 14.838797 | 0.286800 | -0.911184 | 14.838797 |
| 59 | 240037 | SPCE | TC-Z7 | 107 | 6204 | 330.247000 | -0.213000 | 0.195000 | 330.400000 | 330.187000 | -0.626724 | -13.871049 | 5.276898 | 13.871049 |
| 286 | 240133 | SPCE | CLN-IN | 107 | 6193 | 69.281400 | 0.125200 | 1.881400 | 68.156900 | 68.282100 | 11.965165 | -0.430772 | 13.194118 | 13.194118 |

In [ ]:
# LOT별 강건 점수 요약
lot_score = (
    feature.groupby(["LOT_NO", "GRADE"])
           .agg(
               LEVEL_Q75=("LEVEL_MEDIAN_RZ", lambda s: s.abs().quantile(0.75)),
               TREND_Q75=("TREND_DELTA_RZ", lambda s: s.abs().quantile(0.75)),
               VAR_Q75=("VAR_IQR_RZ", lambda s: s.abs().quantile(0.75)),
               MAX_SENSOR_RZ=("MAX_ABS_RZ", "max")
           )
           .reset_index()
)

**결과 — LOT별 참고 점수**

| 행 | LOT_NO | GRADE | LEVEL_Q75 | TREND_Q75 | VAR_Q75 | MAX_SENSOR_RZ |
| --- | --- | --- | --- | --- | --- | --- |
| 6 | 240061 | SPHC | 111.357817 | 30.278625 | 39.604714 | 1504.596173 |
| 2 | 240037 | SPCE | 1.852004 | 2.985614 | 3.010532 | 643.464185 |
| 4 | 240044 | SPHC | 1.640903 | 2.680222 | 3.282569 | 192.050920 |
| 12 | 240121 | SPCE | 2.666482 | 1.461879 | 2.377438 | 77.632485 |
| 8 | 240091 | SPHC | 2.394225 | 0.929158 | 3.944232 | 47.126376 |
| 7 | 240069 | SPHC | 1.964711 | 1.329651 | 2.590493 | 35.410765 |
| 13 | 240133 | SPCE | 3.216802 | 7.801311 | 2.207767 | 19.283582 |
| 0 | 240001 | SGCC | NaN | NaN | NaN | NaN |
| 1 | 240024 | SPFH590 | NaN | NaN | NaN | NaN |
| 3 | 240038 | SPCEN | NaN | NaN | NaN | NaN |
| 5 | 240058 | DP590 | NaN | NaN | NaN | NaN |
| 9 | 240108 | SPFH590 | NaN | NaN | NaN | NaN |
| 10 | 240114 | SPCEN | NaN | NaN | NaN | NaN |
| 11 | 240117 | SPCD | NaN | NaN | NaN | NaN |

In [ ]:
# 그림 4: 단위 없는 점수 지도 대신 실제 값과 그 값이 나온 시간 구간을 나란히 본다.
rt_highlights = []
metrics = [("LEVEL_MEDIAN", "수준 · 관측 중앙값", "℃"),
           ("TREND_DELTA", "변화량 · 마지막 10분 - 처음 10분", "℃"),
           ("VAR_IQR", "변동폭 · IQR(Q75-Q25)", "℃")]
evidence_grades = lot_info["GRADE"].value_counts().loc[lambda s: s >= 3].index.tolist()
for grade in evidence_grades:
    lots = lot_info.loc[lot_info["GRADE"].eq(grade), "LOT_NO"].astype(int).tolist()
    common = rt_common_minutes[grade]
    fig, axes = plt.subplots(3, 2, figsize=(21, 13), gridspec_kw={"width_ratios": [1.3, 3]})
    fig.subplots_adjust(left=.09, right=.97, top=.83, bottom=.08, hspace=.42, wspace=.22)
    text_boxes = []
    for row_index, (metric, label, unit) in enumerate(metrics):
        values = feature.loc[feature["GRADE"].eq(grade) & feature["SENSOR"].isin(RT_TEMP_SIGNALS)]
        spreads = values.groupby("SENSOR")[metric].agg(lambda s: s.max() - s.min())
        sensor = spreads.idxmax()
        selected = values.loc[values["SENSOR"].eq(sensor)].set_index("LOT_NO").loc[lots]
        deviations = (selected[metric] - selected[metric].median()).abs()
        candidate = int(deviations.idxmax())
        left, right = axes[row_index]
        for i, lot in enumerate(lots):
            value = selected.loc[lot, metric]
            left.scatter(value, len(lots) - i - 1, color=rt_lot_color[lot], edgecolors=RT_RED if lot == candidate else "white", linewidths=1.7 if lot == candidate else .5, s=70 if lot == candidate else 35)
            text_boxes.append(left.annotate(f"{value:.3g}", (value, len(lots) - i - 1),
                                             xytext=(8, 0), textcoords="offset points", va="center", fontsize=9))
            rt_minute_line(right, lot, sensor, end=common - 1,
                           lw=2.1 if lot == candidate else 1.2)
        left.set_yticks(range(len(lots)), [str(lot) for lot in lots[::-1]], fontsize=9)
        left.set_ylim(-.7, len(lots) - .3)
        minimum, maximum = selected[metric].min(), selected[metric].max()
        pad = max((maximum - minimum) * .23, abs(maximum) * .00015, .03)
        left.set_xlim(minimum - pad, maximum + pad * 2.3)
        left.set_xlabel(f"{sensor} [{unit}]", fontsize=9)
        left.set_title(label, loc="left", fontsize=10, pad=12)
        right.set_title(f"{next(p for p, ss in TC_GROUPS.items() if sensor in ss)} [{sensor}] · 시간 흐름", loc="left", fontsize=10, pad=12)
        right.set_xlim(0, common)
        right.set_ylabel("온도 [℃]", fontsize=9)
        if metric == "TREND_DELTA":
            right.axvspan(0, min(10, common), color="#dcebf4", alpha=.6)
            right.axvspan(max(common - 10, 0), common, color="#dcebf4", alpha=.6)
        peer_values = pd.concat([rt_profiles[lot][(sensor, "median")].rename(lot)
                                 for lot in lots if lot != candidate], axis=1).reindex(range(common))
        reference = peer_values.median(axis=1).where(peer_values.count(axis=1) >= 2)
        candidate_values = rt_profiles[candidate][(sensor, "median")].reindex(range(common))
        delta = (candidate_values - reference).abs().rolling(5, min_periods=5).mean()
        if delta.notna().any():
            end = int(delta.idxmax()) + 1
            start = max(0, end - 5)
            right.axvspan(start, end, color=RT_RED, alpha=.09, zorder=0)
            rt_highlights.append({"GRADE": grade, "METRIC": metric, "SENSOR": sensor,
                                   "LOT_NO": candidate, "START_MIN": start, "END_MIN": end,
                                   "COMMON_MIN": common,
                                   "DESCRIPTION": "다른 LOT 중앙값 대비 차이가 가장 큰 5분(탐색용)"})
            chosen = (candidate_values - reference).abs().loc[start:end-1].idxmax()
            value = candidate_values.loc[chosen]
            right.scatter(chosen,value,s=43,facecolors="white",edgecolors=RT_RED,lw=1.7,zorder=7)
            low,high=right.get_ylim();right.set_ylim(low,high+(high-low)*.22)
            rt_point_box(right,(float(chosen),float(value)),
                f"LOT {candidate} · {start}–{end}분\n{sensor} · 같은 경과 분 값 차이")
        for ax in [left, right]:
            rt_style(ax)
        if row_index == 2:
            right.set_xlabel("LOT 시작 후 실제 경과시간 [분]")
    rt_header(fig, f"그림 4-{grade}. 요약값과 해당 센서의 시간 흐름",
              f"공통 비교: 처음 {common}분 · 빨간 점·구간: LOT 간 차이 확인")
    rt_lot_legend(fig, lots, y=.95)
    fig._collision_groups = [text_boxes]
    filename = f"04_{grade}_실제값_시간근거.png"
    rt_finish(fig, Path(STATS_DIR) / filename)
rt_highlight_table = pd.DataFrame(rt_highlights)

### 표시된 차이가 의미하는 범위

빨간 구간은 다른 LOT 중앙값과 평균 차이가 큰 5분이다. 행마다 LOT 간 요약값 범위가 가장 큰 TC를 골랐다.
왼쪽 빨간 테두리는 비교 대상 LOT, 오른쪽 빨간 점은 해당 구간의 관측 위치다.
변화량 행의 파란 구간은 처음/마지막 10분이며 IQR은 Q75−Q25이다.
공통 길이의 요약 차이는 주기 위상·초기조건의 영향도 받는다. 고장·이상을 확정하지 않고 전체 관측과 함께 확인한다.


# 6. 같은 강종 안에서 상세 후보 LOT 선정

기본 후보 선정은:

1. 반복 LOT이 3개 이상인 강종
2. Level / Trend / Variance 중 지속적인 차이
3. 데이터 품질 문제가 주원인이 아닌 LOT

순서로 한다.

In [ ]:
grade_counts = lot_info["GRADE"].value_counts()
eligible_grade_set = set(grade_counts[grade_counts >= 3].index)

candidate_table = (
    lot_score[
        lot_score["GRADE"].isin(eligible_grade_set)
    ]
    .sort_values("MAX_SENSOR_RZ", ascending=False)
    .reset_index(drop=True)
)

**결과 — 상세 확인 후보 LOT**

| 행 | LOT_NO | GRADE | LEVEL_Q75 | TREND_Q75 | VAR_Q75 | MAX_SENSOR_RZ |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | 240061 | SPHC | 111.357817 | 30.278625 | 39.604714 | 1504.596173 |
| 1 | 240037 | SPCE | 1.852004 | 2.985614 | 3.010532 | 643.464185 |
| 2 | 240044 | SPHC | 1.640903 | 2.680222 | 3.282569 | 192.050920 |
| 3 | 240121 | SPCE | 2.666482 | 1.461879 | 2.377438 | 77.632485 |
| 4 | 240091 | SPHC | 2.394225 | 0.929158 | 3.944232 | 47.126376 |
| 5 | 240069 | SPHC | 1.964711 | 1.329651 | 2.590493 | 35.410765 |
| 6 | 240133 | SPCE | 3.216802 | 7.801311 | 2.207767 | 19.283582 |

## 자동 후보는 참고용

이후 상세분석에서는 자동 1등 LOT만 보지 말고:

- 정기수리 직전/직후 LOT
- 정비와 직접 겹친 LOT
- 부품교체와 근접한 LOT

도 함께 Focus LOT으로 포함한다.

# 7. 개별 센서 그림 찾기

기존 그림 5의 후보 LOT 초기 120분 그림은 그림 3과 중복되어 제거했다.
같은 강종·공정의 TC·OP·CP 전체 시간 비교는 아래 그림 3B 링크에서 확인한다.
새 그림 5는 온도·용접·설정·정비를 같은 실제 날짜로 묶는 데 사용한다.

In [ ]:
# 사용자가 직접 바꿀 수 있는 상세 후보
DEFAULT_DETAIL_LOT = (
    int(candidate_table.iloc[0]["LOT_NO"])
    if len(candidate_table) > 0
    else int(lot_info.iloc[0]["LOT_NO"])
)

DETAIL_LOT = DEFAULT_DETAIL_LOT
DETAIL_MIN = 120

detail_grade = lot_info.loc[
    lot_info["LOT_NO"] == DETAIL_LOT, "GRADE"
].iloc[0]

peer_lots = (
    lot_info.loc[
        (lot_info["GRADE"] == detail_grade)
        & (lot_info["LOT_NO"] != DETAIL_LOT),
        "LOT_NO"
    ]
    .astype(int)
    .tolist()
)

print("상세 후보:", DETAIL_LOT)
print("강종:", detail_grade)
print("동일 강종 비교 LOT:", peer_lots)

In [ ]:
# 강종·공정별 그림 목차는 아래 마크다운 셀에서 확인한다.

**결과 — 강종·공정별 센서 그림 목차**

| 강종 | 세정 | 건조 | 가열·균열 | 과시효 | 냉각 |
| --- | --- | --- | --- | --- | --- |
| SPHC | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_냉각_센서별.png) |
| SPCE | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_냉각_센서별.png) |
| SPFH590 | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_냉각_센서별.png) |
| SPCEN | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_냉각_센서별.png) |
| SGCC | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_냉각_센서별.png) |
| DP590 | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_냉각_센서별.png) |
| SPCD | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_세정_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_건조_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_가열_균열_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_과시효_센서별.png) | [센서별 실제 시간](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_냉각_센서별.png) |

## LOT별 공정 상세와 단서 탐색 기록

14 LOT × 5공정의 개별 센서 그림, SPCE 원본 초 단위 확대, 후보별 근거·비교 대상·해석 한계를 함께 남긴다. 그림을 생성한 것과 수동 판독을 마친 것을 구분한다.

In [ ]:
# %% LOT별 공정 상세: 각 LOT의 모든 센서와 탐색 구간을 독립적으로 확인한다.
IQ_LOT_DIR=Path(DETAIL_DIR)/"01_공정별"
IQ_ZOOM_DIR=Path(DETAIL_DIR)/"02_원본확대"
for folder in [IQ_LOT_DIR,IQ_ZOOM_DIR]: folder.mkdir(parents=True,exist_ok=True)
iq_lot_manifest=[]
iq_case_rows=[
    {"LOT":240037,"강종":"SPCE","공정":"과시효","센서":"TC-Z9","구간":"29–34분","단서":"처음 107분의 TC-Z9 중앙값 335.481, 다른 SPCE는 324.976/324.998","다음 확인":"TC-Z7·Z8과 동시 변화인지, 같은 강종의 같은 경과시간에서 비교","해석 한계":"관측 LOT와 SPCE BM 두 건은 겹치지 않음"},
    {"LOT":240133,"강종":"SPCE","공정":"냉각","센서":"TC-OUT1","구간":"77–82분","단서":"처음 107분의 종료-초기 10분 차이 +11.4385, 다른 SPCE는 음수","다음 확인":"TC-OUT2와 냉각 주기의 위상·주기·진폭을 비교","해석 한계":"CBM 감지는 57분, 작업은 관측 종료 뒤. 요약 차이는 주기 위상의 영향 가능"},
    {"LOT":240121,"강종":"SPCE","공정":"냉각","센서":"TC-OUT2","구간":"0–5분","단서":"처음 107분 TC-OUT2 IQR 14.543, SPCE 3 LOT 중 가장 큼","다음 확인":"초기 과도응답과 주기 진폭을 원본 초 단위에서 구분","해석 한계":"용접 28 COIL과 시간 겹침. 직접 연결 키 없음"},
    {"LOT":240061,"강종":"SPHC","공정":"가열·균열","센서":"TC-Z3","구간":"0–5분부터 승온 구간","단서":"6월 수리 후 첫 온도 LOT이며 초기 승온과 수준 차이의 탐색 후보","다음 확인":"다른 SPHC 240044·240069·240091의 같은 경과 분 및 TC·OP·CP 비교","해석 한계":"수리 직전 LOT 240058은 DP590. 직접 전후 비교 불가"},
    {"LOT":240117,"강종":"SPCD","공정":"가열·균열/냉각","센서":"TC-Z3~Z6·TC-OUT1/2","구간":"07:40–12:44 작업 전·중·후","단서":"CBM/TBM 작업과 용접 31 COIL이 온도 관측 창에 겹침","다음 확인":"작업 전·중·후 온도/OP/CP와 PWR·SET-PWR·SPD를 실제 시각으로 함께 확인","해석 한계":"SPCD 온도 LOT는 1개. 동일 강종 반복 비교 불가"},
]
iq_cases=pd.DataFrame(iq_case_rows)
for lot in tl_lots.itertuples():
    lot_id=int(lot.LOT_NO)
    profile=rt_profiles[lot_id]
    for process,original_sensors in PROCESS_GROUPS.items():
        sensors=original_sensors.copy()
        if process=="가열·균열": sensors += ["CP-Z4M"]
        sensors=[s for s in sensors if (s,"median") in profile]
        cols=2 if len(sensors)>1 else 1
        nr=int(np.ceil(len(sensors)/cols))
        selected_points=rt_signal_points([lot_id],sensors)
        note_lines,note_height=rt_signal_notes([lot_id],sensors,selected_points)
        height=2.7+2.3*nr+note_height
        fig,axes=plt.subplots(nr,cols,figsize=(20,height),sharex=True,squeeze=False)
        fig.subplots_adjust(left=.075,right=.98,top=1-1.65/height,bottom=(.75+note_height)/height,hspace=.4,wspace=.18)
        for ax,sensor in zip(axes.ravel(),sensors):
            x=profile.index.to_numpy(float)
            ax.fill_between(x,profile[(sensor,"min")].to_numpy(float),profile[(sensor,"max")].to_numpy(float),color=rt_lot_color[lot_id],alpha=.15,lw=0)
            ax.plot(x,profile[(sensor,"median")],color=rt_lot_color[lot_id],lw=1.4)
            rt_mark_op_missing(ax,lot_id,sensor)
            ax.set_ylabel(rt_measure_label(sensor), fontsize=10)
            ax.set_title(f'{process} [{sensor}]',loc="left",fontsize=10,pad=8)
            ax.set_xlim(0,lot.DURATION_MIN)
            points=rt_points_for([lot_id],sensor)
            rt_focus(ax,points,label=rt_review_caption(points))
            rt_style(ax)
        for ax in axes.ravel()[len(sensors):]: ax.axis("off")
        for ax in axes[-1]:
            if ax.axison: ax.set_xlabel("LOT 시작 후 실제 경과시간 [분]")
        fig.suptitle(f"LOT {lot_id} · {lot.GRADE} · {rt_process_label(process)}",x=.02,y=1-.18/height,ha="left",fontsize=17,fontweight="bold",color="#17344d")
        fig.text(.02,1-.72/height,f"{lot.LOT_START:%Y/%m/%d %H:%M:%S}–{lot.LOT_END:%m/%d %H:%M:%S} · {lot.DURATION_MIN:.1f}분",fontsize=10.5,color="#435b6c")
        rt_lot_legend(fig,[lot_id],y=1-.65/height)
        rt_add_signal_notes(fig,note_lines,note_height)
        name=f"04A_LOT_{lot_id}_{process.replace('·','_')}.png"
        target=IQ_LOT_DIR/name
        rt_finish(fig,target,show=(lot_id==240037 and process=="과시효"),dpi=140)
        iq_lot_manifest.append({"LOT":lot_id,"강종":lot.GRADE,"공정":process,"PNG":str(target.resolve())})
    print(f"LOT {lot_id}: 5개 공정 / {len(RT_SIGNALS)}개 신호 상세 저장")

In [ ]:
# %% 단서 확대: SPCE 3개 후보의 원본 초 단위와 동일 경과시간의 다른 LOT.
iq_zoom_manifest=[]
for lot_id,sensor,other_sensors,a0,a1 in [
    (240037,"TC-Z9",["TC-Z7","TC-Z8"],29,34),
    (240133,"TC-OUT1",["TC-OUT2"],77,82),
    (240121,"TC-OUT2",["TC-OUT1"],0,5)]:
    info=tl_lots.loc[tl_lots.LOT_NO.eq(lot_id)].iloc[0]
    peers=tl_lots.loc[tl_lots.GRADE.eq(info.GRADE)]
    start=max(0,a0-10);end=min(rt_common_minutes[info.GRADE],a1+10)
    fig,axes=plt.subplots(2,1,figsize=(20,8.5),sharex=True)
    fig.subplots_adjust(left=.08,right=.98,top=.72,bottom=.12,hspace=.4)
    for p in peers.itertuples():
        raw=temp_flow.loc[temp_flow.LOT_NO.eq(p.LOT_NO)&temp_flow.ELAPSED_MIN.between(start,end)]
        axes[0].plot(raw.ELAPSED_MIN,raw[sensor],lw=.8,alpha=.8,color=rt_lot_color[int(p.LOT_NO)],label=str(int(p.LOT_NO)))
    own=temp_flow.loc[temp_flow.LOT_NO.eq(lot_id)&temp_flow.ELAPSED_MIN.between(start,end)]
    for i,other in enumerate(other_sensors):
        axes[1].plot(own.ELAPSED_MIN,own[other],lw=.9,color=RT_COLORS[i],label=other)
    axes[0].set_title(f"{sensor} · SPCE 3 LOT의 같은 경과시간",loc="left",fontsize=11)
    axes[1].set_title(f"LOT {lot_id} · 함께 확인할 신호",loc="left",fontsize=11)
    for ax in axes:
        ax.axvspan(a0,a1,color=RT_RED,alpha=.09,zorder=0)
        ax.set_ylabel("온도 [℃]")
        rt_style(ax)
    handles0,labels0=axes[0].get_legend_handles_labels()
    handles1,labels1=axes[1].get_legend_handles_labels()
    fig.legend(handles0+handles1,["LOT "+x for x in labels0]+labels1,loc="upper right",bbox_to_anchor=(.98,.86),ncol=5,frameon=False,fontsize=9)
    axes[1].set_xlim(start,end)
    axes[1].set_xlabel("LOT 시작 후 경과시간 [분]")
    points=rt_points_for([lot_id],sensor)
    rt_focus(axes[0],points)
    fig.suptitle(f"단서 확대. SPCE · LOT {lot_id} · {sensor} · {a0}–{a1}분",x=.02,y=.98,ha="left",fontsize=17,fontweight="bold",color="#17344d")
    fig.text(.02,.915,"빨간 점·구간: 같은 경과 분에서 확인할 차이",fontsize=10.5,color="#435b6c")
    target=IQ_ZOOM_DIR/f"04B_LOT_{lot_id}_{sensor}_원본초.png"
    rt_finish(fig,target,show=True,dpi=150)
    iq_zoom_manifest.append({"LOT":lot_id,"SENSOR":sensor,"START_MIN":a0,"END_MIN":a1,"PNG":str(target.resolve())})

<a id="analysis-journey"></a>

### LOT 탐색 기록과 공정별 그림 목차

#### 먼저 볼 LOT와 확인한 단서

자동 점수는 후보를 고르는 보조 자료다. 아래 값은 같은 강종의 공통 107분을 비교한 실제 값이다.
온도 차이·단일 신호의 분포 차이는 고장이나 정비의 결과를 뜻하지 않는다.

| LOT | 강종 | 공정 | 센서 | 구간 | 단서 | 다음 확인 | 해석 한계 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 240037 | SPCE | 과시효 | TC-Z9 | 29–34분 | 처음 107분의 TC-Z9 중앙값 335.481, 다른 SPCE는 324.976/324.998 | TC-Z7·Z8과 동시 변화인지, 같은 강종의 같은 경과시간에서 비교 | 관측 LOT와 SPCE BM 두 건은 겹치지 않음 |
| 240133 | SPCE | 냉각 | TC-OUT1 | 77–82분 | 처음 107분의 종료-초기 10분 차이 +11.4385, 다른 SPCE는 음수 | TC-OUT2와 냉각 주기의 위상·주기·진폭을 비교 | CBM 감지는 57분, 작업은 관측 종료 뒤. 요약 차이는 주기 위상의 영향 가능 |
| 240121 | SPCE | 냉각 | TC-OUT2 | 0–5분 | 처음 107분 TC-OUT2 IQR 14.543, SPCE 3 LOT 중 가장 큼 | 초기 과도응답과 주기 진폭을 원본 초 단위에서 구분 | 용접 28 COIL과 시간 겹침. 직접 연결 키 없음 |
| 240061 | SPHC | 가열·균열 | TC-Z3 | 0–5분부터 승온 구간 | 6월 수리 후 첫 온도 LOT이며 초기 승온과 수준 차이의 탐색 후보 | 다른 SPHC 240044·240069·240091의 같은 경과 분 및 TC·OP·CP 비교 | 수리 직전 LOT 240058은 DP590. 직접 전후 비교 불가 |
| 240117 | SPCD | 가열·균열/냉각 | TC-Z3~Z6·TC-OUT1/2 | 07:40–12:44 작업 전·중·후 | CBM/TBM 작업과 용접 31 COIL이 온도 관측 창에 겹침 | 작업 전·중·후 온도/OP/CP와 PWR·SET-PWR·SPD를 실제 시각으로 함께 확인 | SPCD 온도 LOT는 1개. 동일 강종 반복 비교 불가 |

#### SPCE 단서의 숫자 근거

| LOT_NO | SENSOR | COMPARISON_END_MIN | LEVEL_MEDIAN | TREND_DELTA | VAR_IQR |
| --- | --- | --- | --- | --- | --- |
| 240037 | TC-Z9 | 107 | 335.481 | -9.695 | 8.602 |
| 240037 | TC-OUT1 | 107 | 282.944 | -10.8265 | 17.7135 |
| 240037 | TC-OUT2 | 107 | 282.102 | -6.7945 | 11.844 |
| 240121 | TC-Z9 | 107 | 324.976 | -2.264 | 6.097 |
| 240121 | TC-OUT1 | 107 | 284.105 | -12.4405 | 17.9815 |
| 240121 | TC-OUT2 | 107 | 280.473 | 2.4235 | 14.543 |
| 240133 | TC-Z9 | 107 | 324.998 | 11.773 | 8.94 |
| 240133 | TC-OUT1 | 107 | 287.755 | 11.4385 | 18.351 |
| 240133 | TC-OUT2 | 107 | 278.946 | 2.4895 | 9.706 |

수준=중앙값, 변화량=마지막 10분 중앙값−처음 10분 중앙값, 변동폭=IQR. 모두 처음 107분 기준. 주기 신호의 위상과 초기 과도응답이 요약값을 바꿀 수 있다.

#### 모든 LOT의 개별 공정 그림

| LOT | 강종 | 시작 | 길이(분) | 세정 | 건조 | 가열·균열 | 과시효 | 냉각 | 검토 경로 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 240001 | SGCC | 01/02 03:36 | 688.5 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240001_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240001_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240001_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240001_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240001_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240024 | SPFH590 | 03/07 21:22 | 547.8 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240024_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240024_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240024_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240024_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240024_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240037 | SPCE | 04/13 23:16 | 611.3 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240037_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240037_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240037_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240037_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240037_냉각.png) | 단서 상세 검토 |
| 240038 | SPCEN | 04/16 11:35 | 124.9 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240038_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240038_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240038_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240038_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240038_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240044 | SPHC | 05/03 18:50 | 74.6 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240044_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240044_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240044_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240044_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240044_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240058 | DP590 | 06/09 05:26 | 216.9 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240058_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240058_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240058_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240058_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240058_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240061 | SPHC | 06/15 01:04 | 225.4 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240061_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240061_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240061_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240061_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240061_냉각.png) | 단서 상세 검토 |
| 240069 | SPHC | 07/07 16:57 | 812.6 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240069_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240069_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240069_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240069_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240069_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240091 | SPHC | 09/08 22:03 | 641.2 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240091_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240091_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240091_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240091_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240091_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240108 | SPFH590 | 10/18 04:05 | 465.9 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240108_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240108_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240108_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240108_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240108_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240114 | SPCEN | 11/04 15:40 | 319.0 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240114_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240114_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240114_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240114_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240114_냉각.png) | 개별 그림 생성 · 추가 판독 대상 |
| 240117 | SPCD | 11/13 07:08 | 444.3 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240117_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240117_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240117_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240117_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240117_냉각.png) | 단서 상세 검토 |
| 240121 | SPCE | 11/24 07:31 | 651.2 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240121_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240121_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240121_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240121_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240121_냉각.png) | 단서 상세 검토 |
| 240133 | SPCE | 12/28 14:57 | 107.4 | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240133_세정.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240133_건조.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240133_가열_균열.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240133_과시효.png) | [개별 신호](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/01_공정별/04A_LOT_240133_냉각.png) | 단서 상세 검토 |

#### 강종별 같은 공정 비교

| 강종 | 온도 LOT 수 | 공통 비교(분) | 세정 | 건조 | 가열·균열 | 과시효 | 냉각 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| DP590 | 1 | 216 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_DP590_냉각_센서별.png) |
| SGCC | 1 | 688 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SGCC_냉각_센서별.png) |
| SPCD | 1 | 444 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCD_냉각_센서별.png) |
| SPCE | 3 | 107 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCE_냉각_센서별.png) |
| SPCEN | 2 | 124 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPCEN_냉각_센서별.png) |
| SPFH590 | 2 | 465 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPFH590_냉각_센서별.png) |
| SPHC | 4 | 74 | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_세정_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_건조_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_가열_균열_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_과시효_센서별.png) | [같은 강종 비교](../figures/05_all_data_timeseries_jinny_01/03_강종별비교/02_센서별/03B_SPHC_냉각_센서별.png) |

#### SPCE 원본 초 단위 확대

- [LOT 240037 · TC-Z9 · 29–34분](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04B_LOT_240037_TC-Z9_원본초.png)
- [LOT 240133 · TC-OUT1 · 77–82분](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04B_LOT_240133_TC-OUT1_원본초.png)
- [LOT 240121 · TC-OUT2 · 0–5분](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04B_LOT_240121_TC-OUT2_원본초.png)


#### 다시 그림을 수정할 기준

단서마다 `LOT → 공정 → 센서 → 실제 시간 구간 → 같은 강종 비교 대상 → 함께 볼 정비/용접 → 확인한 사실 → 아직 모르는 원인`을 남긴다.
변화가 보이면 해당 센서의 원본 초 단위 창을 확장하고, TC와 관련 OP/CP를 같은 시간에 비교한다.
주기 신호는 짧은 창의 중앙값·기울기만으로 판정하지 않고 주기·진폭·위상을 먼저 확인한다.
정비 전후 비교는 같은 강종·설정과 실제 관측이 양쪽에 있는 경우에 진행한다. 현재 자료에 없는 구간은 추가 데이터 확인 항목으로 남긴다.

개별 그림 70개는 14 LOT × 5공정의 신호를 모두 제공한다. 위 ‘단서 상세 검토’는 원본 비교까지 수행한 경로이며,
나머지 LOT도 그림을 만들었으나 수동 판독 완료로 표기하지 않는다. 표시된 단서는 고장 판정이 아니다.


#### 원본을 더 보고 수정한 해석

[SPCE TC-Z9의 전체 주기·위상과 SPHC 240061 TC·OP·CP 동시 신호 확인](#analysis-refinement).
SPCE 240037·240121은 TC-Z9 변동 범위와 약 197분 주기가 비슷하고 피크 시점이 다르다.
처음 107분의 중앙값 차이를 지속적인 수준 이상으로 판단하지 않는다. 240133은 한 주기보다 짧다.
240061의 133.1분은 TC-Z3·OP-Z3·CP-Z4를 원본 초 단위로 함께 볼 추가 단서다.


In [ ]:
# %% 관찰 후 해석 수정: 공통 창의 수준 차이를 전체 주기와 대조한다.
iq_period_rows=[]
iq_peaks={}
for lot_id in [240037,240121,240133]:
    p=rt_profiles[lot_id]
    smooth=p[("TC-Z9","median")].rolling(5,center=True,min_periods=1).median()
    # 원본 시계열의 큰 피크를 찾는 탐색 규칙이다. 창의 양 끝은 완결된 피크로 세지 않는다.
    candidates=smooth.index[(smooth.eq(smooth.rolling(81,center=True,min_periods=1).max())) &
                             (smooth>smooth.quantile(.8)) & (smooth.index>10) & (smooth.index<smooth.index.max()-10)]
    peaks=[]
    for point in candidates:
        if not peaks or point-peaks[-1]>80: peaks.append(int(point))
    iq_peaks[lot_id]=peaks
    iq_period_rows.append({"LOT":lot_id,"관측 길이(분)":round(tl_lots.set_index("LOT_NO").loc[lot_id,"DURATION_MIN"],1),
                           "TC-Z9 최소":p[("TC-Z9","median")].min(),"TC-Z9 최대":p[("TC-Z9","median")].max(),
                           "피크 경과 분":", ".join(map(str,peaks)) if peaks else "완결된 피크 확인 불가",
                           "피크 간격(분)":", ".join(map(str,np.diff(peaks))) if len(peaks)>1 else "주기 비교 불가"})
iq_period=pd.DataFrame(iq_period_rows)
fig,axes=plt.subplots(2,1,figsize=(21,9),sharex=True)
fig.subplots_adjust(left=.08,right=.98,top=.80,bottom=.11,hspace=.35)
for lot_id in [240037,240121,240133]:
    p=rt_profiles[lot_id]
    color=rt_lot_color[lot_id]
    duration=tl_lots.set_index("LOT_NO").loc[lot_id,"DURATION_MIN"]
    axes[0].plot(p.index,p[("TC-Z9","median")],color=color,lw=1.7,label=f"{lot_id} · {duration:.1f}분")
    for i,sensor in enumerate(["TC-Z7","TC-Z8"]):
        axes[1].plot(p.index,p[(sensor,"median")],color=color,lw=1.2,ls="-" if i==0 else "--")
    if iq_peaks[lot_id]:
        peak=iq_peaks[lot_id][0]
        axes[0].scatter(peak,p.loc[peak,("TC-Z9","median")],s=43,facecolors="white",edgecolors=RT_RED,lw=1.7,zorder=5)
        axes[0].annotate(f"LOT {lot_id} · 첫 피크 {peak}분",(peak,p.loc[peak,("TC-Z9","median")]),xytext=(7,-15 if lot_id==240037 else -32),textcoords="offset points",fontsize=9,color=RT_RED)
for ax in axes:
    ax.axvspan(0,107,color="#f3c44c",alpha=.14,zorder=0)
    ax.set_ylabel("온도 [℃]")
    rt_style(ax)
axes[0].set_title("과시효 [TC-Z9] · 전체 관측의 주기와 피크 시점",loc="left",fontsize=11)
axes[1].set_title("과시효 [TC-Z7 실선 · TC-Z8 점선] · 인접 구간 비교",loc="left",fontsize=11)
axes[1].set_xlim(0,tl_lots.loc[tl_lots.GRADE.eq("SPCE"),"DURATION_MIN"].max())
axes[1].set_xlabel("LOT 시작 후 실제 경과시간 [분]")
fig.suptitle("해석 수정. SPCE TC-Z9 · 처음 107분의 차이는 전체 주기와 함께 본다",x=.02,y=.98,ha="left",fontsize=17,fontweight="bold",color="#17344d")
fig.text(.02,.92,"빨간 점: 첫 피크 시점 비교 · 옅은 구간: 공통 107분",fontsize=10.5,color="#435b6c")
fig.legend(*axes[0].get_legend_handles_labels(),loc="upper right",bbox_to_anchor=(.98,.875),ncol=3,frameon=False,fontsize=10)
period_file=IQ_ZOOM_DIR/"04C_SPCE_TC-Z9_전체주기_공통창.png"
rt_finish(fig,period_file,dpi=150)

# 단일 LOT에서 TC·OP·CP가 동시에 변한 시각을 함께 보되 인과를 단정하지 않는다.
raw=temp_flow.loc[temp_flow.LOT_NO.eq(240061)].copy()
point=raw.loc[raw["CP-Z4"].idxmax()]
left=point.MEAS_DT-pd.Timedelta(minutes=5);right=point.MEAS_DT+pd.Timedelta(minutes=5)
window=raw.loc[raw.MEAS_DT.between(left,right)]
fig,axes=plt.subplots(3,1,figsize=(21,9.5),sharex=True)
fig.subplots_adjust(left=.08,right=.98,top=.83,bottom=.10,hspace=.35)
for ax,sensor,color in zip(axes,["TC-Z3","OP-Z3","CP-Z4"],RT_COLORS):
    ax.plot(window.ELAPSED_MIN,window[sensor],color=color,lw=1.1)
    ax.axvline(point.ELAPSED_MIN,color=RT_RED,ls=":",lw=1.1)
    ax.scatter(point.ELAPSED_MIN,point[sensor],s=40,facecolors="none",edgecolors=RT_RED,zorder=5)
    ax.set_ylabel(rt_measure_label(sensor))
    ax.set_title(f'가열·균열 [{sensor}]',loc='left',fontsize=10,pad=8)
    rt_style(ax)
axes[-1].set_xlim(point.ELAPSED_MIN-5,point.ELAPSED_MIN+5)
axes[-1].set_xlabel("LOT 시작 후 경과시간 [분]")
axes[-1].text(.99,.94,f"CP 최댓값 {point['CP-Z4']:.4g} · 같은 시각 OP-Z3={point['OP-Z3']:.4g}%",
    transform=axes[-1].transAxes,ha="right",va="top",fontsize=10,color=RT_RED,
    bbox=dict(fc="white",ec="none",alpha=.92,pad=3))
fig.suptitle("추가 단서. LOT 240061 · 같은 시각의 TC-Z3·OP-Z3·CP-Z4",x=.02,y=.98,ha="left",fontsize=17,fontweight="bold",color="#17344d")
fig.text(.02,.92,f"CP-Z4 관측 최댓값 시각 {point.MEAS_DT:%H:%M:%S} · LOT 시작 후 {point.ELAPSED_MIN:.1f}분",fontsize=10.5,color="#435b6c")
joint_file=IQ_ZOOM_DIR/"04D_LOT_240061_TC_OP_CP_원본초.png"
rt_finish(fig,joint_file,dpi=150)


<a id="analysis-refinement"></a>

### 시계열을 보고 수정한 해석

#### SPCE 240037의 높은 요약값을 어떻게 다시 해석했는가

처음 107분의 TC-Z9 중앙값은 240037이 다른 두 SPCE보다 높다. 개별 LOT의 전체 시계열을 열어 보니
240037과 240121은 TC-Z9의 변동 범위와 약 197분 주기가 비슷하고, 피크 시점이 다르다.
따라서 공통 107분의 중앙값 차이를 지속적인 수준 이상으로 바로 해석하지 않는다.
240133은 107.4분으로 약 한 주기보다 짧아 전체 주기나 피크 간격 비교를 할 수 없다.

| LOT | 관측 길이(분) | TC-Z9 최소 | TC-Z9 최대 | 피크 경과 분 | 피크 간격(분) |
| --- | --- | --- | --- | --- | --- |
| 240037 | 611.3 | 319.298 | 342.189 | 27, 223, 421 | 196, 198 |
| 240121 | 651.2 | 319.343 | 342.312 | 150, 348, 543 | 198, 195 |
| 240133 | 107.4 | 319.342 | 338.977 | 완결된 피크 확인 불가 | 주기 비교 불가 |

이 표의 피크는 1분 중앙값을 5분 중앙값으로 완화한 후, ±40분의 큰 피크 중 80분 이상 떨어진 후보를 찾은 탐색 결과다.
완결되지 않은 양 끝 피크는 제외한다. 운영 주기 정의나 고장 판정 기준이 아니다.
TC-Z7·Z8은 상대적으로 좁은 범위에서 변하며 큰 느린 주기는 TC-Z9에서 두드러진다.
이 차이의 의미는 Z9의 공정 역할·설정·운전 모드를 확인한 뒤 판단한다.

**그림을 바꾼 이유:** 짧은 공통 창의 값 차이 → 전체 관측의 주기 확인 → 위상 차이 가능성 확인 → 전체 주기와 공통 창을 함께 보여주는 그림 추가.
다음 비교에서는 유사한 운전 조건 아래 주기·진폭·위상을 분리해서 확인한다.

#### SPHC 240061에서 추가로 본 동시 신호

2024/06/15 03:17:06(LOT 시작 후 133.1분)에 CP-Z4=0.4709(무차원), TC-Z3=870℃, OP-Z3=0%가 함께 관측된다.


이 구간은 SPHC 전체 4 LOT의 공통 비교 74분 밖이다. 같은 경과시간이 존재하는 240069·240091과 따로 비교할 수 있다.
짧은 240044를 133분까지 연장하거나 값을 채우지 않는다. 위 그림은 원본에서 동시 변화가 있는지 확인한 것이며,
OP=0의 장치 의미·신호 지연·설정 정의가 없으므로 원인이나 제어 이상을 확정하지 않는다.
CP-Z4M은 정의가 확인되지 않은 보조 신호로 표시만 하고 물리 해석 근거로 사용하지 않는다.

OP의 단위는 %, CP는 무차원이다. 두 신호는 온도와 독립 축에서 비교한다. 원본은 1초 기록이며 확대 그림의 가로축은 분으로 통일했다.

이번에 원본으로 확인한 단서를 다음 검토의 출발점으로 남긴다. 정비의 결과라고 결론 내린 것은 아니다.


[SPCE 전체 주기 그림](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04C_SPCE_TC-Z9_전체주기_공통창.png)

[240061 TC·OP·CP 원본 초 그림](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04D_LOT_240061_TC_OP_CP_원본초.png)


In [ ]:
# %% CP 원본 확대: 1분 중앙값에 가려진 짧은 관측을 대응 TC·OP와 함께 확인한다.
rt_cp_zoom_manifest=[]
for lot in [240069,240037]:
    p=rt_review_points.loc[rt_review_points.LOT_NO.eq(lot)&rt_review_points.SENSOR.eq("CP-Z4")&rt_review_points.METHOD.eq("raw_extreme")].iloc[0]
    start,end=max(0,p.POINT_MIN-3),p.POINT_MIN+3
    available=[]
    fig,axes=plt.subplots(3,1,figsize=(19,10),sharex=True)
    fig.subplots_adjust(left=.08,right=.97,top=.83,bottom=.09,hspace=.38)
    sensors=["TC-Z4","OP-Z4","CP-Z4"]
    for peer in lot_info.loc[lot_info.GRADE.eq(p.GRADE),"LOT_NO"].astype(int):
        raw=temp_flow.loc[temp_flow.LOT_NO.eq(peer)&temp_flow.ELAPSED_MIN.between(start,end)]
        if raw.empty:continue
        available.append(peer)
        for ax,sensor in zip(axes,sensors):
            ax.plot(raw.ELAPSED_MIN,raw[sensor],color=rt_lot_color[peer],lw=1.1,alpha=.9)
    observed=temp_flow.loc[temp_flow.LOT_NO.eq(lot)&temp_flow.ELAPSED_MIN.eq(p.POINT_MIN)].iloc[0]
    for ax,sensor in zip(axes,sensors):
        value=observed[sensor]
        ax.scatter(p.POINT_MIN,value,s=45,facecolors='white',edgecolors=RT_RED,lw=1.7,zorder=6)
        ax.set_ylabel(rt_measure_label(sensor),fontsize=10)
        ax.set_title(f'가열·균열 [{sensor}]',loc='left',fontsize=10,pad=8)
        rt_point_box(ax,(p.POINT_MIN,value),f'LOT {lot} · {p.POINT_MIN:.2f}분\n{sensor} · CP 급변과 동시 관측')
        rt_style(ax)
    axes[-1].set_xlim(start,end)
    axes[-1].set_xlabel("LOT 시작 후 경과시간 [분]")
    rt_header(fig,f"CP 확인 확대. {p.GRADE} · LOT {lot}","빨간 점: CP 극값과 같은 시점의 TC·OP 관측")
    rt_lot_legend(fig,available,y=.95)
    target=IQ_ZOOM_DIR/f"04E_LOT_{lot}_CP_극값_TC_OP.png"
    rt_finish(fig,target,dpi=150)
    rt_cp_zoom_manifest.append(dict(LOT_NO=lot,GRADE=p.GRADE,POINT_MIN=p.POINT_MIN,PNG=str(target)))


<a id="analysis-cp-original"></a>

### CP 원본 확대의 확인값

##### SPHC LOT 240069 CP 원본 확대

관측 시각 2024/07/08 02:31:29, 경과 574.4833분. CP-Z4=0.909111(무차원), OP-Z4=39.0179%, TC-Z4=860.042℃.

모든 선은 원본 해상도를 유지하며 가로축만 분으로 환산했다. 해당 경과 분이 실제 관측된 LOT만 비교한다. 붉은 점의 동시 관측으로 원인·제어 이상을 확정하지 않는다.

[TC·OP·CP 확대 그림](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04E_LOT_240069_CP_극값_TC_OP.png)

##### SPCE LOT 240037 CP 원본 확대

관측 시각 2024/04/13 23:57:59, 경과 41.9833분. CP-Z4=0.60676(무차원), OP-Z4=54.9267%, TC-Z4=860.4℃.

모든 선은 원본 해상도를 유지하며 가로축만 분으로 환산했다. 해당 경과 분이 실제 관측된 LOT만 비교한다. 붉은 점의 동시 관측으로 원인·제어 이상을 확정하지 않는다.

[TC·OP·CP 확대 그림](../figures/05_all_data_timeseries_jinny_01/04_LOT별상세/02_원본확대/04E_LOT_240037_CP_극값_TC_OP.png)


## 그림 6·7. 용접은 어떤 질문을 확인하는가

그림 6은 같은 조건에서 전력만 0인 COIL을 계측·운전 확인 대상으로 분리한다. 그림 7은 조건별 전력 차이의 발생 날짜와 관측 범위를 확인하는 참고 자료다. 실제 전력 W는 사용자 제공 표의 가정이며 확인이 필요하다. 장기 차이를 설비 마모나 수리 효과로 판정하지 않는다. 조건 기준은 같은 속도·길이·전력/주파수/듀티 설정·게이트 시간과 같은 구간순번의 전체 관측 중앙값이다. 비교 COIL 4개 미만인 조합은 제외한다. 전력 차이 확대 범위 밖 값은 첫 패널과 전체 CSV에 보존한다.

In [ ]:
# 용접은 COIL의 실제 경과 초와 달력 시간을 사용한다.
wld2 = wld.sort_values(["COIL_ID", "MEAS_DT"]).copy()
wld2["RECIPE"] = ("SPD=" + wld2["SPD"].astype(str) + "|LEN=" + wld2["LEN"].astype(str)
                   + "|PWRSET=" + wld2["SET-PWR"].astype(str) + "|FRQ=" + wld2["SET-FRQ"].astype(str) + "|DTY=" + wld2["SET-DTY"].astype(str) + "|GATE=" + wld2["GATE-MS"].astype(str))
recipe_count = wld2["RECIPE"].value_counts().rename_axis("RECIPE").reset_index(name="N")
ref = (wld2.groupby(["RECIPE", "SEQ_NO"]).agg(REF_MEDIAN=("PWR", "median"),
        REF_SCALE=("PWR", robust_scale), REF_N_COIL=("COIL_ID", "nunique")).reset_index())
wld_r = wld2.merge(ref, on=["RECIPE", "SEQ_NO"], how="left", validate="many_to_one")
wld_r["PWR_DELTA"] = (wld_r["PWR"] - wld_r["REF_MEDIAN"]).where(wld_r["REF_N_COIL"].ge(4))
wld_r["PWR_RZ_RECIPE_SEQ"] = (wld_r["PWR_DELTA"] / wld_r["REF_SCALE"]).replace([np.inf, -np.inf], np.nan)
coil_rz = (wld_r.groupby("COIL_ID").agg(START=("MEAS_DT", "min"), END=("MEAS_DT", "max"),
    PWR_RZ_Q75=("PWR_RZ_RECIPE_SEQ", lambda s: s.abs().quantile(.75)),
    PWR_RZ_MAX=("PWR_RZ_RECIPE_SEQ", lambda s: s.abs().max()),
    PWR_ZERO_SHARE=("PWR", lambda s: s.eq(0).mean() * 100),
    PWR_MEDIAN=("PWR", "median"), PWR_DELTA_MEDIAN=("PWR_DELTA", "median")).reset_index())
rt_zero_coils = coil_rz.loc[coil_rz["PWR_ZERO_SHARE"].gt(0)].sort_values("START")
rt_weld_case = (rt_zero_coils.iloc[0].COIL_ID if len(rt_zero_coils) else
                coil_rz.sort_values("PWR_RZ_Q75", ascending=False).iloc[0].COIL_ID)
case_frame = wld2.loc[wld2["COIL_ID"].eq(rt_weld_case)].sort_values("SEQ_NO")
# 순서별 설정 일치율과 시간 근접도로 참고 COIL을 고른다.
case_recipe = case_frame.set_index("SEQ_NO")["RECIPE"]
recipe_matrix = wld2.pivot(index="COIL_ID", columns="SEQ_NO", values="RECIPE")
matches = recipe_matrix.eq(case_recipe, axis="columns").mean(axis=1).rename("MATCH_RATE")
normal = coil_rz.loc[coil_rz["PWR_ZERO_SHARE"].eq(0)].merge(matches, on="COIL_ID")
normal["DISTANCE_S"] = (normal["START"] - case_frame["MEAS_DT"].min()).abs().dt.total_seconds()
normal = normal.sort_values(["MATCH_RATE", "DISTANCE_S"], ascending=[False, True])
rt_weld_reference = normal.iloc[0].COIL_ID
rt_weld_match = normal.iloc[0].MATCH_RATE

def rt_absolute_frame(frame, signals, freq="1min"):
    """달력 시간의 비어 있는 분은 NaN으로 유지하여 관측 창 사이를 잇지 않는다."""
    if frame.empty:
        return pd.DataFrame(columns=signals)
    return frame.set_index("MEAS_DT")[signals].resample(freq).median()

**결과 — 전력 0이 기록된 COIL**

| 행 | COIL_ID | START | END | PWR_ZERO_SHARE |
| --- | --- | --- | --- | --- |
| 649 | C2400650 | 2024-05-04 09:09:37.552 | 2024-05-04 09:11:49.201 | 100.0 |
| 678 | C2400679 | 2024-05-04 12:53:45.618 | 2024-05-04 12:55:57.958 | 100.0 |

In [ ]:
# 그림 6: 온도 없는 사례는 용접 신호 검증으로 분리한다.
WLD_LABELS={'PWR':'실제 전력 [W]','SET-PWR':'설정 전력 [%]','SET-DTY':'설정 듀티 [%]',
    'SET-FRQ':'설정 주파수 [Hz]','SPD':'용접 속도 [mm/s]','LEN':'용접 길이 [mm]','GATE-MS':'게이트 온 시간 [ms]'}
fig,axes=plt.subplots(5,1,figsize=(19,13),gridspec_kw={'height_ratios':[1.3,1,1,1,1.2]})
fig.subplots_adjust(left=.10,right=.97,top=.82,bottom=.07,hspace=.65)
for coil_id,color in [(rt_weld_case,RT_RED),(rt_weld_reference,'#087eaa')]:
    raw=wld_r.loc[wld_r.COIL_ID.eq(coil_id)].sort_values('MEAS_DT')
    minutes=(raw.MEAS_DT-raw.MEAS_DT.min()).dt.total_seconds()/60
    for ax,col in zip(axes[:4],['PWR','SET-PWR','SPD','GATE-MS']):
        ax.plot(minutes,raw[col],color=color,marker='o',ms=3,lw=1,label=coil_id)
for ax,col in zip(axes[:4],['PWR','SET-PWR','SPD','GATE-MS']):
    ax.set_ylabel(WLD_LABELS[col],fontsize=10);rt_style(ax)
axes[3].set_xlabel('COIL 시작 후 경과시간 [분]')
axes[0].set_ylim(-100,2000)
rt_point_box(axes[0],(.8,0),f'{rt_weld_case}\n全 39구간 전력 0 → 계측·운전 기록 확인'.replace('全 ','전체 '))
axes[0].legend(loc='upper right',bbox_to_anchor=(1,1.25),ncol=2,frameon=False,fontsize=10)
zero_date=case_frame.MEAS_DT.min().normalize()
localwork=audit_work.loc[audit_work.EQP_TAG.eq(CAL_TAG) &
    (audit_work.STRT_DT<zero_date+pd.Timedelta(hours=14)) & (audit_work.END_DT>=zero_date)]
context_rows=[]
for yy,task in enumerate(localwork.sort_values('STRT_DT').itertuples()):
    color='#bc5267' if task.SCOPE=='CAL01' else '#4d6cb2'
    axes[4].plot([task.STRT_DT,task.END_DT],[yy]*2,color=color,marker='x',lw=2,ms=5)
    context_rows.append(f'{task.SCOPE} {task.MNT_TYPE} · {task.WO_NO[-5:]}')
for coil in rt_zero_coils.itertuples():
    yy=len(localwork)+len(context_rows)-len(localwork)
    axes[4].plot([coil.START,coil.END],[yy]*2,color=RT_RED,lw=6,solid_capstyle='butt')
    axes[4].scatter(coil.START,yy,color=RT_RED,s=20)
    context_rows.append(f'{coil.COIL_ID} · 전력 0')
axes[4].set_yticks(range(len(context_rows)),context_rows,fontsize=9)
axes[4].set_xlim(zero_date,zero_date+pd.Timedelta(hours=14));axes[4].set_ylim(-.7,len(context_rows)-.3)
axes[4].xaxis.set_major_locator(mdates.HourLocator(interval=2));axes[4].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[4].set_title('05/04 · 소둔로 BM 완료 뒤 전력 0 기록',loc='left',fontsize=11,pad=8)
rt_style(axes[4])
rt_header(fig,'그림 6. 같은 용접 조건인데 전력만 0인가',
    f'{rt_weld_case} ↔ {rt_weld_reference} · 구간별 조건 일치 {rt_weld_match:.0%} · 0 기록은 2 COIL')
rt_finish(fig,Path(WLD_DIR)/'06_WLD01_COIL_실제초_PWR_설정.png')


<a id="analysis-welding"></a>

### 용접을 보는 이유와 한계

#### 그림 6을 보는 이유

용접 실제 전력 0이 **설정 변경에 따른 값인지, 계측·운전 상태를 확인해야 하는 기록인지** 구분한다. C2400650·C2400679는 전체 39구간에서 전력이 0이다. 첫 사례와 가까운 비교 COIL C2400651은 속도·길이·설정 전력·주파수·듀티·게이트 시간의 구간별 조건이 100% 일치하며 전력은 양수다. 같은 설정에서 측정값만 0인 기록을 추가 확인 대상으로 남긴다. 용접 불량·실제 무전력·센서 오류 중 무엇인지는 아직 정해지지 않았다.

**다른 CSV와의 관계:** 5/4 소둔로 BM WO-2024-00717은 08:15에 끝났다. 첫 0 기록은 09:09:37이므로 작업 중 겹침이 아니다. 소둔로 BM REMARK는 동작 불량이다. 이 시각에는 소둔로 온도 관측이 없어 온도 변화나 더미코일 판정에 사용할 수 없다. 그래서 빈 온도 패널이 있던 '그림 5 참고 C2400650'은 본문에서 제외했다.

**단위:** 사용자 제공 정의에 따라 속도 mm/s, 길이 mm, 주파수 Hz, 듀티·설정 전력 %, 게이트 시간 ms를 사용한다. 실제 전력 W는 제공 표의 **가정**이며 장비 신호 정의로 확인해야 한다. OP는 %, CP는 무차원이다.

**조건 비교:** 같은 속도·길이·설정 전력·주파수·듀티·게이트 시간과 같은 구간순번을 비교한다. 전기간 중앙값은 탐색용 비교 기준이며 고장 임계값이나 독립 검증 기준은 아니다.


##### 가까운 작업의 원문

| EQP_TAG | WO_NO | MNT_TYPE | STRT_DT | END_DT | REMARK |
| --- | --- | --- | --- | --- | --- |
| P1-CR1-CAL01 | WO-2024-00717 | BM | 05/04 04:50:00 | 05/04 08:15:00 | 동작 불량 |


## 같은 설정·SEQ 기준은 보조 근거

기준은 전체 관측기간에서 같은 설정·SEQ 조합의 PWR 중앙값이다.
기준 COIL 4개 미만 조합은 보정 차이/점수를 표시하지 않는다.
이는 사후 관측 비교이며 미래 예측 기준이나 고장 판정이 아니다.
실제 PWR와 제어 설정, 발생 날짜를 함께 확인한다.

In [ ]:
# 조건별 기준은 앞 준비 셀에서 계산했다. COIL별 표는 원본 PWR와 함께 보존한다.

**결과 — COIL별 용접 참고 지표**

| 행 | COIL_ID | START | END | PWR_RZ_Q75 | PWR_RZ_MAX | PWR_ZERO_SHARE | PWR_MEDIAN | PWR_DELTA_MEDIAN |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 678 | C2400679 | 2024-05-04 12:53:45.618 | 2024-05-04 12:55:57.958 | 286.995818 | 385.808714 | 100.0 | 0.0 | -1689.0 |
| 649 | C2400650 | 2024-05-04 09:09:37.552 | 2024-05-04 09:11:49.201 | 286.995818 | 385.808714 | 100.0 | 0.0 | -1689.0 |
| 3614 | C2403615 | 2024-12-26 17:23:04.579 | 2024-12-26 17:25:16.311 | 2.192095 | 3.147624 | 0.0 | 1677.0 | -12.0 |
| 0 | C2400001 | 2024-01-03 10:11:45.181 | 2024-01-03 10:13:57.874 | 1.348982 | 1.798642 | 0.0 | 1698.0 | 7.0 |
| 1 | C2400002 | 2024-01-03 10:29:49.718 | 2024-01-03 10:32:02.862 | 1.433293 | 1.798642 | 0.0 | 1698.0 | 8.0 |
| 2 | C2400003 | 2024-01-03 10:59:17.756 | 2024-01-03 11:01:30.348 | 1.348982 | 1.798642 | 0.0 | 1698.0 | 7.0 |
| 3 | C2400004 | 2024-01-03 11:07:13.665 | 2024-01-03 11:09:26.832 | 1.348982 | 1.798642 | 0.0 | 1698.0 | 7.0 |
| 4 | C2400005 | 2024-01-03 11:48:07.014 | 2024-01-03 11:50:18.380 | 2.529340 | 3.372454 | 0.0 | 1704.0 | 14.0 |
| 5 | C2400006 | 2024-01-03 12:28:58.561 | 2024-01-03 12:31:11.753 | 1.348982 | 1.798642 | 0.0 | 1697.0 | 7.0 |
| 6 | C2400007 | 2024-01-03 13:25:19.215 | 2024-01-03 13:27:31.609 | 1.348982 | 1.798642 | 0.0 | 1697.0 | 7.0 |

In [ ]:
# 그림 7: 순위 막대 대신 연중 PWR·조건 보정 차이·설정·관측 범위를 같은 날짜로 비교.
fig, axes = plt.subplots(4, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={"height_ratios": [1.25, 1.1, .8, .8]})
fig.subplots_adjust(left=.09, right=.97, top=.83, bottom=.12, hspace=.3)
# 속도 250인 공통 작업 단계에서 설정 82/83을 구분한다.
phase = wld_r.loc[wld_r["SPD"].eq(250)].copy()
phase_coil = phase.groupby(["COIL_ID", "SET-PWR"]).agg(
    START=("MEAS_DT", "min"), PWR=("PWR", "median"), PWR_DELTA=("PWR_DELTA", "median")).reset_index()
for setting, color in [(82, "#087eaa"), (83, "#e57c12")]:
    frame = phase_coil.loc[phase_coil["SET-PWR"].eq(setting)]
    axes[0].scatter(frame["START"], frame["PWR"], s=7, color=color, alpha=.5, label=f"설정 전력 {setting}%")
    finite = frame.loc[frame["PWR"].gt(0)]
    # 0은 첫 행에서 모두 표시한다. 둘째 행은 양수 기록의 작은 추세를 읽는 확대다.
    axes[1].scatter(finite["START"], finite["PWR_DELTA"], s=7, color=color, alpha=.5)
axes[0].set_ylabel("실제 전력 [W]", fontsize=9)
axes[1].set_ylabel("조건 기준 전력 차이 [W]", fontsize=9)
axes[1].axhline(0, color="#8999a8", ls="--", lw=.8)
PWR_DETAIL_RANGE = (-50, 30)
outside = phase_coil.loc[phase_coil["PWR"].gt(0) &
                         ~phase_coil["PWR_DELTA"].between(*PWR_DETAIL_RANGE) & phase_coil["PWR_DELTA"].notna()]
axes[1].set_ylim(*PWR_DETAIL_RANGE)
axes[1].set_title(f"같은 조건 대비 전력 차이 확대 [-50, +30 W]",
                  loc="left", fontsize=10, pad=8)
axes[0].legend(loc="upper right", bbox_to_anchor=(1, 1.22), ncol=2, fontsize=9, frameon=False)
event_boxes = []
for i,coil in enumerate(rt_zero_coils.itertuples()):
    axes[0].scatter(coil.START,0,s=60,facecolors='none',edgecolors=RT_RED,lw=1.5)
    note=axes[0].annotate(f'{coil.COIL_ID}\n05/04 · 전력 0',xy=(coil.START,0),
        xytext=(.23+.16*i,.20+.12*i),textcoords='axes fraction',ha='center',fontsize=8.5,color=RT_RED,
        bbox=dict(boxstyle='round,pad=.35',fc='white',ec=RT_RED,lw=.7),
        arrowprops=dict(arrowstyle='-',color=RT_RED,lw=.8))
    event_boxes.append(note.get_bbox_patch())
# 소둔로 작업만 용접 계측과 시간으로 대조한다.
frame=audit_work.sort_values('STRT_DT')
offsets=tl_stack(list(zip(frame.STRT_DT,frame.END_DT)),min_days=1)
for task,dy in zip(frame.itertuples(),offsets):
    color=RT_RED if task.WELD_COILS else '#a0aab6'
    axes[2].plot([task.STRT_DT,task.END_DT],[dy]*2,color=color,lw=2)
    axes[2].scatter(task.STRT_DT+(task.END_DT-task.STRT_DT)/2,dy,marker='x',s=20,color=color)
axes[2].set_yticks([0],['CAL01 작업 44건'],fontsize=9)
axes[2].set_ylim(-.6,.6)
axes[2].set_title('소둔로 작업 · 빨강: 용접 관측과 시간 겹침',loc='left',fontsize=10,pad=8)
daily = tl_wld_day
axes[3].bar(daily["DATE"], daily["N_COIL"], width=1.3, color="#e2a95f", alpha=.8)
for lot in lot_info.itertuples():
    axes[3].axvspan(lot.LOT_START, lot.LOT_END, color="#087eaa", alpha=.45)
axes[3].set_ylabel("용접 관측 [COIL]", fontsize=9)
for ax in axes:
    for plan in tl_plans.itertuples():
        ax.axvspan(plan.STRT_DT, plan.END_EXCLUSIVE, color=TL_COLORS["PLAN"], zorder=0)
    rt_style(ax)
axes[3].set_xlim(RT_YEAR0, RT_YEAR1)
axes[3].xaxis.set_major_locator(mdates.MonthLocator())
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
axes[3].set_xticks(pd.date_range(RT_YEAR0, periods=12, freq="MS"))
axes[3].set_xlabel("")
rt_header(fig, "그림 7. WLD01 연중 PWR·설정·온도 관측·계획수리의 시간 대응",
          "용접 속도 250 mm/s · 설정 전력 82% / 83% · 빨간 점: 전력 0 기록 · 파란 음영: 계획수리")
fig._collision_groups = [event_boxes]
rt_finish(fig, Path(WLD_DIR) / "07_WLD01_연간_PWR_설정_관측범위.png")

그림 7의 설정 전력은 %이고 실제 전력은 W 가정이다. 서로 빼는 값이 아니다. 둘째 패널은 실제 전력에서 **같은 조건의 실제 전력 중앙값**을 뺀 값이다. 기준에는 전기간 관측이 포함되므로 독립 고장 검증이 아니다. 마지막 패널은 용접 관측 수이며 짙은 파랑은 온도 LOT 관측 창이다. 작업 교집합의 전수 결과와 최종 연간 지도에서 실제 작업 맥락을 확인한다.

## 그림 5. 동시간 자료를 함께 봐야 하는 네 LOT

온도·제어·용접이 함께 관측된 창에서 변화와 작업 경계의 선후관계를 확인한다. 개별 COIL의 소재가 해당 LOT라고 연결하지 않는다. C2400650은 온도가 없어 그림 6의 용접 확인 분기로 보낸다.

In [ ]:
wld_bounds = (
    wld2.groupby("COIL_ID", as_index=False)
        .agg(
            WLD_START=("MEAS_DT", "min"),
            WLD_END=("MEAS_DT", "max")
        )
)

context_rows = []

for _, lot in lot_info.iterrows():
    start = lot["LOT_START"]
    end = lot["LOT_END"]

    inside = wld_bounds[
        (wld_bounds["WLD_START"] <= end)
        & (wld_bounds["WLD_END"] >= start)
    ]

    pm12 = wld_bounds[
        (wld_bounds["WLD_START"] >= start - pd.Timedelta(hours=12))
        & (wld_bounds["WLD_START"] <= start + pd.Timedelta(hours=12))
    ]

    pm24 = wld_bounds[
        (wld_bounds["WLD_START"] >= start - pd.Timedelta(hours=24))
        & (wld_bounds["WLD_START"] <= start + pd.Timedelta(hours=24))
    ]

    before = wld_bounds[wld_bounds["WLD_END"] <= start]
    after = wld_bounds[wld_bounds["WLD_START"] >= start]

    prev_min = (
        (start - before["WLD_END"].max()).total_seconds() / 60
        if len(before) else np.nan
    )
    next_min = (
        (after["WLD_START"].min() - start).total_seconds() / 60
        if len(after) else np.nan
    )

    context_rows.append({
        "LOT_NO": lot["LOT_NO"],
        "GRADE": lot["GRADE"],
        "LOT_START": start,
        "LOT_END": end,
        "WLD_INSIDE_COIL": len(inside),
        "WLD_PM12H_COIL": len(pm12),
        "WLD_PM24H_COIL": len(pm24),
        "PREV_WLD_MIN": prev_min,
        "NEXT_WLD_MIN": next_min
    })

lot_wld_context = pd.DataFrame(context_rows)

**결과 — LOT와 용접 관측의 시간 맥락**

| 행 | LOT_NO | GRADE | LOT_START | LOT_END | WLD_INSIDE_COIL | WLD_PM12H_COIL | WLD_PM24H_COIL | PREV_WLD_MIN | NEXT_WLD_MIN |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 240001 | SGCC | 2024-01-02 03:36:00 | 2024-01-02 15:04:30 | 0 | 0 | 0 | NaN | 1835.753017 |
| 1 | 240024 | SPFH590 | 2024-03-07 21:22:00 | 2024-03-08 06:29:48 | 0 | 0 | 0 | 62534.498750 | 9390.863550 |
| 2 | 240037 | SPCE | 2024-04-13 23:16:00 | 2024-04-14 09:27:19 | 0 | 0 | 0 | 6103.770533 | 2040.385250 |
| 3 | 240038 | SPCEN | 2024-04-16 11:35:00 | 2024-04-16 13:39:56 | 0 | 12 | 51 | 16.607900 | 22941.976433 |
| 4 | 240044 | SPHC | 2024-05-03 18:50:00 | 2024-05-03 20:04:38 | 0 | 0 | 79 | 1489.654250 | 859.625867 |
| 5 | 240058 | DP590 | 2024-06-09 05:26:00 | 2024-06-09 09:02:53 | 0 | 80 | 87 | 2138.600133 | 233.324733 |
| 6 | 240061 | SPHC | 2024-06-15 01:04:00 | 2024-06-15 04:49:23 | 0 | 36 | 73 | 7633.168900 | 489.855283 |
| 7 | 240069 | SPHC | 2024-07-07 16:57:00 | 2024-07-08 06:29:34 | 0 | 0 | 0 | 10031.774317 | 14113.022950 |
| 8 | 240091 | SPHC | 2024-09-08 22:03:00 | 2024-09-09 08:44:12 | 0 | 0 | 0 | 30532.933817 | 5010.901267 |
| 9 | 240108 | SPFH590 | 2024-10-18 04:05:00 | 2024-10-18 11:50:53 | 12 | 41 | 41 | 7837.310133 | 306.484400 |
| 10 | 240114 | SPCEN | 2024-11-04 15:40:00 | 2024-11-04 20:59:02 | 11 | 30 | 30 | 10.808233 | 0.408033 |
| 11 | 240117 | SPCD | 2024-11-13 07:08:00 | 2024-11-13 14:32:19 | 31 | 48 | 95 | 881.902983 | 116.029450 |
| 12 | 240121 | SPCE | 2024-11-24 07:31:00 | 2024-11-24 18:22:12 | 28 | 28 | 28 | 8050.463800 | 93.970550 |
| 13 | 240133 | SPCE | 2024-12-28 14:57:00 | 2024-12-28 16:44:24 | 0 | 0 | 0 | 2731.728150 | NaN |

In [ ]:
# 그림 5: 동시간 관측 4 LOT에서 작업 경계와 각 신호의 변화를 함께 확인한다.
joint_manifest=[]
for lot_id in audit_lots.loc[audit_lots.WELD_COIL.gt(0),'LOT_NO']:
    info=lot_info.set_index('LOT_NO').loc[lot_id]
    start,end=info.LOT_START,info.LOT_END
    raw=temp.loc[temp.LOT_NO.eq(lot_id)]
    weld=wld_r.loc[wld_r.MEAS_DT.between(start,end)]
    works=audit_work.loc[audit_work.EQP_TAG.eq(CAL_TAG)&(audit_work.STRT_DT<=end)&(audit_work.END_DT>=start)]
    # 작업이 없는 창은 빈 작업 패널을 만들지 않는다.
    npanels=6 if len(works) else 5
    fig,axes=plt.subplots(npanels,1,figsize=(20,3+2*npanels),sharex=True)
    fig.subplots_adjust(left=.10,right=.97,top=.84,bottom=.07,hspace=.50)
    for ax,sensors,process in [(axes[0],['TC-Z3','TC-Z4','TC-Z5','TC-Z6'],'가열·균열'),
        (axes[1],['OP-Z3','OP-Z4','OP-Z5','OP-Z6'],'가열·균열'),
        (axes[2],['CP-Z4','CP-Z4M'],'가열·균열'),
        (axes[3],['TC-OUT1','TC-OUT2'],'냉각')]:
        minute=rt_absolute_frame(raw,sensors)
        for i,s in enumerate(sensors):ax.plot(minute.index,minute[s],lw=1.25,color=RT_COLORS[i],label=s)
        ax.set_title(process+' ['+' · '.join(sensors)+']',loc='left',fontsize=10,pad=8)
        ax.set_ylabel(rt_measure_label(sensors[0]),fontsize=10)
        # 범례는 제목 오른쪽의 패널 바깥 여백에 배치해 신호를 가리지 않는다.
        ax.legend(loc='lower right',bbox_to_anchor=(1,1.06),borderaxespad=0,
                  ncol=len(sensors),fontsize=8,frameon=False)
    for setting,color in [(82,'#df8a09'),(83,'#087eaa'),(38,'#8a94a1'),(37,'#7841bc'),(35,'#7841bc')]:
        s=weld.loc[weld['SET-PWR'].eq(setting)]
        if len(s):axes[4].scatter(s.MEAS_DT,s.PWR,s=9,alpha=.7,color=color,label=f'설정 전력 {setting}%')
    axes[4].set_ylabel('실제 전력 [W]');axes[4].set_title('용접기 · 설정별 실제 전력',loc='left',fontsize=10,pad=8)
    axes[4].legend(loc='lower right',bbox_to_anchor=(1,1.06),borderaxespad=0,
                   ncol=5,fontsize=8,frameon=False)
    if len(works):
        labels=[]
        for yy,task in enumerate(works.sort_values('STRT_DT').itertuples()):
            axes[5].plot([task.STRT_DT,task.END_DT],[yy]*2,marker='x',color='#bc5267',lw=3,ms=6)
            labels.append(f'{task.SCOPE} {task.MNT_TYPE} · {task.WO_NO[-5:]}')
            for ax in axes[:5]:
                ax.axvspan(max(start,task.STRT_DT),min(end,task.END_DT),color='#f5c9cc',alpha=.25,zorder=0)
                ax.axvline(task.STRT_DT,color='#bc5267',lw=.7,ls=':');ax.axvline(task.END_DT,color='#bc5267',lw=.7,ls=':')
        axes[5].set_yticks(range(len(labels)),labels,fontsize=9);axes[5].set_ylim(-.6,len(labels)-.4)
        both_start,both_end=max(works.STRT_DT),min(works.END_DT)
        if both_end>both_start:
            for ax in axes[:5]:ax.axvspan(both_start,both_end,color=RT_RED,alpha=.11,zorder=0)
    for ax in axes:rt_style(ax)
    axes[-1].set_xlim(start,end);axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5,maxticks=8))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    # 날짜는 제목에, 축에는 필요한 시각 눈금만 표시한다.
    rt_header(fig,f'그림 5. LOT {lot_id} · {info.GRADE} · 온도·제어·용접의 동시간 관측',
        f'{start:%Y/%m/%d %H:%M}–{end:%H:%M} · 용접 {weld.COIL_ID.nunique()} COIL'+(' · 붉은 음영: 소둔로 작업' if len(works) else ''))
    target=RT_JOINT_DIR/f'05_LOT_{lot_id}_온도_용접_정비.png'
    rt_finish(fig,target)
    joint_manifest.append(dict(LOT_NO=int(lot_id),PNG=str(target)))


<a id="analysis-joint"></a>

### 온도·용접·작업을 함께 보는 목적

#### 그림 5에서 무엇을 확인하는가

소둔로 온도·OP·CP, 냉각 온도, 용접 실제 전력을 같은 관측 시간에 배치했다. 용접만 비교한 그림이 아니다. 날짜는 제목, 시각은 x축 눈금에만 둔다.

- **240117:** 작업 전·중·후에 가열·냉각 온도와 제어값이 어떻게 달라지는지 확인한다. TBM 07:40–09:35와 CBM 08:54–12:44가 41분 겹치며, 이 LOT 안에 용접 31 COIL이 관측됐다. 처음부터 나타난 가열대 승온이 작업 경계와 어떻게 놓이는지 본다. 작업 전부터 시작된 변화는 해당 작업 때문에 시작됐다고 설명할 수 없다.
- **240108·240114·240121:** 각각 용접 12·11·28 COIL과 온도가 시간상 겹친다. 같은 시간의 조건 변경과 온도 변화를 대조할 수 있는 참고 창이다. 소둔로 작업 겹침은 0건이므로 '정비 효과 사례'로 붙이지 않는다.

**얻을 수 있는 것:** 각 신호의 변화 시각과 작업 경계의 선후관계, 후속 조사에 필요한 시간창. **아직 얻을 수 없는 것:** 용접이 온도 변화를 일으켰다는 인과관계, 동일 소재의 공정 간 대응. CAL01 LOT와 WLD01 COIL을 잇는 소재 추적 키·라인 속도·통과 지연·설정 이력이 필요하다. 임의의 시간 지연이나 최근접 COIL로 소재를 연결하지 않는다.

그림 5의 정비는 CAL01 소둔로 작업만 포함한다. 부품교체도 이 작업의 WO_NO로 연결한 기록만 사용한다. 용접기 자체 정비와 다른 냉연 설비 작업은 분석 대상에서 제외한다. 소둔로 작업–용접 시간 교집합은 메모리의 `audit_overlap`에서 계산하고 마크다운으로 확인한다. 관측 중 감지되지만 작업은 종료 뒤인 240133과 같은 관계는 그림 1B에서 별도로 확인한다.


## 후보 선별에서 강종만으로 더미 적합성을 정하지 않는다

강종명에 따른 '최우선·활용 가능·제외' 분류를 제거했다. 아래 주변 사건 수는 탐색 맥락이며 실제 작업 겹침과 다르다. 최종 셀에서 수리 종료 후 시간과 TC-Z3~Z6의 초기 승온을 14 LOT 전체에 적용하고 관측 근거를 기록한다.

In [ ]:
# 강종만으로 더미 적합성을 부여하지 않는다. 최종 셀에서 시간·승온 근거를 대조한다.
lot_wld_context['DUMMY_GRADE_GROUP']='강종만으로 판단하지 않음'

## LOT 주변 정비·교체 시간 창

감지 건수는 감지 시각 기준 ±24/72시간, 부품은 연결 작업 완료 시각 기준 ±24/72시간이다.
실제 작업 겹침은 그림 1 상세와 새 그림 5의 착수~완료 구간으로 확인한다.
정기수리는 종료일 다음 날 00:00 경계를 사용해 종료일 전체를 포함한다.

In [ ]:
def count_events_around(center, df, col, hours):
    return int(
        (
            (df[col] >= center - pd.Timedelta(hours=hours))
            & (df[col] <= center + pd.Timedelta(hours=hours))
        ).sum()
    )

ctx = lot_wld_context.copy()

for hours in [24, 72]:
    ctx[f"BM_PM{hours}H"] = ctx["LOT_START"].apply(
        lambda t: count_events_around(
            t, cal_maint[cal_maint["MNT_TYPE"] == "BM"], "DETC_DT", hours
        )
    )
    ctx[f"CBM_PM{hours}H"] = ctx["LOT_START"].apply(
        lambda t: count_events_around(
            t, cal_maint[cal_maint["MNT_TYPE"] == "CBM"], "DETC_DT", hours
        )
    )
    ctx[f"TBM_PM{hours}H"] = ctx["LOT_START"].apply(
        lambda t: count_events_around(
            t, cal_maint[cal_maint["MNT_TYPE"] == "TBM"], "DETC_DT", hours
        )
    )
    ctx[f"PART_PM{hours}H"] = ctx["LOT_START"].apply(
        lambda t: count_events_around(
            t, tl_parts, "END_DT", hours
        )
    )

**결과 — LOT 주변 감지·교체 기록**

| 행 | LOT_NO | GRADE | LOT_START | LOT_END | WLD_INSIDE_COIL | WLD_PM12H_COIL | WLD_PM24H_COIL | PREV_WLD_MIN | NEXT_WLD_MIN | DUMMY_GRADE_GROUP | BM_PM24H | CBM_PM24H | TBM_PM24H | PART_PM24H | BM_PM72H | CBM_PM72H | TBM_PM72H | PART_PM72H |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 240001 | SGCC | 2024-01-02 03:36:00 | 2024-01-02 15:04:30 | 0 | 0 | 0 | NaN | 1835.753017 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 1 | 240024 | SPFH590 | 2024-03-07 21:22:00 | 2024-03-08 06:29:48 | 0 | 0 | 0 | 62534.498750 | 9390.863550 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| 2 | 240037 | SPCE | 2024-04-13 23:16:00 | 2024-04-14 09:27:19 | 0 | 0 | 0 | 6103.770533 | 2040.385250 | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 |
| 3 | 240038 | SPCEN | 2024-04-16 11:35:00 | 2024-04-16 13:39:56 | 0 | 12 | 51 | 16.607900 | 22941.976433 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 |
| 4 | 240044 | SPHC | 2024-05-03 18:50:00 | 2024-05-03 20:04:38 | 0 | 0 | 79 | 1489.654250 | 859.625867 | 강종만으로 판단하지 않음 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 0 |
| 5 | 240058 | DP590 | 2024-06-09 05:26:00 | 2024-06-09 09:02:53 | 0 | 80 | 87 | 2138.600133 | 233.324733 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 2 | 2 |
| 6 | 240061 | SPHC | 2024-06-15 01:04:00 | 2024-06-15 04:49:23 | 0 | 36 | 73 | 7633.168900 | 489.855283 | 강종만으로 판단하지 않음 | 0 | 0 | 1 | 0 | 0 | 0 | 4 | 0 |
| 7 | 240069 | SPHC | 2024-07-07 16:57:00 | 2024-07-08 06:29:34 | 0 | 0 | 0 | 10031.774317 | 14113.022950 | 강종만으로 판단하지 않음 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 1 |
| 8 | 240091 | SPHC | 2024-09-08 22:03:00 | 2024-09-09 08:44:12 | 0 | 0 | 0 | 30532.933817 | 5010.901267 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 |
| 9 | 240108 | SPFH590 | 2024-10-18 04:05:00 | 2024-10-18 11:50:53 | 12 | 41 | 41 | 7837.310133 | 306.484400 | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 |
| 10 | 240114 | SPCEN | 2024-11-04 15:40:00 | 2024-11-04 20:59:02 | 11 | 30 | 30 | 10.808233 | 0.408033 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 2 | 0 |
| 11 | 240117 | SPCD | 2024-11-13 07:08:00 | 2024-11-13 14:32:19 | 31 | 48 | 95 | 881.902983 | 116.029450 | 강종만으로 판단하지 않음 | 0 | 1 | 1 | 0 | 0 | 1 | 1 | 0 |
| 12 | 240121 | SPCE | 2024-11-24 07:31:00 | 2024-11-24 18:22:12 | 28 | 28 | 28 | 8050.463800 | 93.970550 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 |
| 13 | 240133 | SPCE | 2024-12-28 14:57:00 | 2024-12-28 16:44:24 | 0 | 0 | 0 | 2731.728150 | NaN | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 1 | 0 | 1 | 0 | 1 |

In [ ]:
# 정기수리 경계 플래그
ctx["REPAIR_BEFORE_72H"] = False
ctx["REPAIR_AFTER_72H"] = False

for i, r in ctx.iterrows():
    t = r["LOT_START"]

    for _, rp in cr1_repair.iterrows():
        if rp["STRT_DT"] - pd.Timedelta(hours=72) <= t < rp["STRT_DT"]:
            ctx.loc[i, "REPAIR_BEFORE_72H"] = True

        if rp["END_EXCLUSIVE"] <= t <= rp["END_EXCLUSIVE"] + pd.Timedelta(hours=72):
            ctx.loc[i, "REPAIR_AFTER_72H"] = True

**결과 — 계획수리 경계를 포함한 LOT 맥락**

| 행 | LOT_NO | GRADE | LOT_START | LOT_END | WLD_INSIDE_COIL | WLD_PM12H_COIL | WLD_PM24H_COIL | PREV_WLD_MIN | NEXT_WLD_MIN | DUMMY_GRADE_GROUP | BM_PM24H | CBM_PM24H | TBM_PM24H | PART_PM24H | BM_PM72H | CBM_PM72H | TBM_PM72H | PART_PM72H | REPAIR_BEFORE_72H | REPAIR_AFTER_72H |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 240001 | SGCC | 2024-01-02 03:36:00 | 2024-01-02 15:04:30 | 0 | 0 | 0 | NaN | 1835.753017 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | False | False |
| 1 | 240024 | SPFH590 | 2024-03-07 21:22:00 | 2024-03-08 06:29:48 | 0 | 0 | 0 | 62534.498750 | 9390.863550 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | False | False |
| 2 | 240037 | SPCE | 2024-04-13 23:16:00 | 2024-04-14 09:27:19 | 0 | 0 | 0 | 6103.770533 | 2040.385250 | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | False | False |
| 3 | 240038 | SPCEN | 2024-04-16 11:35:00 | 2024-04-16 13:39:56 | 0 | 12 | 51 | 16.607900 | 22941.976433 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | False | False |
| 4 | 240044 | SPHC | 2024-05-03 18:50:00 | 2024-05-03 20:04:38 | 0 | 0 | 79 | 1489.654250 | 859.625867 | 강종만으로 판단하지 않음 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | False | False |
| 5 | 240058 | DP590 | 2024-06-09 05:26:00 | 2024-06-09 09:02:53 | 0 | 80 | 87 | 2138.600133 | 233.324733 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 2 | 2 | True | False |
| 6 | 240061 | SPHC | 2024-06-15 01:04:00 | 2024-06-15 04:49:23 | 0 | 36 | 73 | 7633.168900 | 489.855283 | 강종만으로 판단하지 않음 | 0 | 0 | 1 | 0 | 0 | 0 | 4 | 0 | False | True |
| 7 | 240069 | SPHC | 2024-07-07 16:57:00 | 2024-07-08 06:29:34 | 0 | 0 | 0 | 10031.774317 | 14113.022950 | 강종만으로 판단하지 않음 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 1 | False | False |
| 8 | 240091 | SPHC | 2024-09-08 22:03:00 | 2024-09-09 08:44:12 | 0 | 0 | 0 | 30532.933817 | 5010.901267 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | False | False |
| 9 | 240108 | SPFH590 | 2024-10-18 04:05:00 | 2024-10-18 11:50:53 | 12 | 41 | 41 | 7837.310133 | 306.484400 | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | False | False |
| 10 | 240114 | SPCEN | 2024-11-04 15:40:00 | 2024-11-04 20:59:02 | 11 | 30 | 30 | 10.808233 | 0.408033 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 0 | 2 | 0 | True | False |
| 11 | 240117 | SPCD | 2024-11-13 07:08:00 | 2024-11-13 14:32:19 | 31 | 48 | 95 | 881.902983 | 116.029450 | 강종만으로 판단하지 않음 | 0 | 1 | 1 | 0 | 0 | 1 | 1 | 0 | False | True |
| 12 | 240121 | SPCE | 2024-11-24 07:31:00 | 2024-11-24 18:22:12 | 28 | 28 | 28 | 8050.463800 | 93.970550 | 강종만으로 판단하지 않음 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | False | False |
| 13 | 240133 | SPCE | 2024-12-28 14:57:00 | 2024-12-28 16:44:24 | 0 | 0 | 0 | 2731.728150 | NaN | 강종만으로 판단하지 않음 | 0 | 1 | 0 | 1 | 0 | 1 | 0 | 1 | False | False |

# 11. BM / CBM / TBM REMARK 확인

정비 건수만 세지 않고 내용까지 본다.

### BM
사후정비 이유

### CBM
마모/열화/상태기반 이상 여부

### TBM
계획점검인지 실제 조정·작업인지

특히 Focus LOT 주변의 `FAIL_MODE`, `REMARK`, `PART_CHG`를 함께 본다.

In [ ]:
maint_view = cal_maint[
    [
        "WO_NO", "DETC_DT", "STRT_DT", "END_DT",
        "MNT_TYPE", "FAIL_MODE", "PART_CHG", "REMARK"
    ]
].sort_values("DETC_DT")

**결과 — 소둔로 정비 원문**

| 행 | WO_NO | DETC_DT | STRT_DT | END_DT | MNT_TYPE | FAIL_MODE | PART_CHG | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 59 | WO-2024-00060 | 2024-01-11 15:19:00 | 2024-01-11 18:43:00 | 2024-01-11 20:32:00 | TBM | NaN | N | NaN |
| 210 | WO-2024-00211 | 2024-02-09 15:31:00 | 2024-02-09 15:52:00 | 2024-02-09 23:22:00 | BM | NaN | N | 1호기 재발, 원인 미상 |
| 219 | WO-2024-00220 | 2024-02-11 07:13:00 | 2024-02-11 07:14:00 | 2024-02-11 10:32:00 | TBM | NaN | N | 정기점검검 - 이상없음 |
| 377 | WO-2024-00378 | 2024-03-10 15:10:00 | 2024-03-10 15:46:00 | 2024-03-10 18:36:00 | TBM | NaN | N | 1호기 재발, 정기점검 - 이상없음 |
| 384 | WO-2024-00385 | 2024-03-11 15:01:00 | 2024-03-18 23:13:00 | 2024-03-19 04:59:00 | CBM | WER | N | 마모 진행 |
| 507 | WO-2024-00508 | 2024-03-24 14:59:00 | 2024-03-24 15:56:00 | 2024-03-24 23:36:00 | BM | ELE | Y | 인버터 알람 |
| 596 | WO-2024-00597 | 2024-04-09 14:55:00 | 2024-04-09 17:00:00 | 2024-04-09 19:09:00 | TBM | NaN | Y | 정기점검검 - 이상없음 |
| 626 | WO-2024-00627 | 2024-04-14 15:04:00 | 2024-04-23 15:27:00 | 2024-04-23 18:29:00 | CBM | NaN | N | 확인 중 |
| 657 | WO-2024-00658 | 2024-04-20 23:47:00 | 2024-04-23 20:04:00 | 2024-04-23 21:44:00 | CBM | MAL | N | 설비 트립 |
| 667 | WO-2024-00668 | 2024-04-22 22:50:00 | 2024-04-24 06:55:00 | 2024-04-24 16:38:00 | CBM | ETC | N | 현상 재현 안됨 |
| 716 | WO-2024-00717 | 2024-05-03 23:46:00 | 2024-05-04 04:50:00 | 2024-05-04 08:15:00 | BM | MAL | N | 동작 불량 |
| 746 | WO-2024-00747 | 2024-05-08 07:53:00 | 2024-05-08 11:38:00 | 2024-05-08 12:30:00 | TBM | ELE | N | NaN |
| 775 | WO-2024-00776 | 2024-05-14 07:17:00 | 2024-05-19 12:14:00 | 2024-05-19 16:05:00 | CBM | ETC | Y | 재점검 필요 |
| 832 | WO-2024-00833 | 2024-05-25 15:55:00 | 2024-05-25 19:06:00 | 2024-05-26 05:01:00 | BM | ETC | N | 1호기 재발, 확인 중 |
| 915 | WO-2024-00916 | 2024-06-10 06:23:00 | 2024-06-10 08:45:00 | 2024-06-10 10:48:00 | TBM | NaN | Y | NaN |
| 930 | WO-2024-00931 | 2024-06-11 14:51:00 | 2024-06-11 18:40:00 | 2024-06-11 23:37:00 | TBM | WER | N | 마모량 초과 |
| 939 | WO-2024-00940 | 2024-06-12 06:09:00 | 2024-06-12 09:51:00 | 2024-06-12 15:23:00 | TBM | NaN | N | 수리기간 중 점검 |
| 951 | WO-2024-00952 | 2024-06-12 23:51:00 | 2024-06-13 02:51:00 | 2024-06-13 05:16:00 | TBM | NaN | N | 정기수리 부속작업 |
| 959 | WO-2024-00960 | 2024-06-13 14:22:00 | 2024-06-13 18:19:00 | 2024-06-13 22:29:00 | TBM | NaN | N | 1호기 재발, SD작업 수행 |
| 971 | WO-2024-00972 | 2024-06-14 14:09:00 | 2024-06-14 15:35:00 | 2024-06-14 18:03:00 | TBM | NaN | N | 1호기 재발, SD작업 수행 |
| 1093 | WO-2024-01094 | 2024-07-08 08:49:00 | 2024-07-08 11:11:00 | 2024-07-08 19:04:00 | BM | ELE | Y | 1호기 재발, 절연 저하 |
| 1110 | WO-2024-01111 | 2024-07-10 22:24:00 | 2024-07-10 23:15:00 | 2024-07-11 00:05:00 | TBM | NaN | N | NaN |
| 1177 | WO-2024-01178 | 2024-07-24 23:33:00 | 2024-07-25 01:19:00 | 2024-07-25 10:55:00 | BM | ELE | Y | NaN |
| 1258 | WO-2024-01259 | 2024-08-09 22:35:00 | 2024-08-10 01:48:00 | 2024-08-10 06:20:00 | TBM | NaN | N | 점검 이상없음 |
| 1265 | WO-2024-01266 | 2024-08-11 09:57:00 | 2024-08-11 14:06:00 | 2024-08-11 17:07:00 | BM | MAL | N | 오작동 발생 |
| 1280 | WO-2024-01281 | 2024-08-14 06:43:00 | 2024-08-14 09:00:00 | 2024-08-14 12:27:00 | BM | WER | N | 마모량 초과 |
| 1345 | WO-2024-01346 | 2024-08-27 07:37:00 | 2024-08-27 11:03:00 | 2024-08-28 04:27:00 | BM | ELE | Y | 1호기 재발, 모터 과열 |
| 1418 | WO-2024-01419 | 2024-09-11 12:55:00 | 2024-09-11 17:01:00 | 2024-09-11 20:05:00 | BM | ELE | N | 절연 저하 |
| 1420 | WO-2024-01421 | 2024-09-11 23:31:00 | 2024-09-12 01:37:00 | 2024-09-12 06:12:00 | TBM | NaN | N | 정기점검 - 이상없음 |
| 1466 | WO-2024-01467 | 2024-09-17 07:21:00 | 2024-09-23 21:35:00 | 2024-09-24 05:26:00 | CBM | NaN | N | 재점검 필요 |
| 1535 | WO-2024-01536 | 2024-09-22 15:25:00 | 2024-09-22 19:36:00 | 2024-09-22 20:42:00 | BM | ETC | N | 1호기 재발, 확인 중 |
| 1633 | WO-2024-01634 | 2024-10-10 14:37:00 | 2024-10-10 16:59:00 | 2024-10-10 18:18:00 | TBM | NaN | N | 점검 이상없음 |
| 1645 | WO-2024-01646 | 2024-10-12 15:06:00 | 2024-10-12 16:25:00 | 2024-10-12 18:44:00 | BM | NaN | N | 현상 재현 안됨 |
| 1677 | WO-2024-01678 | 2024-10-18 15:50:00 | 2024-10-26 23:53:00 | 2024-10-27 02:42:00 | CBM | ETC | N | 원인 미상 |
| 1774 | WO-2024-01775 | 2024-11-06 07:49:00 | 2024-11-06 10:40:00 | 2024-11-06 12:23:00 | TBM | NaN | N | 정기수리 실시 |
| 1793 | WO-2024-01794 | 2024-11-07 09:20:00 | 2024-11-07 12:53:00 | 2024-11-07 14:14:00 | TBM | NaN | N | 정기수리 부속작업 |
| 1822 | WO-2024-01823 | 2024-11-09 06:05:00 | 2024-11-09 09:58:00 | 2024-11-09 13:02:00 | TBM | NaN | N | 1호기 재발, 정기수리 실시 |
| 1825 | WO-2024-01826 | 2024-11-09 07:39:00 | 2024-11-09 11:15:00 | 2024-11-09 18:20:00 | TBM | NaN | N | 수리기간 중 점검 |
| 1833 | WO-2024-01834 | 2024-11-10 07:00:00 | 2024-11-10 10:40:00 | 2024-11-10 12:52:00 | TBM | NaN | N | 정기수리 부속작업 |
| 1853 | WO-2024-01854 | 2024-11-12 15:00:00 | 2024-11-13 08:54:00 | 2024-11-13 12:44:00 | CBM | WER | N | 마모량 초과 |
| 1858 | WO-2024-01859 | 2024-11-13 07:13:00 | 2024-11-13 07:40:00 | 2024-11-13 09:35:00 | TBM | NaN | N | 계획 점검 수행 |
| 1921 | WO-2024-01922 | 2024-11-25 14:19:00 | 2024-11-30 04:19:00 | 2024-11-30 07:12:00 | CBM | MAL | N | 1호기 재발, 설비 트립 |
| 1998 | WO-2024-01999 | 2024-12-09 23:43:00 | 2024-12-10 01:50:00 | 2024-12-10 05:55:00 | TBM | NaN | N | NaN |
| 2104 | WO-2024-02105 | 2024-12-28 15:54:00 | 2024-12-28 16:54:00 | 2024-12-28 23:33:00 | CBM | WER | Y | 마모량 초과 |

In [ ]:
# Focus LOT 주변 정비 상세
FOCUS_HOURS = 72

focus_rows = []

for _, lot in lot_info.iterrows():
    t = lot["LOT_START"]

    d = cal_maint[
        (cal_maint["DETC_DT"] >= t - pd.Timedelta(hours=FOCUS_HOURS))
        & (cal_maint["DETC_DT"] <= t + pd.Timedelta(hours=FOCUS_HOURS))
    ].copy()

    for _, m in d.iterrows():
        focus_rows.append({
            "LOT_NO": lot["LOT_NO"],
            "GRADE": lot["GRADE"],
            "LOT_START": t,
            "DELTA_HOUR": (m["DETC_DT"] - t).total_seconds() / 3600,
            "WO_NO": m["WO_NO"],
            "MNT_TYPE": m["MNT_TYPE"],
            "FAIL_MODE": m["FAIL_MODE"],
            "PART_CHG": m["PART_CHG"],
            "REMARK": m["REMARK"]
        })

focus_maint = pd.DataFrame(focus_rows)

if len(focus_maint):
    pass
else:
    print("±72시간 내 CAL01 정비 없음")

**결과 — LOT 주변 정비 상세**

| 행 | LOT_NO | GRADE | LOT_START | DELTA_HOUR | WO_NO | MNT_TYPE | FAIL_MODE | PART_CHG | REMARK |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 240024 | SPFH590 | 2024-03-07 21:22:00 | 65.800000 | WO-2024-00378 | TBM | NaN | N | 1호기 재발, 정기점검 - 이상없음 |
| 1 | 240037 | SPCE | 2024-04-13 23:16:00 | 15.800000 | WO-2024-00627 | CBM | NaN | N | 확인 중 |
| 2 | 240038 | SPCEN | 2024-04-16 11:35:00 | -44.516667 | WO-2024-00627 | CBM | NaN | N | 확인 중 |
| 3 | 240044 | SPHC | 2024-05-03 18:50:00 | 4.933333 | WO-2024-00717 | BM | MAL | N | 동작 불량 |
| 4 | 240058 | DP590 | 2024-06-09 05:26:00 | 24.950000 | WO-2024-00916 | TBM | NaN | Y | NaN |
| 5 | 240058 | DP590 | 2024-06-09 05:26:00 | 57.416667 | WO-2024-00931 | TBM | WER | N | 마모량 초과 |
| 6 | 240061 | SPHC | 2024-06-15 01:04:00 | -66.916667 | WO-2024-00940 | TBM | NaN | N | 수리기간 중 점검 |
| 7 | 240061 | SPHC | 2024-06-15 01:04:00 | -49.216667 | WO-2024-00952 | TBM | NaN | N | 정기수리 부속작업 |
| 8 | 240061 | SPHC | 2024-06-15 01:04:00 | -34.700000 | WO-2024-00960 | TBM | NaN | N | 1호기 재발, SD작업 수행 |
| 9 | 240061 | SPHC | 2024-06-15 01:04:00 | -10.916667 | WO-2024-00972 | TBM | NaN | N | 1호기 재발, SD작업 수행 |
| 10 | 240069 | SPHC | 2024-07-07 16:57:00 | 15.866667 | WO-2024-01094 | BM | ELE | Y | 1호기 재발, 절연 저하 |
| 11 | 240091 | SPHC | 2024-09-08 22:03:00 | 62.866667 | WO-2024-01419 | BM | ELE | N | 절연 저하 |
| 12 | 240108 | SPFH590 | 2024-10-18 04:05:00 | 11.750000 | WO-2024-01678 | CBM | ETC | N | 원인 미상 |
| 13 | 240114 | SPCEN | 2024-11-04 15:40:00 | 40.150000 | WO-2024-01775 | TBM | NaN | N | 정기수리 실시 |
| 14 | 240114 | SPCEN | 2024-11-04 15:40:00 | 65.666667 | WO-2024-01794 | TBM | NaN | N | 정기수리 부속작업 |
| 15 | 240117 | SPCD | 2024-11-13 07:08:00 | -16.133333 | WO-2024-01854 | CBM | WER | N | 마모량 초과 |
| 16 | 240117 | SPCD | 2024-11-13 07:08:00 | 0.083333 | WO-2024-01859 | TBM | NaN | N | 계획 점검 수행 |
| 17 | 240121 | SPCE | 2024-11-24 07:31:00 | 30.800000 | WO-2024-01922 | CBM | MAL | N | 1호기 재발, 설비 트립 |
| 18 | 240133 | SPCE | 2024-12-28 14:57:00 | 0.950000 | WO-2024-02105 | CBM | WER | Y | 마모량 초과 |

# 12. 반복사례와 반례

가설을 세웠다면 반드시 반례를 찾는다.

예:

> `정기수리 후 더미후보 강종에서 ramp-up이 나타난다`

라면 다음도 찾아야 한다.

- 수리 후인데 ramp-up 없는 LOT
- SPHC인데 안정적인 LOT
- SPHC가 아닌데 ramp-up 있는 LOT
- 정비 있는데 센서 변화가 작은 LOT
- 센서 변화가 큰데 정비가 없는 LOT

이 단계가 있어야 단순 사례 나열이 아니라 검증이 된다.

In [ ]:
# 분석가가 검토할 비교표
review_table = (
    ctx.merge(
        lot_score,
        on=["LOT_NO", "GRADE"],
        how="left"
    )
)

**결과 — 반복 사례·반례 검토표**

| 행 | LOT_NO | GRADE | DUMMY_GRADE_GROUP | REPAIR_BEFORE_72H | REPAIR_AFTER_72H | WLD_INSIDE_COIL | WLD_PM12H_COIL | BM_PM72H | CBM_PM72H | TBM_PM72H | PART_PM72H | LEVEL_Q75 | TREND_Q75 | VAR_Q75 | MAX_SENSOR_RZ |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 6 | 240061 | SPHC | 강종만으로 판단하지 않음 | False | True | 0 | 36 | 0 | 0 | 4 | 0 | 111.357817 | 30.278625 | 39.604714 | 1504.596173 |
| 2 | 240037 | SPCE | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 0 | 1 | 0 | 0 | 1.852004 | 2.985614 | 3.010532 | 643.464185 |
| 4 | 240044 | SPHC | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 1 | 0 | 0 | 0 | 1.640903 | 2.680222 | 3.282569 | 192.050920 |
| 12 | 240121 | SPCE | 강종만으로 판단하지 않음 | False | False | 28 | 28 | 0 | 1 | 0 | 0 | 2.666482 | 1.461879 | 2.377438 | 77.632485 |
| 8 | 240091 | SPHC | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 1 | 0 | 0 | 0 | 2.394225 | 0.929158 | 3.944232 | 47.126376 |
| 7 | 240069 | SPHC | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 1 | 0 | 0 | 1 | 1.964711 | 1.329651 | 2.590493 | 35.410765 |
| 13 | 240133 | SPCE | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 0 | 1 | 0 | 1 | 3.216802 | 7.801311 | 2.207767 | 19.283582 |
| 0 | 240001 | SGCC | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 0 | 0 | 0 | 0 | NaN | NaN | NaN | NaN |
| 1 | 240024 | SPFH590 | 강종만으로 판단하지 않음 | False | False | 0 | 0 | 0 | 0 | 1 | 0 | NaN | NaN | NaN | NaN |
| 3 | 240038 | SPCEN | 강종만으로 판단하지 않음 | False | False | 0 | 12 | 0 | 1 | 0 | 0 | NaN | NaN | NaN | NaN |
| 5 | 240058 | DP590 | 강종만으로 판단하지 않음 | True | False | 0 | 80 | 0 | 0 | 2 | 2 | NaN | NaN | NaN | NaN |
| 9 | 240108 | SPFH590 | 강종만으로 판단하지 않음 | False | False | 12 | 41 | 0 | 1 | 0 | 0 | NaN | NaN | NaN | NaN |
| 10 | 240114 | SPCEN | 강종만으로 판단하지 않음 | True | False | 11 | 30 | 0 | 0 | 2 | 0 | NaN | NaN | NaN | NaN |
| 11 | 240117 | SPCD | 강종만으로 판단하지 않음 | False | True | 31 | 48 | 0 | 1 | 1 | 0 | NaN | NaN | NaN | NaN |

# 13. 공통 경과시간 안의 연속 Episode 후보

각 강종에서 모든 LOT에 관측이 있는 공통 경과 분까지만 비교한다.
기준은 자기 LOT를 제외한 같은 경과 분의 다른 LOT 2개 이상이다.
5분 중 3분 이상 |RZ|≥2인 연속 구간을 탐색하되,
기준이 없는 분과 서로 떨어진 구간을 이어 붙이지 않는다.
시작·종료 분과 실제 날짜, 구간 길이를 저장한다. 탐색 기준이며 관리 한계나 고장 정답은 아니다.

In [ ]:
# 실제 경과 분의 연속 구간만 Episode로 묶는다. 서로 떨어진 구간은 합치지 않는다.
EPISODE_THRESHOLD = 2.0
WINDOW = 5
MIN_HIT = 3
episode_rows = []
for grade, glots in lot_info.dropna(subset=["GRADE"]).groupby("GRADE"):
    lots = glots["LOT_NO"].astype(int).tolist()
    if len(lots) < 3:
        continue
    common = rt_common_minutes[grade]
    for sensor in [s for s in ANALYSIS_SIGNALS if s.startswith("TC-")]:
        matrix = pd.concat([rt_profiles[lot][(sensor, "median")].rename(lot) for lot in lots], axis=1).reindex(range(common))
        for lot in lots:
            peers = matrix.drop(columns=lot)
            centers = peers.median(axis=1).where(peers.count(axis=1) >= 2)
            scale = peers.apply(robust_scale, axis=1).where(peers.count(axis=1) >= 2)
            rz = ((matrix[lot] - centers) / scale).replace([np.inf, -np.inf], np.nan)
            hit = rz.abs().ge(EPISODE_THRESHOLD).where(rz.notna())
            persistent = (hit.rolling(WINDOW, min_periods=WINDOW).sum().ge(MIN_HIT) & rz.notna())
            labels = persistent.ne(persistent.shift(fill_value=False)).cumsum()
            for _, block in persistent[persistent].groupby(labels[persistent]):
                start_min, end_min = int(block.index.min()), int(block.index.max()) + 1
                info = lot_info.loc[lot_info["LOT_NO"].eq(lot)].iloc[0]
                episode_rows.append({"GRADE": grade, "LOT_NO": lot, "SENSOR": sensor,
                    "START_MIN": start_min, "END_MIN": end_min,
                    "START_DT": info.LOT_START + pd.Timedelta(minutes=start_min),
                    "END_DT": info.LOT_START + pd.Timedelta(minutes=end_min),
                    "DURATION_MIN": end_min - start_min,
                    "MAX_ABS_RZ": rz.loc[block.index].abs().max(), "MIN_PEER_LOTS": 2})
episode_df = pd.DataFrame(episode_rows)
if len(episode_df):
    pass
else:
    print("현재 탐색기준에서 연속 Episode 후보 없음")

**결과 — 연속 Episode 후보**

| 행 | GRADE | LOT_NO | SENSOR | START_MIN | END_MIN | START_DT | END_DT | DURATION_MIN | MAX_ABS_RZ | MIN_PEER_LOTS |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 115 | SPHC | 240061 | TC-Z5 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 23309.051667 | 2 |
| 108 | SPHC | 240061 | TC-Z4 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 13664.508297 | 2 |
| 121 | SPHC | 240061 | TC-Z6 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 8248.347498 | 2 |
| 29 | SPCE | 240133 | TC-Z3 | 91 | 97 | 2024-12-28 16:28:00 | 2024-12-28 16:34:00 | 6 | 6386.753002 | 2 |
| 127 | SPHC | 240061 | TC-Z7 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 5109.604748 | 2 |
| 73 | SPCE | 240133 | TC-OUT1 | 51 | 69 | 2024-12-28 15:48:00 | 2024-12-28 16:06:00 | 18 | 4879.555222 | 2 |
| 128 | SPHC | 240091 | TC-Z7 | 4 | 74 | 2024-09-08 22:07:00 | 2024-09-08 23:17:00 | 70 | 2184.675570 | 2 |
| 26 | SPCE | 240133 | TC-Z3 | 34 | 48 | 2024-12-28 15:31:00 | 2024-12-28 15:45:00 | 14 | 1713.881020 | 2 |
| 137 | SPHC | 240069 | TC-Z9 | 45 | 74 | 2024-07-07 17:42:00 | 2024-07-07 18:11:00 | 29 | 1125.886955 | 2 |
| 65 | SPCE | 240121 | TC-Z7 | 24 | 107 | 2024-11-24 07:55:00 | 2024-11-24 09:18:00 | 83 | 1098.239579 | 2 |
| 100 | SPHC | 240061 | TC-Z3 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 1064.450186 | 2 |
| 129 | SPHC | 240061 | TC-Z8 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 993.322541 | 2 |
| 133 | SPHC | 240044 | TC-Z9 | 39 | 74 | 2024-05-03 19:29:00 | 2024-05-03 20:04:00 | 35 | 964.009173 | 2 |
| 76 | SPCE | 240037 | TC-OUT2 | 4 | 39 | 2024-04-13 23:20:00 | 2024-04-13 23:55:00 | 35 | 842.528891 | 2 |
| 159 | SPHC | 240044 | TC-OUT2 | 40 | 74 | 2024-05-03 19:30:00 | 2024-05-03 20:04:00 | 34 | 670.829238 | 2 |
| 64 | SPCE | 240121 | TC-Z7 | 4 | 19 | 2024-11-24 07:35:00 | 2024-11-24 07:50:00 | 15 | 643.239354 | 2 |
| 130 | SPHC | 240091 | TC-Z8 | 4 | 74 | 2024-09-08 22:07:00 | 2024-09-08 23:17:00 | 70 | 539.929853 | 2 |
| 86 | SPHC | 240061 | TC-Z1 | 4 | 53 | 2024-06-15 01:08:00 | 2024-06-15 01:57:00 | 49 | 507.054243 | 2 |
| 134 | SPHC | 240061 | TC-Z9 | 10 | 31 | 2024-06-15 01:14:00 | 2024-06-15 01:35:00 | 21 | 500.594778 | 2 |
| 71 | SPCE | 240133 | TC-OUT1 | 4 | 26 | 2024-12-28 15:01:00 | 2024-12-28 15:23:00 | 22 | 438.655065 | 2 |
| 161 | SPHC | 240061 | TC-OUT2 | 52 | 74 | 2024-06-15 01:56:00 | 2024-06-15 02:18:00 | 22 | 428.494344 | 2 |
| 136 | SPHC | 240069 | TC-Z9 | 11 | 17 | 2024-07-07 17:08:00 | 2024-07-07 17:14:00 | 6 | 353.555793 | 2 |
| 59 | SPCE | 240133 | TC-Z6 | 18 | 38 | 2024-12-28 15:15:00 | 2024-12-28 15:35:00 | 20 | 310.265749 | 2 |
| 25 | SPCE | 240133 | TC-Z3 | 7 | 31 | 2024-12-28 15:04:00 | 2024-12-28 15:28:00 | 24 | 241.467692 | 2 |
| 1 | SPCE | 240037 | TC-Z1 | 40 | 62 | 2024-04-13 23:56:00 | 2024-04-14 00:18:00 | 22 | 196.546607 | 2 |
| 105 | SPHC | 240091 | TC-Z3 | 63 | 69 | 2024-09-08 23:06:00 | 2024-09-08 23:12:00 | 6 | 185.372544 | 2 |
| 94 | SPHC | 240061 | TC-Z2 | 4 | 74 | 2024-06-15 01:08:00 | 2024-06-15 02:18:00 | 70 | 184.103859 | 2 |
| 93 | SPHC | 240044 | TC-Z2 | 53 | 74 | 2024-05-03 19:43:00 | 2024-05-03 20:04:00 | 21 | 148.803038 | 2 |
| 10 | SPCE | 240037 | TC-Z2 | 48 | 61 | 2024-04-14 00:04:00 | 2024-04-14 00:17:00 | 13 | 146.364495 | 2 |
| 72 | SPCE | 240133 | TC-OUT1 | 29 | 48 | 2024-12-28 15:26:00 | 2024-12-28 15:45:00 | 19 | 145.536782 | 2 |

## 최종 검토 결과와 시각화

관측 범위 → 시간 겹침 → 센서 원본 → 동시간 비교 순서로 얻은 단서를 합친다. 실제 더미코일은 확정하지 않고 승온·더미 운전을 확인할 LOT와 중점 공정·구간을 구체화한다. 14 LOT 모두에 검토 결과를 채운다.

In [ ]:
# %% 최종 검토: 강종명만으로 더미코일을 고르지 않고 시간·관측 근거를 남긴다.
startup_rows=[]
for info in lot_info.itertuples():
    raw=temp_flow.loc[temp_flow.LOT_NO.eq(info.LOT_NO)]
    early=raw.loc[raw.ELAPSED_MIN.lt(5),['TC-Z3','TC-Z4','TC-Z5','TC-Z6']].median()
    later=raw.loc[raw.ELAPSED_MIN.between(80,100),early.index].median()
    previous=tl_plans.loc[tl_plans.END_EXCLUSIVE<=info.LOT_START].sort_values('END_EXCLUSIVE')
    gap=(info.LOT_START-previous.END_EXCLUSIVE.iloc[-1]).total_seconds()/60 if len(previous) else np.nan
    delta=later-early
    startup_rows.append(dict(LOT_NO=int(info.LOT_NO),GRADE=info.GRADE,
        TC3_START_MEDIAN=early['TC-Z3'],TC3_LATER_MEDIAN=later['TC-Z3'],TC3_RISE=delta['TC-Z3'],
        FOUR_ZONE_RISE_MIN=delta.min(),FOUR_ZONE_RISE_MAX=delta.max(),
        N_LATER_ROWS=int(raw.ELAPSED_MIN.between(80,100).sum()),POST_PLAN_MIN=gap,
        POST_PLAN_72H=bool(pd.notna(gap) and 0<=gap<=72*60)))
startup=pd.DataFrame(startup_rows)
# 20℃는 이번 탐색의 큰 초기 승온 구분값이며 공정 합격 기준이 아니다.
startup['WARMUP_REVIEW']=startup.POST_PLAN_72H & startup.FOUR_ZONE_RISE_MIN.ge(20) & startup.N_LATER_ROWS.gt(0)
assert set(startup.loc[startup.WARMUP_REVIEW,'LOT_NO'])=={240061,240117}
decision={
 240061:('우선 검토','가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4]','0–100분','수리 후 64분 · 4대역 초기 승온','승온·더미 운전 확인 후보'),
 240117:('우선 검토','가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접','0–100분 및 07:40–12:44','초기 승온 + CAL 작업 2건 + 용접 31 COIL','승온·더미 운전 확인 후보'),
 240069:('신호 확인','가열·균열 [CP-Z4, TC-Z4, OP-Z4]','571.48–577.48분','CP 원본 0.909111 · 동시 TC/OP 대조','CP 급변 확인'),
 240037:('신호·주기 확인','가열·균열 [TC-Z4·OP-Z4·CP-Z4] / 과시효 [TC-Z7~TC-Z9]','CP 38.98–44.98분 / TC-Z9 전체','CP 원본 0.60676 · TC-Z9 약 197분 주기','CP 확인 · 주기 위상 비교'),
 240058:('전환 확인','가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4]','130분 이후~종료','수리 전 관측 · 후반 CP와 TC/OP 하강','종료 조건 전환 확인'),
 240133:('감지·교체 연결 확인','가열·균열 [CP-Z4] / 냉각 [TC-OUT1·TC-OUT2]','감지 57분 / 종료 9.6분 뒤 착수','CBM 02105 → 가스 분석계 셀 INS-006 3개 · 작업 후 온도 없음','감지 전후 신호·WO 교체 확인'),
 240121:('공동 관측 참고','냉각 [TC-OUT1·TC-OUT2] / 용접','처음 107분 및 LOT 전체','냉각 변동폭 비교 · 용접 28 COIL','위상·조건 비교'),
 240108:('공동 관측 참고','가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접','TC-Z3 449–454분 / LOT 전체','같은 강종 240024 비교 · 용접 12 COIL','온도·용접 시간 대조'),
 240114:('공동 관측 참고','가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접','TC-Z3 31–36분 / LOT 전체','같은 강종 240038 비교 · 용접 11 COIL','온도·용접 시간 대조'),
 240091:('같은 강종 비교','가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4]','전체 경과 분','SPHC 반복 비교 · 소둔로 작업 겹침 없음','같은 강종 참고'),
 240044:('비교 기준','세정 [CLN-IN] / 건조 [TC-Z1·TC-Z2] / 가열·균열 [TC-Z3~TC-Z6] / 과시효 [TC-Z7~TC-Z9] / 냉각 [TC-OUT1·TC-OUT2]','전체 74.6분','SPHC 반복 비교 · 80–100분은 미관측','같은 강종 참고'),
 240024:('비교 기준','가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2]','240108과 공통 경과 분','SPFH590 반복 비교','같은 강종 참고'),
 240038:('비교 기준','가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2]','240114와 공통 경과 분','SPCEN 반복 비교','같은 강종 참고'),
 240001:('단독 관측 참고','가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4]','TC-Z3 596–601분','SGCC 1 LOT · 강종 내 반복 비교 불가','원본 변화 위치 보존')}
final_review=review_table.merge(startup.drop(columns='GRADE'),on='LOT_NO',validate='one_to_one').merge(
    audit_lots.drop(columns=['GRADE','START','END']),on='LOT_NO',validate='one_to_one')
for col,idx in [('PRIORITY',0),('FOCUS_PROCESS',1),('FOCUS_MIN_OR_CLOCK',2),('REVIEW_NOTE',3),('FINAL_CLASS',4)]:
    final_review[col]=final_review.LOT_NO.map(lambda lot:decision[int(lot)][idx])
final_review['DUMMY_GRADE_GROUP']=np.where(final_review.WARMUP_REVIEW,'승온·더미 운전 확인 후보','강종만으로 판단하지 않음')
final_review['DUMMY_CONFIRMED']=False
final_cols=['LOT_NO','GRADE','PRIORITY','FOCUS_PROCESS','FOCUS_MIN_OR_CLOCK','FINAL_CLASS','REVIEW_NOTE',
    'DUMMY_GRADE_GROUP','DUMMY_CONFIRMED','TC3_RISE','FOUR_ZONE_RISE_MIN','FOUR_ZONE_RISE_MAX','POST_PLAN_MIN',
    'CAL01_WORK','CAL01_DETECT','WELD_COIL','CAL01_PART_WO','DETECT_LINKED_PART_WO']

# 최종 결과 A: 후보를 고른 초기 온도 차이를 14 LOT 전부와 대조한다.
fig,axes=plt.subplots(1,2,figsize=(21,8),gridspec_kw={'width_ratios':[1,1.6]})
fig.subplots_adjust(left=.12,right=.97,top=.78,bottom=.13,wspace=.3)
ordered=startup.sort_values('TC3_RISE',ascending=False,na_position='last').reset_index(drop=True)
for i,r in enumerate(ordered.itertuples()):
    yy=len(ordered)-i-1;color=RT_RED if r.WARMUP_REVIEW else '#99b1bf'
    if pd.notna(r.TC3_RISE):
        axes[0].barh(yy,r.TC3_RISE,height=.55,color=color)
        axes[0].text(max(0,r.TC3_RISE)+.6,yy,f'{r.TC3_RISE:+.1f}',va='center',fontsize=9,color=color)
    else:axes[0].text(.6,yy,'80–100분 미관측',va='center',fontsize=9,color='#64748b')
axes[0].set_yticks(range(len(ordered)),[f'{r.LOT_NO} · {r.GRADE}' for r in ordered.itertuples()][::-1],fontsize=9)
axes[0].set_xlim(-3,50);axes[0].set_xlabel('TC-Z3: 80–100분 중앙값 - 첫 5분 중앙값 [℃]')
axes[0].set_title('14 LOT 전체에서 초기 승온을 비교',loc='left',fontsize=12,pad=14)
for lot in [240061,240117,240069,240091]:
    p=rt_profiles[lot].loc[:110]
    color={240061:RT_RED,240117:'#d16d23',240069:'#087eaa',240091:'#16856b'}[lot]
    axes[1].plot(p.index,p[('TC-Z3','median')],color=color,lw=1.6,label=f'LOT {lot}')
axes[1].axvspan(0,5,color=RT_RED,alpha=.08);axes[1].axvspan(80,100,color='#dff0f8')
axes[1].set_title('가열·균열 [TC-Z3] · 두 후보와 SPHC 참고 LOT',loc='left',fontsize=12,pad=14)
axes[1].set_ylabel('온도 [℃]');axes[1].set_xlabel('LOT 시작 후 경과시간 [분]')
axes[1].legend(loc='lower right',ncol=2,fontsize=10,frameon=True)
rt_point_box(axes[1],(2,float(rt_profiles[240061].loc[2,('TC-Z3','median')])),
    'LOT 240061 · 2분\nTC-Z3 초기 승온 · 운전 모드 확인')
for ax in axes:rt_style(ax)
rt_header(fig,'최종 후보 근거. 240061·240117의 초기 승온을 먼저 확인한다',
    '두 LOT의 TC-Z3~Z6 모두 약 37~42℃ 상승 · 더미코일 확정 0건')
rt_finish(fig,Path(FINAL_DIR)/'01_승온_더미운전_확인후보.png')

# 최종 결과 B: 소둔로 작업과 연결된 부품·관측의 연간 지도.
fig,ax=plt.subplots(figsize=(24,9.5))
fig.subplots_adjust(left=.10,right=.985,top=.80,bottom=.11)
for plan in tl_plans.itertuples():
    ax.fill_between([plan.STRT_DT,plan.END_EXCLUSIVE],[-.25]*2,[.25]*2,color='#dff0f8',zorder=0)
    ax.plot([plan.STRT_DT,plan.END_EXCLUSIVE],[0,0],lw=8,color='#3386a5',solid_capstyle='butt')
    ax.text(plan.STRT_DT,-.35,f'{plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}',ha='center',fontsize=9,color='#28667f')
for kind in ['BM','CBM','TBM']:
    subset=audit_work.loc[audit_work.MNT_TYPE.eq(kind)].sort_values('STRT_DT')
    offsets=tl_stack(list(zip(subset.STRT_DT,subset.END_DT)),min_days=.7)
    for task,dy in zip(subset.itertuples(),offsets):
        ax.plot([task.STRT_DT,task.END_DT],[3+dy]*2,lw=1.7,color=TL_COLORS[kind],alpha=.75)
        ax.scatter(task.STRT_DT+(task.END_DT-task.STRT_DT)/2,3+dy,marker='x',s=22,color=TL_COLORS[kind])
for lot in tl_lots.itertuples():
    ax.plot([lot.LOT_START,lot.LOT_END],[4,4],color=TL_COLORS['LOT'],lw=5)
    ax.scatter(lot.LOT_START,4,s=24,color=TL_COLORS['LOT'])
    if int(lot.LOT_NO) in [240061,240117,240133]:
        ax.scatter(lot.LOT_START,4,s=115,facecolors='none',edgecolors=RT_RED,lw=2,zorder=8)
ax.scatter(tl_wld_day.DATE,[1]*len(tl_wld_day),s=16,color=TL_COLORS['WLD'])
for p in audit_parts.itertuples():ax.scatter(p.END_DT,2,marker='D',s=30,color=TL_PART_COLORS.get(p.PART_CD,TL_COLORS['PART']),alpha=.8)
ax.set_yticks([4,3,2,1,0],['온도 14 LOT','CAL01 작업 44건','CAL01 부품 연결 9행','WLD01 관측 77일','CR1 계획수리 2회'])
ax.set_ylim(-.8,6.0);ax.set_xlim(YEAR_START-pd.Timedelta(days=4),YEAR_END)
ax.set_xticks(pd.date_range(YEAR_START,periods=12,freq='MS'));ax.xaxis.set_major_formatter(mdates.DateFormatter('%m월'))
tl_style(ax);tl_place_lot_labels(ax,list(tl_lots.itertuples()),4)
rt_header(fig,'최종 연간 지도. 소둔로 작업·부품과 관측의 연결',
    '빨간 원: 수리 후 승온 240061·240117 / 감지와 부품 WO 연결 240133 · 작업은 CAL01 한정')
rt_finish(fig,Path(FINAL_DIR)/'02_연간_후속조사_지도.png')

# 최종 결과 C: 다음에 무엇을 볼지 바로 읽을 수 있는 결론표.
result_rows=[
 ['1','240061 · SPHC','가열·균열 [TC-Z3~TC-Z6]\n0–100분','수리 후 64분\n4대역 약 39–42℃ 승온','운전 모드·투입 소재\nTC·OP·CP 변화 순서','더미 운전 확인 후보'],
 ['2','240117 · SPCD','가열 [TC-Z3~TC-Z6]\n냉각 [TC-OUT1·TC-OUT2]\n작업 전·중·후','4대역 약 37–40℃ 승온\nCAL 2작업·용접 31 COIL','승온과 작업 경계 구분\n소재 추적·작업 원문','더미 운전 확인 후보'],
 ['3','240069 · SPHC\n240037 · SPCE','가열·균열 [CP-Z4]\n원본 확대','CP 0.909111 / 0.60676\n1분 요약에 가려진 급변','동시 TC-Z4·OP-Z4\n신호 정의·계측 확인','신호 확인 후보'],
 ['4','240058 · DP590','가열 [TC-Z3~TC-Z6·CP-Z4]\n130분 이후','수리 전 후반\nCP·TC·OP 하강','종료 조건·운전 전환\n강종 다른 전후 비교 제한','전환 구간 확인'],
 ['5','240037·240121\n240133 · SPCE','과시효 [TC-Z7~TC-Z9]\n냉각 [TC-OUT1·TC-OUT2]\n전체 주기 비교','TC-Z9 약 197분 주기\n240133 관측 107.4분','위상·주기·진폭 비교\nBM 집중과 연결하지 않음','주기 차이 재검토'],
 ['참고','240108·240114\n240121','가열 [TC-Z3~TC-Z6]\n냉각 [TC-OUT1·TC-OUT2]\n용접 · LOT 관측 창','동시간 용접\n12 / 11 / 28 COIL','같은 강종 LOT 대조\n소재 대응·지연 확인','공동 관측 참고'],
 ['별도','C2400650\nC2400679','용접\n각 39구간','전체 구간 전력 0\n이 시각 온도 관측 없음','같은 조건 COIL 비교\n계측·운전 상태 확인','온도·더미 판정 제외']]
fig,ax=plt.subplots(figsize=(23,10));fig.subplots_adjust(left=.025,right=.98,top=.80,bottom=.05);ax.axis('off')
table=ax.table(cellText=result_rows,colLabels=['순서','대상','중점 공정·구간','선정 근거','다음 확인','현재 판단'],
    colWidths=[.045,.14,.19,.23,.23,.165],cellLoc='left',loc='center')
table.auto_set_font_size(False);table.set_fontsize(10.8)
for (i,j),cell in table.get_celld().items():
    cell.set_edgecolor('white');cell.set_height(.082 if i==0 else .119)
    cell.set_facecolor('#dfeaf1' if i==0 else ('#fff0ee' if i in [1,2] else '#f2f6f9'))
    if i==0:cell.set_text_props(weight='bold',color='#17344d')
    if i in [1,2] and j in [1,5]:cell.set_text_props(weight='bold',color=RT_RED)
rt_header(fig,'최종 결과. 무엇을 먼저 보고 어떤 판단을 보류할 것인가',
    '승온 확인 2 LOT · CP 원본 확인 2 LOT · 전환·주기·공동 관측은 분리 검토 · 실제 더미코일 확정 0건')
rt_finish(fig,Path(FINAL_DIR)/'03_최종_후보와_다음확인.png')


<a id="analysis-final"></a>

### 최종 후보와 다음 확인

#### 최종 결과와 후보를 좁힌 과정

1. **연간 관측 범위:** 온도 14 LOT 밖은 온도로 판단할 수 없다. 소둔로 CAL01 작업 44건과 작업번호로 연결된 부품교체 9행만 대조한다. 다른 설비의 작업은 소둔로 근거에 넣지 않는다.
2. **시간 단서:** 소둔로 작업 겹침은 240117이다. 240133은 감지가 관측 중이며, 관측 종료 9.6분 뒤 시작된 CBM에 가스 분석계 셀 3개 교체 이력이 연결된다. 작업 후 온도가 없어 교체 효과는 평가할 수 없다. 용접 동시 관측은 240108·240114·240117·240121이다. 수리 전후 첫/마지막 관측은 시간 맥락을 제공하지만 소재·강종 조건이 달라 직접적인 수리 효과 비교가 아니다.
3. **온도 단서:** 14 LOT 모두의 첫 5분과 80–100분 TC-Z3~Z6 중앙값을 비교했다. 네 가열대 모두 20℃ 이상 낮게 시작하며 수리 종료 후 72시간 안에 관측된 대상은 240061·240117이다. 20℃·72시간은 이번 탐색의 후보 선별 규칙이며 품질 합격 기준이나 고장 임계값이 아니다. 240044는 80분 전에 관측이 끝나 이 규칙으로 평가하지 않는다.
4. **후속 판독:** 240061·240117은 초기 승온과 운전 모드, 240069·240037은 CP 원본 급변, 240058은 종료 전 조건 전환을 먼저 확인한다. SPCE TC-Z9의 공통 107분 수준 차이는 전체 약 197분 주기의 위상과 함께 읽어 지속적인 이상 판정을 보류했다.
5. **용접의 별도 분기:** C2400650·C2400679 전력 0은 계측·운전 확인 후보다. 온도가 없고 정비 완료 뒤 발생했으므로 온도 이상 또는 더미코일로 연결하지 않는다. 용접의 장기 전력 차이도 조건별 탐색값이며 마모·수리 효과로 단정하지 않는다.

##### 더미코일은 어떻게 특정하는가

현재 특정한 것은 **승온·더미 운전을 확인할 LOT 시간창 240061·240117**이다. 강종의 가공성·품질·비용 정보를 제공받지 않았으므로 강종명으로 더미 활용 적합성을 정하지 않는다. 기존 강종별 '최우선/활용 가능/제외' 분류는 제거했다.

두 LOT에서 초기 승온이 뚜렷하다는 사실만으로 실제 더미코일 투입을 확정할 수 없다. 공정 준비 운전, 제품 생산 중 초기조건, 센서 상태 등 다른 설명을 구분해야 한다. **확정된 더미 LOT/COIL은 0건**이다.

240061은 온도와 용접이 겹치지 않는다. 240117에는 31 COIL의 시간 교집합이 있지만 이들을 더미코일로 지정하지 않는다. LOT–COIL 소재 추적, 더미/제품 운전 코드, 투입·배출 기록과 작업자 확인을 확보한 뒤 정확한 COIL ID를 판정한다. 검토표의 WARMUP_REVIEW는 후보 플래그이며 DUMMY_CONFIRMED는 모두 False다.

##### 조사 순서

먼저 240061의 0–100분 가열대 TC·OP·CP와 수리 후 운전 이력을 확인한다. 같은 SPHC 240044·240069·240091은 관측된 공통 시간까지만 비교한다. 다음은 240117의 초기 승온과 07:40–12:44 작업 경계를 분리해서 확인한다. 그 뒤 240069 571.48–577.48분, 240037 38.98–44.98분의 CP 원본을 같은 시각 TC/OP와 함께 점검한다.

SPCE에 사후정비가 집중됐다는 결론, 용접이 온도 변화를 일으켰다는 결론, 초기 승온이 곧 더미코일이라는 결론은 현재 자료로 내리지 않는다. 파일에서 관측한 사실과 필요한 후속 확인은 아래 14 LOT 표에 모두 남겼다.


##### 14 LOT의 개별 검토 결과

| LOT_NO | GRADE | PRIORITY | FOCUS_PROCESS | FOCUS_MIN_OR_CLOCK | FINAL_CLASS | REVIEW_NOTE |
| --- | --- | --- | --- | --- | --- | --- |
| 240001 | SGCC | 단독 관측 참고 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | TC-Z3 596–601분 | 원본 변화 위치 보존 | SGCC 1 LOT · 강종 내 반복 비교 불가 |
| 240024 | SPFH590 | 비교 기준 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] | 240108과 공통 경과 분 | 같은 강종 참고 | SPFH590 반복 비교 |
| 240037 | SPCE | 신호·주기 확인 | 가열·균열 [TC-Z4·OP-Z4·CP-Z4] / 과시효 [TC-Z7~TC-Z9] | CP 38.98–44.98분 / TC-Z9 전체 | CP 확인 · 주기 위상 비교 | CP 원본 0.60676 · TC-Z9 약 197분 주기 |
| 240038 | SPCEN | 비교 기준 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] | 240114와 공통 경과 분 | 같은 강종 참고 | SPCEN 반복 비교 |
| 240044 | SPHC | 비교 기준 | 세정 [CLN-IN] / 건조 [TC-Z1·TC-Z2] / 가열·균열 [TC-Z3~TC-Z6] / 과시효 [TC-Z7~TC-Z9] / 냉각 [TC-OUT1·TC-OUT2] | 전체 74.6분 | 같은 강종 참고 | SPHC 반복 비교 · 80–100분은 미관측 |
| 240058 | DP590 | 전환 확인 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 130분 이후~종료 | 종료 조건 전환 확인 | 수리 전 관측 · 후반 CP와 TC/OP 하강 |
| 240061 | SPHC | 우선 검토 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 0–100분 | 승온·더미 운전 확인 후보 | 수리 후 64분 · 4대역 초기 승온 |
| 240069 | SPHC | 신호 확인 | 가열·균열 [CP-Z4, TC-Z4, OP-Z4] | 571.48–577.48분 | CP 급변 확인 | CP 원본 0.909111 · 동시 TC/OP 대조 |
| 240091 | SPHC | 같은 강종 비교 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 전체 경과 분 | 같은 강종 참고 | SPHC 반복 비교 · 소둔로 작업 겹침 없음 |
| 240108 | SPFH590 | 공동 관측 참고 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | TC-Z3 449–454분 / LOT 전체 | 온도·용접 시간 대조 | 같은 강종 240024 비교 · 용접 12 COIL |
| 240114 | SPCEN | 공동 관측 참고 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | TC-Z3 31–36분 / LOT 전체 | 온도·용접 시간 대조 | 같은 강종 240038 비교 · 용접 11 COIL |
| 240117 | SPCD | 우선 검토 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | 0–100분 및 07:40–12:44 | 승온·더미 운전 확인 후보 | 초기 승온 + CAL 작업 2건 + 용접 31 COIL |
| 240121 | SPCE | 공동 관측 참고 | 냉각 [TC-OUT1·TC-OUT2] / 용접 | 처음 107분 및 LOT 전체 | 위상·조건 비교 | 냉각 변동폭 비교 · 용접 28 COIL |
| 240133 | SPCE | 감지·교체 연결 확인 | 가열·균열 [CP-Z4] / 냉각 [TC-OUT1·TC-OUT2] | 감지 57분 / 종료 9.6분 뒤 착수 | 감지 전후 신호·WO 교체 확인 | CBM 02105 → 가스 분석계 셀 INS-006 3개 · 작업 후 온도 없음 |

[14 LOT 승온 비교](../figures/05_all_data_timeseries_jinny_01/07_최종결과/01_승온_더미운전_확인후보.png)

[연간 후속 조사 지도](../figures/05_all_data_timeseries_jinny_01/07_최종결과/02_연간_후속조사_지도.png)

[최종 결과](../figures/05_all_data_timeseries_jinny_01/07_최종결과/03_최종_후보와_다음확인.png)


**결과 — 14 LOT의 최종 검토 결과**

| 행 | LOT_NO | GRADE | PRIORITY | FOCUS_PROCESS | FOCUS_MIN_OR_CLOCK | FINAL_CLASS | REVIEW_NOTE | DUMMY_GRADE_GROUP | DUMMY_CONFIRMED | TC3_RISE | FOUR_ZONE_RISE_MIN | FOUR_ZONE_RISE_MAX | POST_PLAN_MIN | CAL01_WORK | CAL01_DETECT | WELD_COIL | CAL01_PART_WO | DETECT_LINKED_PART_WO |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 240001 | SGCC | 단독 관측 참고 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | TC-Z3 596–601분 | 원본 변화 위치 보존 | SGCC 1 LOT · 강종 내 반복 비교 불가 | 강종만으로 판단하지 않음 | False | -0.0370 | -0.6430 | -0.0370 | NaN | 0 | 0 | 0 | 0 | 0 |
| 1 | 240024 | SPFH590 | 비교 기준 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] | 240108과 공통 경과 분 | 같은 강종 참고 | SPFH590 반복 비교 | 강종만으로 판단하지 않음 | False | 1.2750 | 0.0990 | 1.2750 | NaN | 0 | 0 | 0 | 0 | 0 |
| 2 | 240037 | SPCE | 신호·주기 확인 | 가열·균열 [TC-Z4·OP-Z4·CP-Z4] / 과시효 [TC-Z7~TC-Z9] | CP 38.98–44.98분 / TC-Z9 전체 | CP 확인 · 주기 위상 비교 | CP 원본 0.60676 · TC-Z9 약 197분 주기 | 강종만으로 판단하지 않음 | False | -0.7870 | -0.7870 | 0.0170 | NaN | 0 | 0 | 0 | 0 | 0 |
| 3 | 240038 | SPCEN | 비교 기준 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] | 240114와 공통 경과 분 | 같은 강종 참고 | SPCEN 반복 비교 | 강종만으로 판단하지 않음 | False | -0.7670 | -0.7670 | 0.1500 | NaN | 0 | 0 | 0 | 0 | 0 |
| 4 | 240044 | SPHC | 비교 기준 | 세정 [CLN-IN] / 건조 [TC-Z1·TC-Z2] / 가열·균열 [TC-Z3~... | 전체 74.6분 | 같은 강종 참고 | SPHC 반복 비교 · 80–100분은 미관측 | 강종만으로 판단하지 않음 | False | NaN | NaN | NaN | NaN | 0 | 0 | 0 | 0 | 0 |
| 5 | 240058 | DP590 | 전환 확인 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 130분 이후~종료 | 종료 조건 전환 확인 | 수리 전 관측 · 후반 CP와 TC/OP 하강 | 강종만으로 판단하지 않음 | False | -0.9950 | -0.9950 | 0.2330 | NaN | 0 | 0 | 0 | 0 | 0 |
| 6 | 240061 | SPHC | 우선 검토 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 0–100분 | 승온·더미 운전 확인 후보 | 수리 후 64분 · 4대역 초기 승온 | 승온·더미 운전 확인 후보 | False | 41.8060 | 38.7000 | 41.8060 | 64.0 | 0 | 0 | 0 | 0 | 0 |
| 7 | 240069 | SPHC | 신호 확인 | 가열·균열 [CP-Z4, TC-Z4, OP-Z4] | 571.48–577.48분 | CP 급변 확인 | CP 원본 0.909111 · 동시 TC/OP 대조 | 강종만으로 판단하지 않음 | False | -1.0450 | -1.0450 | 0.1120 | 32697.0 | 0 | 0 | 0 | 0 | 0 |
| 8 | 240091 | SPHC | 같은 강종 비교 | 가열·균열 [TC-Z3~TC-Z6 · OP-Z3~OP-Z6 · CP-Z4] | 전체 경과 분 | 같은 강종 참고 | SPHC 반복 비교 · 소둔로 작업 겹침 없음 | 강종만으로 판단하지 않음 | False | 0.6560 | -0.2360 | 0.6560 | 123723.0 | 0 | 0 | 0 | 0 | 0 |
| 9 | 240108 | SPFH590 | 공동 관측 참고 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | TC-Z3 449–454분 / LOT 전체 | 온도·용접 시간 대조 | 같은 강종 240024 비교 · 용접 12 COIL | 강종만으로 판단하지 않음 | False | 0.0450 | 0.0450 | 0.1060 | 180245.0 | 0 | 0 | 12 | 0 | 0 |
| 10 | 240114 | SPCEN | 공동 관측 참고 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | TC-Z3 31–36분 / LOT 전체 | 온도·용접 시간 대조 | 같은 강종 240038 비교 · 용접 11 COIL | 강종만으로 판단하지 않음 | False | 5.7340 | -0.6570 | 5.7340 | 205420.0 | 0 | 0 | 11 | 0 | 0 |
| 11 | 240117 | SPCD | 우선 검토 | 가열·균열 [TC-Z3~TC-Z6] / 냉각 [TC-OUT1·TC-OUT2] / 용접 | 0–100분 및 07:40–12:44 | 승온·더미 운전 확인 후보 | 초기 승온 + CAL 작업 2건 + 용접 31 COIL | 승온·더미 운전 확인 후보 | False | 37.1195 | 37.1195 | 40.0005 | 3308.0 | 2 | 1 | 31 | 0 | 0 |
| 12 | 240121 | SPCE | 공동 관측 참고 | 냉각 [TC-OUT1·TC-OUT2] / 용접 | 처음 107분 및 LOT 전체 | 위상·조건 비교 | 냉각 변동폭 비교 · 용접 28 COIL | 강종만으로 판단하지 않음 | False | 0.1670 | -0.2170 | 0.1670 | 19171.0 | 0 | 0 | 28 | 0 | 0 |
| 13 | 240133 | SPCE | 감지·교체 연결 확인 | 가열·균열 [CP-Z4] / 냉각 [TC-OUT1·TC-OUT2] | 감지 57분 / 종료 9.6분 뒤 착수 | 감지 전후 신호·WO 교체 확인 | CBM 02105 → 가스 분석계 셀 INS-006 3개 · 작업 후 온도 없음 | 강종만으로 판단하지 않음 | False | 0.4290 | -0.0030 | 0.4290 | 68577.0 | 0 | 1 | 0 | 0 | 1 |

# 15. 결과 확인

계산 결과는 각 분석 단계의 마크다운 표에서 확인합니다. 결과 CSV를 저장하거나 다시 읽지 않습니다.
그림은 [주제별 그림 목록](../figures/05_all_data_timeseries_jinny_01/README.md)에서 찾을 수 있습니다.

In [ ]:
# 계산표는 각 분석 단계에서 마크다운으로 표시한다. 결과 CSV는 생성하지 않는다.

## 결과를 읽는 순서

그림 1에서 연간 맥락을 찾고, 1B에서 실제 신호·작업 경계, 1C에서 수리·부품 WO·다음 LOT의 연결을 확인한다. 그림 2는 관측 범위, 그림 3·4는 같은 강종의 실제 경과 분과 원본 단서, 그림 5는 동시간 비교, 그림 6·7은 용접 확인이다. 마지막 세 그림에서 초기 승온 근거 → 소둔로 연간 지도 → 최종 대상·공정·다음 확인 순서로 결론을 읽는다. 상세 REMARK와 연결 규칙은 이 노트북의 [정비·부품 원문](#analysis-records)과 [소둔로 작업·부품·관측의 연결](#analysis-overlap)에서 확인한다.